# Coder, phase 4: the engineer exam (Ollama on Kaggle's free GPU)

**The question:** does a free coding model work like an engineer *on its own*, with no style guide or checklist in
the prompt? Each model in `MODELS` gets every task as a user would write it (plus the project's files for a bug
fix, review or refactor, and one line on how to send files back). Nothing else: no system prompt.

**How a reply is graded (0-100):** the task's hidden tests, never shown (55); did it write its own tests, and do
they pass (15); lint with a fixed ruff rule set (10); security with bandit (10); does it explain what it did (10).
Stray files (scratch scripts nothing uses) cost 5. "Solved" = every hidden test passes. A reply that names no file
has its Python blocks placed by a fixed fallback (see `fallback_files`); the report counts how many needed it.

It saves everything to `exam.json` (Output tab) after every task: every reply, every grade and the raw test output.
Time: at most `MINUTES_PER_MODEL` minutes per model (download included), so about 4½ hours for 4 models.

**Settings (right side):** Accelerator **GPU T4 x2**, Internet **on**. Then **Run all**.

**Or with no clicks:** on GitHub, **Actions → Coder model on Kaggle → Run workflow → action `exam`**, or ask Claude.

In [ ]:
#@title 1. Settings: which models, how many tasks, time per model
import glob, json, os, re, shutil, subprocess, sys, time, urllib.error, urllib.request
from collections import Counter

# The models: names from ollama.com/library. Claude and GitHub Actions fill this line.
MODELS = ["qwen3-coder:30b", "qwen2.5-coder:14b", "gpt-oss:20b", "qwen2.5-coder:7b"]
# How many tasks to use (None = all). Claude and GitHub Actions fill this line.
LIMIT = None
MINUTES_PER_MODEL = 60  # a model stops after this (download, loading and answers), so the run ends in time
SAMPLES = 1  # answers per task (with temperature 0 they are all the same)
OPTIONS = {"temperature": 0, "num_ctx": 16384, "num_predict": 8192}  # room for the task, its files and whole files back
THINK = {"gpt-oss": "medium"}  # thinking models: how much to think first
THINKING_NUM_PREDICT = 12288  # thinking models: room for the thinking and the answer (the prompt still fits num_ctx)
# Downloads go to the big disk. Not /kaggle/working: it holds only about 20 GB, and all of it is saved as output.
MODELS_DIR = "/tmp/ollama-models"
GRADE_DIR = "/tmp/exam-grading"  # each reply is tested in a fresh folder here (deleted after)
DELETE_AFTER = True  # delete each model after its turn, so the disk holds only one at a time
WORKING_DIR = "/kaggle/working"  # exam.json is saved here (the notebook's Output tab)

In [ ]:
#@title 2. Helpers (no need to read; built from coder/fixeval.py, coder/exam/grade.py and coder/exam/exam.py)
def post(url, path, body, timeout=600):
    request = urllib.request.Request(url + path, json.dumps(body).encode(), {"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as resp:
            return json.load(resp)
    except urllib.error.HTTPError as e:  # Ollama says why in the body: {"error": "..."}
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:500]}") from None


def pull(url, model, log=print):
    """Download a model (like `ollama pull`). Its size in GB. Prints each new step."""
    request = urllib.request.Request(url + "/api/pull", json.dumps({"model": model, "stream": True}).encode(),
                                     {"Content-Type": "application/json"})
    sizes, last = {}, None
    try:
        with urllib.request.urlopen(request, timeout=900) as resp:
            for line in resp:
                if not line.strip():
                    continue
                step = json.loads(line)
                if step.get("error"):
                    raise RuntimeError(step["error"])
                if step.get("digest") and step.get("total"):
                    sizes[step["digest"]] = step["total"]
                status = re.sub(r"pulling [0-9a-f]{12,}", "pulling a part", step.get("status", ""))
                if status != last:
                    log(f"   {model}: {status}")
                    last = status
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:500]}") from None
    if last != "success":
        raise RuntimeError(f"the download did not finish (last step: {last})")
    return round(sum(sizes.values()) / 1e9, 1)


def in_memory(url, model):
    """The model as Ollama holds it now (from /api/ps: size, size_vram, ...), or None when it is not loaded."""
    with urllib.request.urlopen(url + "/api/ps", timeout=10) as resp:
        models = json.load(resp).get("models") or []
    names = {model, model + ":latest"}
    return next((m for m in models if m.get("name") in names or m.get("model") in names), None)


def processor(url, model):
    """Where the model runs, as `ollama ps` says it: "100% GPU", "100% CPU", "25%/75% CPU/GPU", or None."""
    try:
        m = in_memory(url, model)
    except Exception:  # noqa: BLE001
        return None
    if not m or not m.get("size"):
        return None
    cpu = round(100 * (m["size"] - (m.get("size_vram") or 0)) / m["size"])
    return "100% GPU" if cpu <= 0 else "100% CPU" if cpu >= 100 else f"{cpu}%/{100 - cpu}% CPU/GPU"


def unload(url, model):
    """Take the model out of memory (keep_alive 0) and wait until it is gone."""
    post(url, "/api/generate", {"model": model, "keep_alive": 0}, timeout=120)
    for _ in range(60):
        if in_memory(url, model) is None:
            return
        time.sleep(1)


def delete(url, model):
    """Delete the downloaded model (like `ollama rm`), to free the disk for the next one."""
    request = urllib.request.Request(url + "/api/delete", json.dumps({"model": model}).encode(),
                                     {"Content-Type": "application/json"}, method="DELETE")
    with urllib.request.urlopen(request, timeout=120):
        pass


def text_of(command):
    """What a command printed ("" when it is missing or fails), e.g. `ollama ps` or nvidia-smi."""
    try:
        return subprocess.run(command, shell=True, capture_output=True, text=True, timeout=60,
                              check=False).stdout.strip()
    except Exception:  # noqa: BLE001
        return ""


def think_for(model, think):
    """The "think" setting for this model ({"gpt-oss": "low"} → "low" for gpt-oss:20b), or None."""
    return next((level for prefix, level in (think or {}).items() if model.startswith(prefix)), None)


def disk_free_gb(path):
    try:
        return round(shutil.disk_usage(path).free / 1e9, 1)
    except OSError:
        return None


def tidy_up(url, model, entry, delete_after=True, log=print):
    """Unload the model (and delete it, to free the disk). Problems are recorded, never raised."""
    for step, do in (("unload", unload), ("delete", delete if delete_after else None)):
        if do is None:
            continue
        try:
            do(url, model)
        except Exception as e:  # noqa: BLE001 - the next model must still run
            entry.setdefault("errors", []).append(f"{step}: {e!r}"[:300])
            log(f"   {model}: {step} failed: {e!r}")


# ---- coder/exam/grade.py ----
"""The engineer exam (coder phase 4): load the tasks, ask in plain words, read the files out of a reply, grade them.

No production checklist in the prompt on purpose: the exam measures what a model does on its own. A reply is graded
on: the task's hidden tests (55), its own tests (15), lint (10), security (10) and an explanation (10); stray files
cost 5. Plain Python; pytest, ruff and bandit are run as programs (bandit is optional). No network.

coder/exam/build_exam.py puts this file, up to "the command line", into coder/exam/exam.ipynb, so Kaggle grades
exactly as here. Task writers and CI check every task's reference solution with:

    python -m coder.exam.grade --check-references                 the tasks in coder/exam/tasks
    python -m coder.exam.grade --check-references --tasks DIR     another folder of tasks
"""

import ast
import importlib.util
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

KINDS = ("build", "bugfix", "review", "refactor")
TASK_NAME = re.compile(r"^\d{2}-[a-z0-9][a-z0-9-]*$")  # e.g. 01-inventory-ledger
DEFAULT_TIMEOUT = 60  # seconds for one pytest run, when meta.json gives none
MAX_TIMEOUT = 600
HIDDEN_TEST = "test_hidden.py"
# One neutral line on the reply format; {path} is the task's first expected file (a real name, never a placeholder).
REPLY_FORMAT = ("Reply with every file you create or change in full, each in its own fenced code block that starts with "
                "the file's path, like ```{path} — you may write a short explanation first.")
WEIGHTS = {"hidden": 55, "own_tests": 15, "lint": 10, "security": 10, "explains": 10}  # of 100
JUNK_PENALTY = 5  # points off the total when the reply leaves stray files
EXPLAIN_WORDS = 40  # an explanation is more than this many words outside the code blocks
LINT_ZERO = 10  # this many lint issues (or more) = lint 0
CUT = 3000  # longest raw output kept per check
CHECK_SECONDS = 120  # ruff and bandit
# Lint: a fixed set (pycodestyle errors, pyflakes, bugbear, pyupgrade, simplify, pylint's errors, warnings and
# complexity), line length 120. Never the model's own config.
RUFF_CONFIG = """line-length = 120
target-version = "py310"

[lint]
select = ["E", "F", "W", "B", "UP", "SIM", "PLE", "PLW", "PLR0911", "PLR0912", "PLR0915", "PLR1714", "PLR5501"]
"""
TEST_FILE = re.compile(r"(?:^|/)(?:test_[^/]*|[^/]*_test)\.py$")  # test_x.py, x_test.py: tests pytest finds
TEST_HELPER = re.compile(r"(?:^|/)conftest\.py$|(?:^|/)tests?/")  # conftest.py, anything in tests/
DOC_FILE = re.compile(r"(?:^|/)(?:README|CHANGELOG|CHANGES)[^/]*$|\.(?:md|rst)$", re.IGNORECASE)
CONFIG_FILES = {"pyproject.toml", "setup.cfg", "setup.py", "pytest.ini", "tox.ini", ".gitignore", "requirements.txt",
                "requirements-dev.txt"}
SAFE_PATH = re.compile(r"^[\w.\-/]+$")
RESERVED_DIRS = {"hidden", "reference", ".git"}


# ---- the tasks ----

def safe_path(path):
    """The path as a clean relative path ("./a\\b.py" → "a/b.py"), or None when it is not safe to write:
    absolute, "..", odd characters, or inside hidden/, reference/ or .git/."""
    if not isinstance(path, str):
        return None
    path = path.strip().replace("\\", "/")
    while path.startswith("./"):
        path = path[2:]
    parts = path.split("/")
    if (not path or path.startswith(("/", "~")) or re.match(r"^[A-Za-z]:", path) or not SAFE_PATH.match(path)
            or any(p in ("", ".", "..") for p in parts) or parts[0] in RESERVED_DIRS or len(path) > 200):
        return None
    return path


def read_folder(folder):
    """{relative path: text} of every file under folder (no caches)."""
    files = {}
    for path in sorted(Path(folder).rglob("*")):
        rel = path.relative_to(folder).as_posix()
        if path.is_dir() or "__pycache__" in rel.split("/") or ".pytest_cache" in rel or path.suffix == ".pyc":
            continue
        try:
            files[rel] = path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            raise ValueError(f"{path}: not UTF-8 text (tasks hold text files only)") from None
    return files


def check_meta(meta, problems):
    """meta.json, with the default timeout filled in. Adds what is wrong to problems."""
    if not isinstance(meta, dict):
        problems.append("meta.json must be a JSON object")
        return {}
    meta = {**meta, "timeout": meta.get("timeout", DEFAULT_TIMEOUT)}
    if meta.get("kind") not in KINDS:
        problems.append(f"meta.json: kind must be one of {', '.join(KINDS)} (got {meta.get('kind')!r})")
    difficulty = meta.get("difficulty")
    if isinstance(difficulty, bool) or difficulty not in (1, 2, 3):
        problems.append(f"meta.json: difficulty must be 1, 2 or 3 (got {difficulty!r})")
    files = meta.get("files")
    if not isinstance(files, list) or not files or any(safe_path(f) != f for f in files):
        problems.append(f"meta.json: files must be a list of relative paths like [\"inventory.py\"] (got {files!r})")
    timeout = meta["timeout"]
    if isinstance(timeout, bool) or not isinstance(timeout, int) or not 1 <= timeout <= MAX_TIMEOUT:
        problems.append(f"meta.json: timeout must be whole seconds, 1 to {MAX_TIMEOUT} (got {timeout!r})")
    return meta


def load_task(folder):
    """One task folder as a dict: id, task_md, meta, starter, hidden, reference. ValueError says what is wrong."""
    folder = Path(folder)
    problems = []
    if not TASK_NAME.match(folder.name):
        problems.append("the folder name must look like 01-short-name (two digits, a dash, lowercase words)")
    task_md = (folder / "task.md").read_text(encoding="utf-8").strip() if (folder / "task.md").is_file() else ""
    if not task_md:
        problems.append("task.md is missing or empty")
    meta = {}
    if not (folder / "meta.json").is_file():
        problems.append("meta.json is missing")
    else:
        try:
            meta = check_meta(json.loads((folder / "meta.json").read_text(encoding="utf-8")), problems)
        except json.JSONDecodeError as e:
            problems.append(f"meta.json is not valid JSON: {e}")
    hidden_file = folder / "hidden" / HIDDEN_TEST
    hidden = hidden_file.read_text(encoding="utf-8") if hidden_file.is_file() else ""
    if not hidden.strip():
        problems.append(f"hidden/{HIDDEN_TEST} is missing or empty")
    elif not re.search(r"^\s*(?:async\s+)?def test_|^class Test", hidden, re.MULTILINE):
        problems.append(f"hidden/{HIDDEN_TEST} has no test functions")
    extra = sorted(set(read_folder(folder / "hidden")) - {HIDDEN_TEST}) if (folder / "hidden").is_dir() else []
    if extra:
        problems.append(f"hidden/ must hold only {HIDDEN_TEST} (also found {extra})")
    starter, reference = {}, {}
    for name, files in (("starter", starter), ("reference", reference)):
        if (folder / name).is_dir():
            try:
                files.update(read_folder(folder / name))
            except ValueError as e:
                problems.append(str(e))
            bad = [p for p in files if safe_path(p) != p]
            if bad:
                problems.append(f"{name}/ has paths that are not allowed: {bad}")
    if not reference:
        problems.append("reference/ is missing or empty")
    for path in meta.get("files") if isinstance(meta.get("files"), list) else []:
        if isinstance(path, str) and path not in starter and path not in reference:
            problems.append(f"meta.json names {path}, but neither starter/ nor reference/ has it")
    if problems:
        raise ValueError(f"{folder.name}: " + "; ".join(problems))
    return {"id": folder.name, "task_md": task_md, "meta": meta, "starter": starter, "hidden": hidden,
            "reference": reference}


def load_tasks(folder):
    """Every task in folder (sorted by name). ValueError lists every task that is not in the format;
    FileNotFoundError when the folder is missing."""
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f"no folder {folder}")
    tasks, problems = [], []
    for sub in sorted(folder.iterdir()):
        if not sub.is_dir() or sub.name.startswith((".", "_")):
            continue
        try:
            tasks.append(load_task(sub))
        except ValueError as e:
            problems.append(str(e))
    if problems:
        raise ValueError(f"{len(problems)} task(s) in {folder} are not in the exam format:\n"
                         + "\n".join(f" - {p}" for p in problems))
    return tasks


def fenced(path, text):
    """A file as a fenced block whose first line is its path (a longer fence when the text holds ```)."""
    longest = max((len(run) for run in re.findall(r"`{3,}", text)), default=0)
    fence = "`" * max(3, longest + 1)
    return f"{fence}{path}\n{text.rstrip(chr(10))}\n{fence}"


def prompt_for(task):
    """The user message: the request as written, the project's files (if any), and one neutral line on the reply
    format. No checklist: what an engineer would add on their own is what the exam measures."""
    parts = [task["task_md"].strip()]
    if task.get("starter"):
        parts.append("The project's files:")
        parts += [fenced(path, text) for path, text in sorted(task["starter"].items())]
    parts.append(reply_format(task))
    return "\n\n".join(parts)


def reply_format(task):
    """The format line, with the task's first expected file as the example."""
    return REPLY_FORMAT.format(path=task["meta"]["files"][0])


# ---- the reply ----

THINKING = re.compile(r"<think(?:ing)?>.*?</think(?:ing)?>", re.DOTALL)
FENCE = re.compile(r"^[ \t]{0,3}(`{3,}|~{3,})[ \t]*(.*?)[ \t]*$")
MARKER = re.compile(r"^\s*(?:#|//|<!--)\s*(?:file(?:name)?|path)\s*:\s*(\S+?)\s*(?:-->)?\s*$", re.IGNORECASE)
# A first line that is only a comment with a file name ("# inventory.py"): known extensions only, so "# os.path" is
# not read as a name.
BARE_MARKER = re.compile(r"^\s*(?:#|//)\s*([\w./\\-]*[\w-]\.(?:py|md|txt|toml|cfg|ini|json|ya?ml|rst))\s*$")
PATHLIKE = re.compile(r"^[\w./\\-]*[\w-]\.[A-Za-z][A-Za-z0-9]*$")  # x.py, src/x.py (not 3.11)
# A placeholder copied from a format line ("path/to/file.ext"): never a file, its block counts as unnamed.
PLACEHOLDER = re.compile(r"(?:^|/)path/to/|\.ext$", re.IGNORECASE)
LANGS = {"python", "python3", "py", "text", "txt", "bash", "sh", "shell", "console", "json", "toml", "yaml", "yml",
         "ini", "markdown", "md", "diff", "patch", "plaintext", "cfg"}


def without_thinking(reply):
    """The reply without <think>...</think> parts (a thinking model's notes before its answer)."""
    reply = THINKING.sub("", reply or "")
    if "</think>" in reply:  # the opening tag was in the prompt template, only the closing one came back
        reply = reply.rsplit("</think>", 1)[1]
    return reply


def _clean(token):
    return token.strip().strip("`'\"*_").strip()


def path_in_info(info):
    """The path in a fence's info string: "a.py", "python a.py", "python:a.py", "python title=a.py"; or None."""
    tokens = info.split()
    if tokens and ":" in tokens[0] and not re.match(r"^[A-Za-z]:[\\/]", tokens[0]):  # python:a.py (not C:\a.py)
        tokens = [*tokens[0].split(":", 1), *tokens[1:]]
    for token in tokens:
        token = _clean(token.split("=", 1)[1] if "=" in token else token)
        if token and token.lower() not in LANGS and PATHLIKE.match(token):
            return token
    return None


def path_in_heading(line):
    """A path named by the line just before a block: "**a.py**", "### `a.py`", "File: a.py", "File a.py", "a.py:",
    "`a.py`:", "a.py (new)", "Here is the new `a.py`:"; or None."""
    line = re.sub(r"^\s*(?:#{1,6}\s*|[-*+]\s+|\d+[.)]\s+)", "", line or "").strip()
    plain = re.sub(r"^[*_`]*(?:file(?:name)?|path)\b[*_`]*(?:\s*:\s*|\s+)", "", line, flags=re.IGNORECASE)
    plain = re.sub(r"\s*\((?:new|updated|changed|modified|fixed)[^)]*\)\W*$", "", plain, flags=re.IGNORECASE)
    plain = _clean(_clean(plain).rstrip(":"))
    if PATHLIKE.match(plain):
        return plain
    if line.endswith(":"):  # a sentence: exactly one path in backticks or bold
        named = [a or b for a, b in re.findall(r"`([^`\s]+)`|\*\*([^*\s]+)\*\*", line)]
        named = [n for n in named if PATHLIKE.match(n)]
        if len(named) == 1:
            return named[0]
    return None


def read_reply(reply, task=None):
    """The reply taken apart: {"files": {path: text}, "named_by": {path: how}, "found_by": {path: "named" or
    "fallback: ..."}, "fallback": None (not needed or no task given), "used" or "gave up: why", "rejected":
    [(path, why)], "unnamed_blocks": n, "cut_off": the last block was never closed, "explanation": the text outside
    the blocks, "words": its word count}. A later block for the same path replaces the earlier one. With the task,
    a reply in which no block names a path gets its unnamed Python blocks placed by fallback_files."""
    lines = without_thinking(reply).split("\n")
    prose, files, named_by, rejected, blocks = [], {}, {}, [], []
    unnamed, named, cut_off, before = 0, 0, False, ""
    i = 0
    while i < len(lines):
        opening = FENCE.match(lines[i])
        if not opening or (opening.group(1)[0] == "`" and "`" in opening.group(2)):
            prose.append(lines[i])
            if lines[i].strip():
                before = lines[i]
            i += 1
            continue
        fence, info = opening.groups()
        path, how = path_in_info(info), "fence"
        markdown = (path or "").lower().endswith((".md", ".markdown")) or info.lower() in ("md", "markdown")
        body, depth, closed = [], 0, False
        i += 1
        while i < len(lines):
            line = FENCE.match(lines[i])
            if line and line.group(1)[0] == fence[0] and len(line.group(1)) >= len(fence):
                if not line.group(2):  # a bare fence closes (or ends a block nested in markdown)
                    if depth == 0:
                        closed = True
                        break
                    depth -= 1
                elif markdown:
                    depth += 1
            body.append(lines[i])
            i += 1
        i += 1
        cut_off = not closed
        if path is None and body and (MARKER.match(body[0]) or BARE_MARKER.match(body[0])):
            path, how, body = (MARKER.match(body[0]) or BARE_MARKER.match(body[0])).group(1), "first line", body[1:]
        if path is None:
            path, how = path_in_heading(before), "heading"
        before = ""
        if path is not None and PLACEHOLDER.search(path.replace("\\", "/")):
            rejected.append((path, "a placeholder, not a file name"))
            path = None
        if path is None:
            unnamed += 1
            lang = (info.split() or [""])[0].lower()  # ```path/to/file.ext: no language given
            blocks.append({"lang": "" if PLACEHOLDER.search(lang) else lang,
                           "text": "\n".join(body).rstrip("\n") + "\n"})
            continue
        named += 1
        clean = safe_path(path)
        if clean is None:
            rejected.append((path, "not a safe relative path (absolute, '..', hidden/ or reference/)"))
            continue
        if how == "heading" and named_by.get(clean, "heading") != "heading":
            continue  # "Usage of `a.py`:" and an example must not replace a.py written in the asked format
        files[clean] = "\n".join(body).rstrip("\n") + "\n" if any(b.strip() for b in body) else ""
        named_by[clean] = how
    explanation = "\n".join(prose).strip()
    words = len([w for w in explanation.split() if re.search(r"\w", w)])
    found_by, fallback = dict.fromkeys(files, "named"), None
    if task is not None and not named and not files:
        files, found_by, fallback = fallback_files(blocks, task)
    return {"files": files, "named_by": named_by, "found_by": found_by, "fallback": fallback, "rejected": rejected,
            "unnamed_blocks": unnamed, "cut_off": cut_off, "explanation": explanation, "words": words}


def parse_files(reply, task=None):
    """{path: full text} of every file in the reply (see read_reply)."""
    return read_reply(reply, task)["files"]


# ---- the fallback: a reply that names no file ----
#
# Used only when no block in the reply names a path (a placeholder like path/to/file.ext is not a name). Only
# Python blocks count (```python, ```py, ```python3, or no language and the text is Python that defines or imports
# something); shell, console, text, output and other blocks are ignored. Then, in this order:
#   1. a block of pytest tests (a top-level def test_..., class Test... or import pytest) → test_<stem>.py, where
#      <stem> is the first expected file's name (test_1.py, test_2.py, ... when there are several such blocks);
#   2. a block that imports an expected module is an example of using it, not the module: left out;
#   3. one expected .py file: the one remaining block is that file; several are joined in order (a later definition
#      wins, as in Python), keeping only the blocks that define a top-level def or class when some do;
#   4. several expected .py files: each block goes to the file whose names it defines (the top-level names in that
#      file's starter, and the `names` on the task.md lines that mention the file); a block that matches no file or
#      several, and so the whole reply, is given up on.
# Giving up means no files, as before. grade_reply records how each file was found ("named" or "fallback: ...").

PY_LANGS = {"python", "python3", "py"}
TEST_BLOCK = re.compile(r"^(?:async\s+)?def test_\w*\s*\(|^class Test\w*\b|^import pytest\b|^from pytest import ",
                        re.MULTILINE)
TOP_NAME = re.compile(r"^(?:async\s+def|def|class)\s+([A-Za-z_]\w*)", re.MULTILINE)


def python_block(block):
    """Is this unnamed block Python code (not a shell session, output or text)?"""
    if block["lang"] in PY_LANGS:
        return True
    if block["lang"]:
        return False
    try:
        tree = ast.parse(block["text"])
    except (SyntaxError, ValueError):
        return False
    return any(isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom))
               for n in tree.body)


def imports_module(text, module):
    return re.search(rf"^\s*(?:from\s+{re.escape(module)}\s+import\b|import\s+{re.escape(module)}\b)", text,
                     re.MULTILINE) is not None


def names_for(task, expected):
    """{expected file: names the task ties to it} (see the fallback rules); a name tied to two files is dropped."""
    names = {}
    for path in expected:
        found = set(TOP_NAME.findall((task.get("starter") or {}).get(path, "")))
        for line in task["task_md"].splitlines():
            if path in line or path.rsplit("/", 1)[-1] in line:
                found |= set(re.findall(r"`([A-Za-z_]\w*)", line))
        names[path] = found
    shared = {n for p in expected for q in expected if p != q for n in names[p] & names[q]}
    return {p: n - shared for p, n in names.items()}


def fallback_files(blocks, task):
    """Place the unnamed blocks of a reply that names no file: (files, found_by, "used" or "gave up: why")."""
    expected = [p for p in task["meta"]["files"] if p.endswith(".py")]
    python = [b for b in blocks if python_block(b)]
    if not expected:
        return {}, {}, "gave up: no expected .py file"
    if not python:
        return {}, {}, "gave up: no Python block"
    tests = [b for b in python if TEST_BLOCK.search(b["text"])]
    modules = [module_of(p) for p in expected]
    code = [b for b in python if not any(b is t for t in tests) and not any(imports_module(b["text"], m)
                                                                         for m in modules)]
    if len(code) > 1 and any(TOP_NAME.search(b["text"]) for b in code):
        code = [b for b in code if TOP_NAME.search(b["text"])]
    if not code:
        return {}, {}, "gave up: only tests or examples of use, no code"
    files, found_by = {}, {}
    if len(expected) == 1:
        files[expected[0]] = "\n\n".join(b["text"].rstrip("\n") for b in code) + "\n"
        found_by[expected[0]] = ("fallback: the one code block" if len(code) == 1
                                 else f"fallback: {len(code)} code blocks joined in order")
    else:
        names = names_for(task, expected)
        placed = {}
        for b in code:
            defined = set(TOP_NAME.findall(b["text"]))
            hits = [p for p in expected if defined & names[p]]
            if len(hits) != 1:
                return {}, {}, (f"gave up: {len(expected)} expected files and a block that matches "
                                + ("none" if not hits else "several"))
            placed.setdefault(hits[0], []).append(b)
        for path, chosen in placed.items():
            files[path] = "\n\n".join(b["text"].rstrip("\n") for b in chosen) + "\n"
            used = sorted({n for b in chosen for n in TOP_NAME.findall(b["text"])} & names[path])
            found_by[path] = "fallback: the block defining " + ", ".join(used)
    stem = expected[0].rsplit("/", 1)[-1][:-3]
    for n, b in enumerate(tests, 1):
        path = f"test_{stem}.py" if len(tests) == 1 else f"test_{n}.py"
        files[path], found_by[path] = b["text"], "fallback: a block of tests"
    return files, found_by, "used"


# ---- running the checks ----

def cut(text, limit=CUT):
    """Long output cut to about limit characters: its start and its end (where pytest's summary is)."""
    text = text or ""
    if len(text) <= limit:
        return text
    head = limit // 3
    return text[:head] + f"\n... ({len(text) - limit} characters left out) ...\n" + text[-(limit - head):]


def run(command, cwd, timeout, env=None):
    """Run a program in its own process group: {"code", "out", "err", "timed_out", "seconds"}. At the time limit the
    whole group is killed (a test that starts processes or never ends cannot hang the exam)."""
    started = time.time()
    proc = subprocess.Popen(command, cwd=cwd, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True, encoding="utf-8", errors="replace",
                            start_new_session=True)
    timed_out = False
    try:
        out, err = proc.communicate(timeout=timeout)
    except subprocess.TimeoutExpired:
        timed_out = True
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except (OSError, AttributeError):
            proc.kill()
        try:
            out, err = proc.communicate(timeout=10)
        except subprocess.TimeoutExpired:  # something outside the group still holds the output open
            out, err = "", ""
    return {"code": proc.returncode, "out": out or "", "err": err or "", "timed_out": timed_out,
            "seconds": round(time.time() - started, 2)}


def program(module, name):
    """How to run a checker: python -m module when it is installed for this Python, else the program on PATH."""
    if importlib.util.find_spec(module):
        return [sys.executable, "-m", module]
    found = shutil.which(name)
    return [found] if found else None


def checker_versions():
    """The Python and checker versions (None: not installed)."""
    versions = {"python": sys.version.split()[0]}
    for module in ("pytest", "ruff", "bandit"):
        command = program(module, module)
        done = run([*command, "--version"], None, 60) if command else None
        text = (done["out"] + done["err"]).strip() if done and done["code"] == 0 else ""
        versions[module] = (text.splitlines() or [None])[0]
    return versions


def test_env():
    """The environment for pytest: no plugins loaded by themselves, no .pyc files, no options from outside."""
    env = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}
    env.pop("PYTEST_ADDOPTS", None)
    return env


def workspace(base, name, *layers):
    """A fresh folder base/name with the files of each layer written in order (later layers win). Files that
    cannot be written (a path that is also a folder) are listed in the second value."""
    root = Path(base) / name
    root.mkdir(parents=True)
    problems = []
    for layer in layers:
        for path, text in layer.items():
            target = root / path
            try:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(text, encoding="utf-8")
            except OSError as e:
                problems.append(f"could not write {path}: {e.strerror or e}")
    return root, problems


def junit_counts(path):
    """(passed, total) from a pytest junit xml file; (0, 0) when there is none."""
    try:
        cases = ET.parse(path).getroot().iter("testcase")
    except (OSError, ET.ParseError):
        return 0, 0
    passed = total = 0
    for case in cases:
        total += 1
        passed += not any(child.tag in ("failure", "error", "skipped") for child in case)
    return passed, total


def run_pytest(root, tests, timeout, junit, isolated):
    """pytest on these test files in root. isolated: ignore any pytest config and conftest.py in root (the hidden
    tests must not be steered by the model's files)."""
    command = [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--tb=short", "-rfE",
               f"--junitxml={junit}", "--rootdir", str(root)]
    if isolated:
        ini = Path(junit).parent / "pytest-exam.ini"
        ini.write_text("[pytest]\n", encoding="utf-8")
        command += ["-c", str(ini), "--noconftest"]
    done = run([*command, *tests], root, timeout, test_env())
    passed, total = (0, 0) if done["timed_out"] else junit_counts(junit)
    output = done["out"] + done["err"]
    if done["timed_out"]:
        output += f"\ntimeout after {timeout} s"
    return {"passed": passed, "total": total, "exit_code": done["code"], "timed_out": done["timed_out"],
            "seconds": done["seconds"], "output": cut(output)}


def hidden_tests(task, files, base, name="hidden"):
    """The task's hidden tests on starter + files: {"passed", "total", "score", ...}."""
    root, problems = workspace(base, name, task.get("starter") or {}, files, {HIDDEN_TEST: task["hidden"]})
    result = run_pytest(root, [HIDDEN_TEST], task["meta"]["timeout"], Path(base) / f"{name}.xml", isolated=True)
    result["score"] = round(result["passed"] / result["total"], 4) if result["total"] else 0.0
    if problems:
        result["output"] = cut("\n".join(problems) + "\n" + result["output"])
    return result


def run_ruff(root, paths):
    """Lint issues in these files with the exam's fixed rules: (count or None when ruff fails, text)."""
    command = program("ruff", "ruff")
    if command is None:
        return None, "ruff is not installed"
    config = root.parent / "ruff-exam.toml"
    config.write_text(RUFF_CONFIG, encoding="utf-8")
    done = run([*command, "check", "--no-cache", "--config", str(config), "--output-format", "json", "--exit-zero",
                *paths], root, CHECK_SECONDS)
    try:
        issues = json.loads(done["out"])
    except json.JSONDecodeError:
        return None, cut(f"ruff did not run (exit {done['code']}): {done['err'] or done['out']}")
    lines = [f"{i.get('filename', '?').replace(str(root) + os.sep, '')}:{(i.get('location') or {}).get('row')} "
             f"{i.get('code') or 'syntax'} {i.get('message')}" for i in issues]
    return len(issues), cut("\n".join(lines))


def run_bandit(root, paths):
    """Bandit's findings in these files: (list of {severity, test_id, file, line, text}, or None when bandit is not
    installed or fails; text)."""
    command = program("bandit", "bandit")
    if command is None:
        return None, "bandit is not installed: security is left out of the total"
    done = run([*command, "-q", "-f", "json", *paths], root, CHECK_SECONDS)
    try:
        report = json.loads(done["out"])
    except json.JSONDecodeError:
        return None, cut(f"bandit did not run (exit {done['code']}): {done['err'] or done['out']}")
    found = [{"severity": r.get("issue_severity"), "test_id": r.get("test_id"), "file": r.get("filename"),
              "line": r.get("line_number"), "text": r.get("issue_text")} for r in report.get("results") or []]
    text = "\n".join(f"{f['file']}:{f['line']} {f['severity']} {f['test_id']} {f['text']}" for f in found)
    errors = report.get("errors") or []
    return found, cut(text + ("\nerrors: " + json.dumps(errors) if errors else ""))


def module_of(path):
    """"shop/cart.py" → "shop.cart"; "shop/__init__.py" → "shop"."""
    parts = path[:-3].split("/")
    return ".".join(parts[:-1] if parts[-1] == "__init__" else parts)


def imports_of(path, text):
    """Every module name this file may import (each dotted prefix too, relative imports resolved)."""
    try:
        tree = ast.parse(text)
    except (SyntaxError, ValueError):
        return set()
    package = module_of(path).split(".")[:-1] if not path.endswith("__init__.py") else module_of(path).split(".")
    names = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            full = [a.name for a in node.names]
        elif isinstance(node, ast.ImportFrom):
            base = package[: len(package) - node.level + 1] if node.level else []
            stem = ".".join([*base, *(node.module.split(".") if node.module else [])])
            full = [stem] + [f"{stem}.{a.name}" if stem else a.name for a in node.names]
        else:
            continue
        for name in filter(None, full):
            parts = name.split(".")
            names.update(".".join(parts[:i]) for i in range(1, len(parts) + 1))
    return names


def junk_files(task, files):
    """Files the reply wrote that nothing needs: not an expected file or a starter file, not a test, README/docs or
    project config, not inside an expected file's package folder, and not imported by any of those (a stray
    scratch.py or example_usage.py)."""
    expected = set(task["meta"]["files"]) | set(task.get("starter") or {})
    folders = {f.rsplit("/", 1)[0] + "/" for f in expected if "/" in f}

    def needed(path):
        return (path in expected or TEST_FILE.search(path) or TEST_HELPER.search(path) or DOC_FILE.search(path)
                or path in CONFIG_FILES or any(path.startswith(f) for f in folders))

    keep = {p for p in files if needed(p)}
    while True:
        imported = set()
        for path in keep | expected:
            text = files.get(path, (task.get("starter") or {}).get(path, ""))
            if path.endswith(".py"):
                imported |= imports_of(path, text)
        more = {p for p in files if p not in keep and p.endswith(".py") and module_of(p) in imported}
        if not more:
            break
        keep |= more
    return sorted(set(files) - keep)


def total_of(scores, junk):
    """0-100: the weighted sub-scores (parts that are None are left out and the rest scaled up), minus the junk
    penalty."""
    used = {k: w for k, w in WEIGHTS.items() if scores.get(k) is not None}
    total = 100 * sum(w * scores[k] for k, w in used.items()) / sum(used.values()) if used else 0.0
    return round(max(0.0, total - (JUNK_PENALTY if junk else 0)), 1)


def grade(task, files, workdir=None, reply=None):
    """Grade the files a reply wrote (put over the task's starter files, in fresh temporary folders under workdir).

    hidden     share of the hidden tests that pass (1.0 = solved)
    own_tests  0: the reply wrote no tests; 0.5: it did, but they fail; 1: they pass (run alone)
    lint       1 - issues/10 (at least 0) with the exam's fixed ruff rules, on the reply's .py files
    security   bandit on the reply's code (not its tests): 1 no findings, 0.5 only low ones, 0 medium or high;
               None when bandit is not installed (then left out of the total)
    explains   1 when the reply explains itself in more than 40 words outside the code (None: no reply given)
    junk       stray files (see junk_files): 5 points off the total
    """
    started = time.time()
    meta = task["meta"]
    files = {safe_path(p): t for p, t in files.items() if safe_path(p)}
    python = sorted(p for p in files if p.endswith(".py"))
    tests = sorted(p for p in python if TEST_FILE.search(p))
    code = [p for p in python if not TEST_FILE.search(p) and not TEST_HELPER.search(p)]
    result = {"task": task["id"], "kind": meta["kind"], "difficulty": meta["difficulty"], "files": sorted(files),
              "own_test_files": tests, "errors": []}
    base = Path(tempfile.mkdtemp(prefix="exam-", dir=workdir))
    try:
        hidden = hidden_tests(task, files, base)
        result.update(hidden=hidden["score"], hidden_passed=hidden["passed"], hidden_total=hidden["total"],
                      hidden_timed_out=hidden["timed_out"])
        outputs = {"hidden": hidden["output"]}

        result["own_tests"], outputs["own_tests"] = 0.0, "no test files"
        if tests:
            root, _ = workspace(base, "own", task.get("starter") or {}, files)
            own = run_pytest(root, tests, meta["timeout"], base / "own.xml", isolated=False)
            result["own_tests"] = 1.0 if own["exit_code"] == 0 and not own["timed_out"] else 0.5
            result.update(own_tests_passed=own["passed"], own_tests_total=own["total"])
            outputs["own_tests"] = own["output"]

        root, _ = workspace(base, "files", files)
        issues, outputs["lint"] = run_ruff(root, python) if python else (LINT_ZERO, "no .py files")
        result["lint_issues"] = issues
        result["lint"] = None if issues is None else round(max(0.0, 1 - issues / LINT_ZERO), 2)

        found, outputs["security"] = run_bandit(root, code) if code else ([], "no code files")
        result["security_issues"] = found
        if found is None:
            result["security"] = None
        elif not code:
            result["security"] = 0.0 if program("bandit", "bandit") else None
        else:
            levels = {f["severity"] for f in found}
            result["security"] = 0.0 if levels & {"MEDIUM", "HIGH"} else 0.5 if levels else 1.0
    finally:
        shutil.rmtree(base, ignore_errors=True)
    if reply is None:
        result.update(explains=None, explanation_words=None)
    else:
        words = read_reply(reply)["words"]
        result.update(explains=1.0 if words > EXPLAIN_WORDS else 0.0, explanation_words=words)
    result["junk_files"] = junk_files(task, files)
    result["junk"] = bool(result["junk_files"])
    result["total"] = total_of(result, result["junk"])
    result["solved"] = result["hidden"] == 1.0
    result["outputs"] = outputs
    result["seconds"] = round(time.time() - started, 2)
    return result


def grade_reply(task, reply, workdir=None):
    """Read the files out of a model's reply and grade them; also how the reply was written and how each file was
    found ("found_by": "named", or "fallback: ..." when no block named a path)."""
    parsed = read_reply(reply, task)
    graded = grade(task, parsed["files"], workdir, reply)
    graded.update(named_by=parsed["named_by"], found_by=parsed["found_by"], fallback=parsed["fallback"],
                  rejected=parsed["rejected"], unnamed_blocks=parsed["unnamed_blocks"], cut_off=parsed["cut_off"])
    return graded


# ---- checking the tasks themselves ----

def check_task(task, workdir=None):
    """Grade a task's reference as if a model wrote it (it must pass every hidden test), and run the hidden tests on
    the starter alone (they must not all pass: else there is nothing to do). A row with "problems"."""
    graded = grade(task, task["reference"], workdir)
    problems = []
    if not graded["solved"]:
        problems.append(f"the reference passes {graded['hidden_passed']} of {graded['hidden_total']} hidden tests:\n"
                        + graded["outputs"]["hidden"][-1500:])
    base = Path(tempfile.mkdtemp(prefix="exam-", dir=workdir))
    try:
        alone = hidden_tests(task, {}, base, "starter")
    finally:
        shutil.rmtree(base, ignore_errors=True)
    if alone["total"] and alone["passed"] == alone["total"]:
        problems.append("the starter files alone already pass every hidden test")
    return {"task": task["id"], "kind": task["meta"]["kind"], "difficulty": task["meta"]["difficulty"],
            "grade": graded, "starter_passed": alone["passed"], "starter_total": alone["total"],
            "problems": problems}


def check_references(tasks, workdir=None, workers=4):
    """check_task for every task (a few at a time). (rows, ok)."""
    with ThreadPoolExecutor(max_workers=workers) as pool:
        rows = list(pool.map(lambda t: check_task(t, workdir), tasks))
    return rows, all(not r["problems"] for r in rows)


def score_text(value):
    return "-" if value is None else f"{value:.0%}" if isinstance(value, float) else str(value)


def reference_table(rows):
    """The reference check as text lines."""
    lines = [(f"{'task':32s} {'kind':9s} {'diff':>4s} {'hidden':>7s} {'starter':>7s} {'total':>6s} {'own':>5s} "
              f"{'lint':>5s} {'sec':>5s}  result")]
    for r in rows:
        g = r["grade"]
        hidden, starter = f"{g['hidden_passed']}/{g['hidden_total']}", f"{r['starter_passed']}/{r['starter_total']}"
        lines.append(f"{r['task'][:32]:32s} {r['kind']:9s} {r['difficulty']:>4} {hidden:>7s} {starter:>7s} "
                     f"{g['total']:>6} {score_text(g['own_tests']):>5s} {score_text(g['lint']):>5s} "
                     f"{score_text(g['security']):>5s}  " + ("ok" if not r["problems"] else "FAIL"))
    for r in rows:
        for problem in r["problems"]:
            lines.append(f"\n{r['task']}: {problem}")
    return lines


# ---- coder/exam/exam.py ----
"""One model takes the engineer exam through Ollama (coder phase 4), and the report of a run.

coder/exam/build_exam.py puts this file into exam.ipynb after the Ollama helpers of coder/fixeval.py and after
coder/exam/grade.py (so the two "from coder..." imports below are left out there). coder/exam/run_exam.py prints
the report with table() and weakest(), so the notebook and the report agree.
"""

import time


SETTING = ("the task as a user writes it, the project's files and one line on the reply format; no system prompt, no "
           "checklist or style guide; greedy (temperature 0)")
PARTS = ("hidden", "own_tests", "lint", "security", "explains")
PART_NAMES = {"hidden": "hidden", "own_tests": "own tests", "lint": "lint", "security": "security",
              "explains": "explains"}


def ask_model(url, model, prompt, options, think=None, timeout=1200):
    """One chat request, no system message. The final answer only (a thinking model's thinking is kept apart)."""
    body = {"model": model, "messages": [{"role": "user", "content": prompt}], "stream": False, "options": options,
            "keep_alive": "30m", **({"think": think} if think else {})}
    started = time.time()
    data = post(url, "/api/chat", body, timeout=timeout)
    message = data.get("message") or {}
    return {"reply": message.get("content") or "", "thinking_chars": len(message.get("thinking") or ""),
            "seconds": round(time.time() - started, 2), "tokens": data.get("eval_count") or 0,
            "prompt_tokens": data.get("prompt_eval_count") or 0,
            "token_seconds": (data.get("eval_duration") or 0) / 1e9, "done_reason": data.get("done_reason")}


def average(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else None


def rounded(value, digits=3):
    return None if value is None else round(value, digits)


def attempt_scores(attempt):
    """One answer's scores (all 0 when the request or the grading failed)."""
    graded = attempt.get("grade") or {}
    if "total" not in graded:
        return {"total": 0.0, "solved": False, **{part: 0.0 for part in PARTS}}
    return {"total": graded["total"], "solved": graded["solved"], **{part: graded.get(part) for part in PARTS}}


def score_task(record):
    """A task's scores: the average over its answers (SAMPLES of them)."""
    scores = [attempt_scores(a) for a in record["attempts"]]
    record["total"] = rounded(average([s["total"] for s in scores]) or 0.0, 1)
    record["solved"] = bool(scores) and all(s["solved"] for s in scores)
    record["solve_rate"] = rounded(average([float(s["solved"]) for s in scores]) or 0.0)
    for part in PARTS:
        record[part] = rounded(average([s[part] for s in scores]))


def take_exam(url, model, tasks, entry, minutes, options, think=None, thinking_num_predict=None, samples=1,
              save=lambda: None, log=print, models_dir="/tmp", workdir=None):
    """Download and load one model, give it every task, grade each reply; fill entry (a dict in the results) as it
    goes, so a stop part-way keeps everything done until then. Stops when `minutes` have passed (download and
    loading included), or after 3 failed requests in a row. A thinking model gets thinking_num_predict tokens
    (its thinking and its answer) instead of options' num_predict."""
    deadline = time.time() + minutes * 60
    level = think_for(model, think)
    options = {**options, **({"num_predict": thinking_num_predict} if level and thinking_num_predict else {})}
    entry.update(model=model, think=level, options=options, samples=samples, total=len(tasks), done=0, solved=0,
                 score=None, score_done=None, pulled=False, pull_seconds=None, size_gb=None, load_seconds=None,
                 processor=None, ollama_ps=None, gpu_memory=None, avg_seconds=None, tokens_per_s=None, cut_off=0,
                 no_files=0, fallback=0, junk=0, stopped_by_time=False, minutes=None,
                 disk_free_gb_before=disk_free_gb(models_dir), errors=[], tasks=[])
    started = time.time()

    log(f"\n== {model}: download")
    entry["size_gb"] = pull(url, model, log)
    entry.update(pulled=True, pull_seconds=round(time.time() - started, 1))
    log(f"   {model}: {entry['size_gb']} GB in {entry['pull_seconds']} s")
    save()

    loading = time.time()
    post(url, "/api/generate", {"model": model, "options": options, "keep_alive": "30m"}, timeout=900)  # only load
    entry["load_seconds"] = round(time.time() - loading, 1)
    log(f"   {model}: loaded in {entry['load_seconds']} s")
    save()

    failures_in_a_row = 0
    for i, task in enumerate(tasks, 1):
        if time.time() > deadline:
            entry["stopped_by_time"] = True
            log(f"   {model}: stopped at {minutes} minutes, after {entry['done']} of {len(tasks)} tasks")
            break
        meta = task["meta"]
        record = {"task": task["id"], "kind": meta["kind"], "difficulty": meta["difficulty"], "attempts": []}
        for sample in range(samples):
            attempt = {"sample": sample}
            try:
                answer = ask_model(url, model, prompt_for(task), {**options, **({"seed": sample} if samples > 1
                                                                                  else {})}, level)
            except Exception as e:  # noqa: BLE001 - count it as 0 and go on
                attempt["error"] = f"request failed: {e!r}"[:300]
                entry["errors"].append(f"{task['id']}: {attempt['error']}")
                failures_in_a_row += 1
            else:
                failures_in_a_row = 0
                attempt.update(answer)
                try:
                    attempt["grade"] = grade_reply(task, answer["reply"], workdir)
                except Exception as e:  # noqa: BLE001 - a grading bug must not stop the exam
                    attempt["error"] = f"grading failed: {e!r}"[:300]
                    entry["errors"].append(f"{task['id']}: {attempt['error']}")
            record["attempts"].append(attempt)
        score_task(record)
        entry["tasks"].append(record)
        entry["done"] += 1
        graded = (record["attempts"][0].get("grade") or {}) if record["attempts"] else {}
        log(f"   {i:3d}/{len(tasks)} {task['id'][:30]:30s} {record['total']:5.1f}  "
            f"hidden {graded.get('hidden_passed', 0)}/{graded.get('hidden_total', 0)}  "
            + "  ".join(f"{PART_NAMES[p]} {'-' if record[p] is None else f'{record[p]:.0%}'}" for p in PARTS[1:])
            + f"  {record['attempts'][0].get('seconds', '-')} s"
            + (f"  {record['attempts'][0]['error'][:80]}" if record["attempts"][0].get("error") else ""))
        if i == 1:  # where it runs, once it has answered (`ollama ps`)
            entry.update(processor=processor(url, model), ollama_ps=text_of("ollama ps") or None,
                         gpu_memory=text_of("nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader")
                         .replace("\n", "; ") or None)
            log(f"   {model} runs on: {entry['processor']}")
        summarise(entry, started)
        save()  # after every task: nothing is lost if the run stops
        if failures_in_a_row >= 3:
            entry["errors"].append("3 requests in a row failed: stopped this model")
            log(f"   {model}: 3 requests in a row failed, going on to the next model")
            break
    summarise(entry, started)
    save()
    return entry


def summarise(entry, started=None):
    """A model's totals from its task records."""
    tasks = entry.get("tasks") or []
    attempts = [a for t in tasks for a in t["attempts"]]
    answered = [a for a in attempts if "seconds" in a]
    graded = [a["grade"] for a in attempts if a.get("grade")]
    token_seconds = sum(a["token_seconds"] for a in answered)
    entry["done"] = len(tasks)
    entry["solved"] = sum(t["solved"] for t in tasks)
    entry["score_done"] = rounded(average([t["total"] for t in tasks]), 1)
    entry["score"] = entry["score_done"] if tasks and entry["done"] == entry.get("total") else None
    for part in PARTS:
        entry[part] = rounded(average([t[part] for t in tasks]))
    kinds = {}
    for t in tasks:
        kinds.setdefault(t["kind"], []).append(t)
    entry["by_kind"] = {k: {"tasks": len(ts), "total": rounded(average([t["total"] for t in ts]), 1),
                            "solved": sum(t["solved"] for t in ts)} for k, ts in sorted(kinds.items())}
    entry["avg_seconds"] = rounded(average([a["seconds"] for a in answered]), 2)
    entry["tokens_per_s"] = round(sum(a["tokens"] for a in answered) / token_seconds, 1) if token_seconds else None
    entry["cut_off"] = sum(a.get("done_reason") == "length" for a in answered)
    entry["no_files"] = sum(not g.get("files") for g in graded)
    entry["fallback"] = sum(any(how.startswith("fallback") for how in (g.get("found_by") or {}).values())
                            for g in graded)  # replies whose files only the fallback found (no block named a path)
    entry["junk"] = sum(bool(g.get("junk")) for g in graded)
    if started is not None:
        entry["minutes"] = round((time.time() - started) / 60, 1)


# ---- the report ----

def percent(value):
    return "-" if value is None else f"{value:.0%}"


def table(results):
    """The results as text lines: one row per model, then each model's scores by kind of task."""
    lines = [(f"{'model':22s} {'total':>5s} {'solved':>12s} {'hidden':>6s} {'own':>4s} {'lint':>4s} {'sec':>4s} "
              f"{'expl':>4s} {'s/task':>6s}  note")]
    for m in results.get("models") or []:
        done, total = m.get("done") or 0, m.get("total") or 0
        score = m.get("score") if m.get("score") is not None else m.get("score_done")
        note = []
        if not m.get("pulled"):
            note.append("download failed")
        elif m.get("load_seconds") is None:
            note.append("did not load")
        if m.get("stopped_by_time"):
            note.append(f"stopped at the time limit after {done} of {total} tasks")
        elif done and done < total:
            note.append(f"stopped after {done} of {total} tasks")
        for key, words in (("cut_off", "cut off"), ("no_files", "with no files"),
                           ("fallback", "read by the fallback (no file named)"), ("junk", "with stray files")):
            if m.get(key):
                note.append(f"{m[key]} {words}")
        if m.get("think"):
            note.append(f"think {m['think']}")
        if m.get("processor") and m["processor"] != "100% GPU":
            note.append(m["processor"])
        if m.get("errors"):
            note.append(f"{len(m['errors'])} error" + ("s" if len(m["errors"]) > 1 else ""))
        solved = f"{m.get('solved') or 0}/{done}" + ("" if done == total else f" of {total}")
        lines.append(f"{m.get('model', '?')[:22]:22s} {'-' if score is None else f'{score:.1f}':>5s} {solved:>12s} "
                     f"{percent(m.get('hidden')):>6s} {percent(m.get('own_tests')):>4s} {percent(m.get('lint')):>4s} "
                     f"{percent(m.get('security')):>4s} {percent(m.get('explains')):>4s} "
                     f"{m.get('avg_seconds') or '-':>6}  {'; '.join(note)}".rstrip())
    for m in results.get("models") or []:
        if m.get("by_kind"):
            lines.append(f"{m['model']} by kind: " + ", ".join(
                f"{kind} {k['total']} ({k['solved']}/{k['tasks']} solved)" for kind, k in m["by_kind"].items()))
    lines.append(f"(total 0-100 = hidden tests {WEIGHTS['hidden']} + own tests {WEIGHTS['own_tests']} + lint "
                 f"{WEIGHTS['lint']} + security {WEIGHTS['security']} + explanation {WEIGHTS['explains']}, "
                 f"-{JUNK_PENALTY} for stray files; solved = every hidden test passes; of the tasks done)")
    return lines


def why(record):
    """In a few words, what went wrong with a task's (first) answer."""
    attempt = (record.get("attempts") or [{}])[0]
    if attempt.get("error"):
        return attempt["error"][:160]
    g = attempt.get("grade") or {}
    reasons = []
    if not g.get("files"):
        notes = [n for n in ("cut off" if attempt.get("done_reason") == "length" else None, g.get("fallback")) if n]
        reasons.append("no files in the reply" + (f" ({'; '.join(notes)})" if notes else ""))
    elif g.get("hidden_timed_out"):
        reasons.append("the hidden tests ran out of time")
    elif g.get("hidden", 0) < 1:
        failed = [line.strip() for line in (g.get("outputs") or {}).get("hidden", "").splitlines()
                  if line.startswith(("FAILED", "ERROR"))]
        reasons.append("; ".join(f[:110] for f in failed[:2]) or f"{g.get('hidden_passed')}/{g.get('hidden_total')}")
    if any(how.startswith("fallback") for how in (g.get("found_by") or {}).values()):
        reasons.append("files found by the fallback (no file named)")
    if g.get("files") and not g.get("own_test_files"):
        reasons.append("no tests")
    elif g.get("own_tests") == 0.5:
        reasons.append("its own tests fail")
    if g.get("lint_issues"):
        reasons.append(f"{g['lint_issues']} lint issues")
    if g.get("security_issues"):
        reasons.append("bandit: " + ", ".join(sorted({f"{s['test_id']} {s['severity']}"
                                                      for s in g["security_issues"]})))
    if g.get("explains") == 0:
        reasons.append(f"no explanation (under {EXPLAIN_WORDS} words)")
    if g.get("junk_files"):
        reasons.append("stray files: " + ", ".join(g["junk_files"][:3]))
    return "; ".join(reasons) or "-"


def weakest(entry, n=5):
    """The n tasks this model did worst on, as text lines."""
    worst = sorted(entry.get("tasks") or [], key=lambda t: (t["total"], t["task"]))[:n]
    lines = []
    for t in worst:
        g = (t["attempts"][0].get("grade") or {}) if t["attempts"] else {}
        lines.append(f"   {t['task']} ({t['kind']}, difficulty {t['difficulty']}): total {t['total']}, hidden "
                     f"{g.get('hidden_passed', 0)}/{g.get('hidden_total', 0)}: {why(t)}")
    return lines


RESULTS_FILE = 'exam.json'

RESULTS = {"exam": "engineer exam (coder phase 4)", "setting": SETTING, "reply_format": REPLY_FORMAT,
           "weights": WEIGHTS, "junk_penalty": JUNK_PENALTY, "ruff_config": RUFF_CONFIG, "options": OPTIONS,
           "think": THINK, "thinking_num_predict": THINKING_NUM_PREDICT, "samples": SAMPLES,
           "minutes_per_model": MINUTES_PER_MODEL, "limit": LIMIT, "models_asked": MODELS, "models_dir": MODELS_DIR, "ollama_version": None, "gpu": None, "gpus": None,
           "disk": None, "pip": None, "checkers": None, "tasks_total": None, "tasks_used": None, "task_ids": None,
           "kinds": None, "reference_check": None, "tasks_left_out": [], "models": [], "errors": []}
OLLAMA_LOG = "/tmp/ollama.log"  # what `ollama serve` prints (cell 4)


def save_results():
    """Write RESULTS to exam.json. Done after every stage, so a stop part-way loses nothing."""
    if os.path.isdir(WORKING_DIR):
        with open(os.path.join(WORKING_DIR, "exam.json"), "w", encoding="utf-8") as fh:
            json.dump(RESULTS, fh, ensure_ascii=False, indent=1)


def failed(stage, error):
    """Record a failed stage and go on (Kaggle's Run all stops at the first error that is not caught)."""
    message = f"{stage}: {error!r}" if isinstance(error, Exception) else f"{stage}: {error}"
    print("FAILED", message)
    RESULTS["errors"].append(message)
    save_results()


def sh(command, timeout=120):
    """What a shell command printed, or "" when the command is missing or fails."""
    try:
        return subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout).stdout.strip()
    except Exception:
        return ""

In [ ]:
#@title 3. Install the checkers: pytest, ruff, bandit (about 1 minute)
# pytest runs the tests, ruff lints, bandit looks for security problems (optional: without it, security is left
# out of the total).
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "pytest", "ruff",
                      "bandit"], capture_output=True, text=True, timeout=900)
RESULTS["pip"] = (pip.stdout + pip.stderr).strip()[-1500:] or "ok"
importlib.invalidate_caches()
RESULTS["checkers"] = checker_versions()
print(RESULTS["checkers"])
if not RESULTS["checkers"]["pytest"]:
    failed("install pytest", RESULTS["pip"])
    raise RuntimeError("pytest did not install, so no reply can be tested (see above)")
for name in ("ruff", "bandit"):
    if not RESULTS["checkers"][name]:
        print(f"{name} did not install: its part is left out of the total")
save_results()

In [ ]:
#@title 4. Install Ollama and start it (about 1–2 minutes)
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ["OLLAMA_MODELS"] = MODELS_DIR  # where `ollama serve` keeps the downloads
os.environ["OLLAMA_NUM_PARALLEL"] = "1"  # one question at a time: less GPU memory

OLLAMA_URL = "http://127.0.0.1:11434"


def ollama_version():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=5) as resp:
            return json.load(resp).get("version")
    except Exception:
        return None


if not ollama_version():
    if not shutil.which("ollama"):
        # zstd unpacks Ollama's download (.tar.zst). The download already holds the CUDA libraries; lspci is
        # left out on purpose: with it (and no nvidia-smi) install.sh would try to install NVIDIA drivers.
        if not shutil.which("zstd"):
            sh("apt-get update -qq; DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd", timeout=600)
        install = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True,
                                 capture_output=True, text=True, timeout=1200)
        print("\n".join((install.stdout + install.stderr).replace("\r", "\n").splitlines()[-15:]))
        if not shutil.which("ollama"):
            failed("install Ollama", (install.stdout + install.stderr)[-1500:])
            raise RuntimeError("Ollama did not install (see above)")
    print(sh("ollama --version 2>&1"))
    log = open(OLLAMA_LOG, "w")  # stays open: the server writes to it
    server = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
                              stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_version() or server.poll() is not None:
            break
        time.sleep(1)
if not ollama_version():
    tail = sh(f"tail -n 40 {OLLAMA_LOG}")
    print(tail)
    failed("start Ollama", tail[-1500:])
    raise RuntimeError("Ollama did not start (its log is above)")
RESULTS["ollama_version"] = ollama_version()
RESULTS["gpu"] = (sh("nvidia-smi --query-gpu=name --format=csv,noheader").splitlines() or [None])[0]
print("Ollama", RESULTS["ollama_version"], "is running. GPU:", RESULTS["gpu"])
save_results()

RESULTS["gpus"] = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") or None
RESULTS["disk"] = sh(f"df -h {MODELS_DIR} {WORKING_DIR} 2>/dev/null") or None
print("GPUs:", RESULTS["gpus"])
print(RESULTS["disk"])
save_results()

In [ ]:
#@title 5. The tasks (no need to read; built from coder/exam/tasks)
# Each: id, task_md (what the model is asked), meta, starter (files it is shown), hidden (the tests it
# never sees) and reference (a solution, used only to check the checker).
EXAM_TASKS = [
    json.loads('{"hidden": "import time\\n\\nimport pytest\\n\\nimport inventory\\nfrom inventory import InsufficientStock, Ledger\\n\\n\\ndef test_receive_and_stock():\\n    led = Ledger()\\n    led.receive(\\"ATTA-5KG\\", 10)\\n    led.receive(\\"ATTA-5KG\\", 5)\\n    assert led.stock(\\"ATTA-5KG\\") == 15\\n\\n\\ndef test_unknown_sku_is_zero():\\n    led = Ledger()\\n    assert led.stock(\\"NEVER-SEEN\\") == 0\\n    assert led.history(\\"NEVER-SEEN\\") == []\\n\\n\\ndef test_issue_reduces_stock():\\n    led = Ledger()\\n    led.receive(\\"SALT-1KG\\", 10)\\n    led.issue(\\"SALT-1KG\\", 4)\\n    assert led.stock(\\"SALT-1KG\\") == 6\\n    led.issue(\\"SALT-1KG\\", 6)\\n    assert led.stock(\\"SALT-1KG\\") == 0\\n\\n\\ndef test_insufficient_stock_leaves_state_unchanged():\\n    led = Ledger()\\n    led.receive(\\"OIL-1L\\", 3)\\n    with pytest.raises(InsufficientStock):\\n        led.issue(\\"OIL-1L\\", 4)\\n    assert led.stock(\\"OIL-1L\\") == 3\\n    assert len(led.history(\\"OIL-1L\\")) == 1\\n    with pytest.raises(InsufficientStock):\\n        led.issue(\\"NEW-SKU\\", 1)\\n    assert led.stock(\\"NEW-SKU\\") == 0\\n\\n\\ndef test_insufficient_stock_is_an_exception_class():\\n    assert issubclass(inventory.InsufficientStock, Exception)\\n\\n\\n@pytest.mark.parametrize(\\"bad\\", [0, -1, -100, 1.5])\\n@pytest.mark.parametrize(\\"method\\", [\\"receive\\", \\"issue\\"])\\ndef test_invalid_quantity_raises_value_error(method, bad):\\n    led = Ledger()\\n    led.receive(\\"X\\", 10)\\n    with pytest.raises(ValueError):\\n        getattr(led, method)(\\"X\\", bad)\\n    assert led.stock(\\"X\\") == 10\\n    assert len(led.history(\\"X\\")) == 1\\n\\n\\n@pytest.mark.parametrize(\\"bad\\", [\\"5\\", None])\\ndef test_non_number_quantity_rejected(bad):\\n    led = Ledger()\\n    with pytest.raises((ValueError, TypeError)):\\n        led.receive(\\"X\\", bad)\\n    assert led.stock(\\"X\\") == 0\\n\\n\\ndef test_duplicate_move_id_counts_once():\\n    led = Ledger()\\n    led.receive(\\"SUGAR-1KG\\", 10, move_id=\\"m-1\\")\\n    led.receive(\\"SUGAR-1KG\\", 10, move_id=\\"m-1\\")\\n    assert led.stock(\\"SUGAR-1KG\\") == 10\\n    led.issue(\\"SUGAR-1KG\\", 3, move_id=\\"m-2\\")\\n    led.issue(\\"SUGAR-1KG\\", 3, move_id=\\"m-2\\")\\n    led.issue(\\"SUGAR-1KG\\", 3, move_id=\\"m-2\\")\\n    assert led.stock(\\"SUGAR-1KG\\") == 7\\n    assert len(led.history(\\"SUGAR-1KG\\")) == 2\\n\\n\\ndef test_moves_without_id_always_count():\\n    led = Ledger()\\n    led.receive(\\"A\\", 1)\\n    led.receive(\\"A\\", 1)\\n    assert led.stock(\\"A\\") == 2\\n\\n\\ndef test_failed_move_does_not_burn_its_id():\\n    led = Ledger()\\n    with pytest.raises(ValueError):\\n        led.receive(\\"A\\", 0, move_id=\\"r-1\\")\\n    led.receive(\\"A\\", 5, move_id=\\"r-1\\")\\n    assert led.stock(\\"A\\") == 5\\n    with pytest.raises(InsufficientStock):\\n        led.issue(\\"A\\", 8, move_id=\\"i-1\\")\\n    led.receive(\\"A\\", 5)\\n    led.issue(\\"A\\", 8, move_id=\\"i-1\\")  # retried after restock: must apply now\\n    assert led.stock(\\"A\\") == 2\\n\\n\\ndef test_history_order_and_content():\\n    led = Ledger()\\n    led.receive(\\"DAL-1KG\\", 7, move_id=\\"a\\")\\n    led.issue(\\"DAL-1KG\\", 2, move_id=\\"b\\")\\n    led.receive(\\"DAL-1KG\\", 1)\\n    hist = led.history(\\"DAL-1KG\\")\\n    assert [(h[\\"move_id\\"], h[\\"kind\\"], h[\\"qty\\"]) for h in hist] == [\\n        (\\"a\\", \\"receive\\", 7),\\n        (\\"b\\", \\"issue\\", 2),\\n        (None, \\"receive\\", 1),\\n    ]\\n\\n\\ndef test_history_cannot_be_changed_from_outside():\\n    led = Ledger()\\n    led.receive(\\"A\\", 5, move_id=\\"a\\")\\n    hist = led.history(\\"A\\")\\n    hist.clear()\\n    assert len(led.history(\\"A\\")) == 1\\n    assert led.stock(\\"A\\") == 5\\n\\n\\ndef test_ledgers_do_not_share_state():\\n    a = Ledger()\\n    b = Ledger()\\n    a.receive(\\"A\\", 5, move_id=\\"m\\")\\n    b.receive(\\"A\\", 2, move_id=\\"m\\")\\n    assert a.stock(\\"A\\") == 5\\n    assert b.stock(\\"A\\") == 2\\n    assert len(b.history(\\"A\\")) == 1\\n\\n\\ndef test_many_moves_are_fast():\\n    led = Ledger()\\n    start = time.perf_counter()\\n    for i in range(50_000):\\n        led.receive(f\\"SKU-{i % 100}\\", 2, move_id=f\\"r{i}\\")\\n    for i in range(50_000):\\n        led.issue(f\\"SKU-{i % 100}\\", 1, move_id=f\\"i{i}\\")\\n    elapsed = time.perf_counter() - start\\n    assert led.stock(\\"SKU-7\\") == 500\\n    assert len(led.history(\\"SKU-7\\")) == 1000\\n    assert elapsed < 3.0\\n", "id": "01-inventory-ledger", "meta": {"difficulty": 2, "files": ["inventory.py"], "kind": "build", "timeout": 60}, "reference": {"inventory.py": "\\"\\"\\"Stock ledger for a small shop: receive and issue stock, idempotent by move id.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import defaultdict\\nfrom numbers import Integral\\n\\n\\nclass InsufficientStock(Exception):\\n    \\"\\"\\"Raised when an issue asks for more units than are in stock.\\"\\"\\"\\n\\n\\ndef _check_qty(qty: object) -> int:\\n    \\"\\"\\"The quantity as an int; ValueError unless it is a whole number above zero.\\"\\"\\"\\n    if isinstance(qty, Integral) and not isinstance(qty, bool) and qty > 0:\\n        return int(qty)\\n    raise ValueError(f\\"quantity must be a whole number above zero, got {qty!r}\\")\\n\\n\\nclass Ledger:\\n    \\"\\"\\"In-memory stock ledger. Each instance keeps its own state.\\"\\"\\"\\n\\n    def __init__(self) -> None:\\n        self._stock: dict[str, int] = defaultdict(int)\\n        self._history: dict[str, list[dict]] = defaultdict(list)\\n        self._seen_moves: set = set()\\n\\n    def receive(self, sku: str, qty: int, move_id: object = None) -> int:\\n        \\"\\"\\"Add qty units of sku. Returns the new stock level.\\"\\"\\"\\n        return self._apply(sku, qty, move_id, \\"receive\\")\\n\\n    def issue(self, sku: str, qty: int, move_id: object = None) -> int:\\n        \\"\\"\\"Take qty units of sku out. Returns the new stock level.\\"\\"\\"\\n        return self._apply(sku, qty, move_id, \\"issue\\")\\n\\n    def stock(self, sku: str) -> int:\\n        return self._stock.get(sku, 0)\\n\\n    def history(self, sku: str) -> list[dict]:\\n        return [dict(move) for move in self._history.get(sku, ())]\\n\\n    def _apply(self, sku: str, qty: int, move_id: object, kind: str) -> int:\\n        qty = _check_qty(qty)\\n        if move_id is not None and move_id in self._seen_moves:\\n            return self.stock(sku)\\n        current = self._stock.get(sku, 0)\\n        if kind == \\"issue\\":\\n            if qty > current:\\n                raise InsufficientStock(f\\"{sku}: asked for {qty}, only {current} in stock\\")\\n            new = current - qty\\n        else:\\n            new = current + qty\\n        self._stock[sku] = new\\n        self._history[sku].append({\\"move_id\\": move_id, \\"kind\\": kind, \\"qty\\": qty})\\n        if move_id is not None:\\n            self._seen_moves.add(move_id)\\n        return new\\n"}, "starter": {}, "task_md": "I run a small kirana shop and want to stop tracking stock in a notebook. Please write `inventory.py` with a class `Ledger`.\\n`receive(sku, qty, move_id=None)` adds stock, `issue(sku, qty, move_id=None)` takes stock out, `stock(sku)` returns the current quantity (0 for a product we have never seen), and `history(sku)` returns that product\'s moves, oldest first, as dicts with `move_id`, `kind` (`\\"receive\\"` or `\\"issue\\"`) and `qty`.\\nQuantities are whole units, so raise `ValueError` for an invalid quantity.\\nIf there isn\'t enough stock for an issue, raise `InsufficientStock` (define it in the module) and leave everything as it was.\\nOur billing app retries when the network is bad, so the same `move_id` can arrive more than once; it must only count once."}'),
    json.loads('{"hidden": "import copy\\nimport time\\nfrom decimal import Decimal\\n\\nimport pytest\\n\\nfrom gst import invoice_totals\\n\\n\\ndef D(s):\\n    return Decimal(s)\\n\\n\\ndef test_single_line():\\n    r = invoice_totals([{\\"price\\": \\"100\\", \\"qty\\": 2, \\"gst_rate\\": 18}])\\n    assert r == {\\"subtotal\\": D(\\"200\\"), \\"gst\\": D(\\"36\\"), \\"total\\": D(\\"236\\"), \\"payable\\": D(\\"236\\")}\\n    assert all(isinstance(v, Decimal) for v in r.values())\\n\\n\\ndef test_mixed_input_types():\\n    r = invoice_totals([\\n        {\\"price\\": \\"199.50\\", \\"qty\\": 1, \\"gst_rate\\": 5},\\n        {\\"price\\": 40, \\"qty\\": 3, \\"gst_rate\\": 0},\\n        {\\"price\\": D(\\"12.25\\"), \\"qty\\": 4, \\"gst_rate\\": 12},\\n    ])\\n    # 199.50 + 120 + 49.00 ; gst 9.975->9.98 + 0 + 5.88\\n    assert r[\\"subtotal\\"] == D(\\"368.50\\")\\n    assert r[\\"gst\\"] == D(\\"15.86\\")\\n    assert r[\\"total\\"] == D(\\"384.36\\")\\n    assert r[\\"payable\\"] == D(\\"384\\")\\n\\n\\ndef test_gst_rounds_half_up_to_paisa():\\n    # 0.25 * 18% = 0.045 -> 0.05 (half up); binary floats give 0.04\\n    r = invoice_totals([{\\"price\\": \\"0.25\\", \\"qty\\": 1, \\"gst_rate\\": 18}])\\n    assert r[\\"gst\\"] == D(\\"0.05\\")\\n    r = invoice_totals([{\\"price\\": \\"1.15\\", \\"qty\\": 1, \\"gst_rate\\": 5}])\\n    # 0.0575 -> 0.06\\n    assert r[\\"gst\\"] == D(\\"0.06\\")\\n\\n\\ndef test_gst_is_rounded_per_line():\\n    line = {\\"price\\": \\"0.25\\", \\"qty\\": 1, \\"gst_rate\\": 18}\\n    r = invoice_totals([line, dict(line)])\\n    assert r[\\"gst\\"] == D(\\"0.10\\")  # 0.05 + 0.05, not round(0.09)\\n    assert r[\\"total\\"] == D(\\"0.60\\")\\n\\n\\ndef test_no_float_drift():\\n    r = invoice_totals([{\\"price\\": \\"0.1\\", \\"qty\\": 3, \\"gst_rate\\": 0}, {\\"price\\": \\"0.2\\", \\"qty\\": 1, \\"gst_rate\\": 0}])\\n    assert r[\\"subtotal\\"] == D(\\"0.50\\")\\n    assert str(r[\\"total\\"]) == \\"0.50\\"\\n\\n\\n@pytest.mark.parametrize(\\"price, payable\\", [(\\"100.50\\", \\"101\\"), (\\"100.49\\", \\"100\\"), (\\"102.50\\", \\"103\\"), (\\"99.99\\", \\"100\\")])\\ndef test_payable_rounds_half_up_to_rupee(price, payable):\\n    r = invoice_totals([{\\"price\\": price, \\"qty\\": 1, \\"gst_rate\\": 0}])\\n    assert r[\\"payable\\"] == D(payable)\\n\\n\\ndef test_money_values_have_paise():\\n    r = invoice_totals([{\\"price\\": \\"10\\", \\"qty\\": 1, \\"gst_rate\\": 5}])\\n    assert r[\\"total\\"] == D(\\"10.50\\")\\n    assert r[\\"total\\"].as_tuple().exponent == -2\\n    assert r[\\"gst\\"].as_tuple().exponent == -2\\n\\n\\ndef test_empty_invoice_is_zero():\\n    r = invoice_totals([])\\n    assert r == {\\"subtotal\\": 0, \\"gst\\": 0, \\"total\\": 0, \\"payable\\": 0}\\n\\n\\n@pytest.mark.parametrize(\\"rate\\", [15, -5, 100, \\"eighteen\\", None])\\ndef test_unknown_rate_raises(rate):\\n    with pytest.raises(ValueError):\\n        invoice_totals([{\\"price\\": \\"10\\", \\"qty\\": 1, \\"gst_rate\\": rate}])\\n\\n\\n@pytest.mark.parametrize(\\"qty\\", [0, -1, 1.5, \\"abc\\"])\\ndef test_bad_quantity_raises(qty):\\n    with pytest.raises(ValueError):\\n        invoice_totals([{\\"price\\": \\"10\\", \\"qty\\": qty, \\"gst_rate\\": 5}])\\n\\n\\n@pytest.mark.parametrize(\\"price\\", [\\"abc\\", \\"\\", \\"-5\\", \\"NaN\\", None])\\ndef test_bad_price_raises(price):\\n    with pytest.raises(ValueError):\\n        invoice_totals([{\\"price\\": price, \\"qty\\": 1, \\"gst_rate\\": 5}])\\n\\n\\ndef test_input_is_not_modified():\\n    lines = [{\\"price\\": \\"99.99\\", \\"qty\\": 2, \\"gst_rate\\": 12}, {\\"price\\": 5, \\"qty\\": 1, \\"gst_rate\\": 0}]\\n    before = copy.deepcopy(lines)\\n    invoice_totals(lines)\\n    assert lines == before\\n\\n\\ndef test_same_input_same_output():\\n    lines = [{\\"price\\": \\"33.33\\", \\"qty\\": 3, \\"gst_rate\\": 28}]\\n    assert invoice_totals(lines) == invoice_totals(lines)\\n\\n\\ndef test_large_invoice_is_fast_and_exact():\\n    lines = [{\\"price\\": \\"0.10\\", \\"qty\\": 1, \\"gst_rate\\": 0}] * 100_000\\n    start = time.perf_counter()\\n    r = invoice_totals(lines)\\n    assert time.perf_counter() - start < 3.0\\n    assert r[\\"subtotal\\"] == D(\\"10000.00\\")\\n", "id": "02-gst-invoice", "meta": {"difficulty": 2, "files": ["gst.py"], "kind": "build", "timeout": 60}, "reference": {"gst.py": "\\"\\"\\"Invoice totals with Indian GST, using exact decimal arithmetic.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom decimal import ROUND_HALF_UP, Decimal, InvalidOperation\\nfrom numbers import Integral\\n\\nGST_RATES = frozenset({0, 5, 12, 18, 28})\\nPAISA = Decimal(\\"0.01\\")\\nRUPEE = Decimal(1)\\n\\n\\ndef _money(value: Decimal) -> Decimal:\\n    return value.quantize(PAISA, rounding=ROUND_HALF_UP)\\n\\n\\ndef _price(raw: object) -> Decimal:\\n    \\"\\"\\"The price as a Decimal. Floats are refused: they cannot hold paise exactly.\\"\\"\\"\\n    price = None\\n    if isinstance(raw, (str, int, Decimal)) and not isinstance(raw, bool):\\n        try:\\n            price = Decimal(raw.strip() if isinstance(raw, str) else raw)\\n        except InvalidOperation:\\n            price = None\\n    if price is None or not price.is_finite() or price < 0:\\n        raise ValueError(f\\"bad price {raw!r} (give a string, int or Decimal of at least 0)\\")\\n    return price\\n\\n\\ndef _qty(raw: object) -> int:\\n    if isinstance(raw, bool) or not isinstance(raw, Integral) or raw <= 0:\\n        raise ValueError(f\\"bad quantity {raw!r}\\")\\n    return int(raw)\\n\\n\\ndef _rate(raw: object) -> int:\\n    if isinstance(raw, bool) or raw not in GST_RATES:\\n        raise ValueError(f\\"unknown GST rate {raw!r}\\")\\n    return int(raw)\\n\\n\\ndef invoice_totals(lines: list[dict]) -> dict[str, Decimal]:\\n    \\"\\"\\"Return subtotal, gst, total and payable (nearest rupee) for the invoice lines.\\"\\"\\"\\n    subtotal = Decimal(0)\\n    gst = Decimal(0)\\n    for line in lines:\\n        price = _price(line[\\"price\\"])\\n        qty = _qty(line[\\"qty\\"])\\n        rate = _rate(line[\\"gst_rate\\"])\\n        amount = _money(price * qty)\\n        subtotal += amount\\n        gst += _money(amount * rate / 100)\\n    subtotal = _money(subtotal)\\n    gst = _money(gst)\\n    total = subtotal + gst\\n    payable = total.quantize(RUPEE, rounding=ROUND_HALF_UP)\\n    return {\\"subtotal\\": subtotal, \\"gst\\": gst, \\"total\\": total, \\"payable\\": payable}\\n"}, "starter": {}, "task_md": "We need the bill totals for our shop counter. Please write `gst.py` with `invoice_totals(lines)`.\\nEach line is a dict with `price` (unit price before tax in rupees; it can come as a string like `\\"199.50\\"`, an int or a `Decimal`), `qty` (int) and `gst_rate` (percent, one of 0, 5, 12, 18 or 28).\\nIt returns a dict with `subtotal`, `gst`, `total` and `payable`, all `Decimal`. GST is worked out per line and rounded to the paisa (half up, so 0.005 goes up), `total` is subtotal + GST, and `payable` is the total rounded to the nearest rupee (also half up), like on our printed bills.\\nRaise `ValueError` for an unknown GST rate, a bad quantity or a bad price."}'),
    json.loads('{"hidden": "import time\\nfrom decimal import Decimal\\n\\nimport pytest\\n\\nfrom order_import import import_orders\\n\\nHEADER = \\"order_id,customer,sku,qty,price\\\\n\\"\\n\\n\\ndef write(tmp_path, text, name=\\"orders.csv\\", encoding=\\"utf-8\\"):\\n    p = tmp_path / name\\n    p.write_text(text, encoding=encoding)\\n    return p\\n\\n\\ndef test_good_file(tmp_path):\\n    p = write(tmp_path, HEADER + \\"A1,Ravi,ATTA-5KG,2,245.50\\\\nA2,Meena,SALT-1KG,1,28\\\\n\\")\\n    orders, errors = import_orders(p)\\n    assert errors == []\\n    assert orders == [\\n        {\\"order_id\\": \\"A1\\", \\"customer\\": \\"Ravi\\", \\"sku\\": \\"ATTA-5KG\\", \\"qty\\": 2, \\"price\\": Decimal(\\"245.50\\")},\\n        {\\"order_id\\": \\"A2\\", \\"customer\\": \\"Meena\\", \\"sku\\": \\"SALT-1KG\\", \\"qty\\": 1, \\"price\\": Decimal(\\"28\\")},\\n    ]\\n    assert isinstance(orders[0][\\"qty\\"], int)\\n    assert isinstance(orders[0][\\"price\\"], Decimal)\\n\\n\\ndef test_accepts_str_path(tmp_path):\\n    p = write(tmp_path, HEADER + \\"A1,Ravi,ATTA-5KG,2,245.50\\\\n\\")\\n    orders, errors = import_orders(str(p))\\n    assert len(orders) == 1 and errors == []\\n\\n\\ndef test_column_order_and_extra_columns(tmp_path):\\n    p = write(tmp_path, \\"price,notes,qty,sku,customer,order_id\\\\n10.00,urgent,3,DAL-1KG,Asha,B7\\\\n\\")\\n    orders, errors = import_orders(p)\\n    assert errors == []\\n    assert orders == [{\\"order_id\\": \\"B7\\", \\"customer\\": \\"Asha\\", \\"sku\\": \\"DAL-1KG\\", \\"qty\\": 3, \\"price\\": Decimal(\\"10.00\\")}]\\n\\n\\ndef test_excel_bom_and_spaces_in_header(tmp_path):\\n    text = \\"order_id, customer, sku, qty, price\\\\r\\\\nA1, Ravi, ATTA-5KG, 2, 245.50\\\\r\\\\n\\"\\n    p = write(tmp_path, text, encoding=\\"utf-8-sig\\")\\n    orders, errors = import_orders(p)\\n    assert errors == []\\n    assert len(orders) == 1\\n    assert orders[0][\\"order_id\\"] == \\"A1\\"\\n    assert orders[0][\\"sku\\"] == \\"ATTA-5KG\\"\\n    assert orders[0][\\"qty\\"] == 2\\n    assert orders[0][\\"price\\"] == Decimal(\\"245.50\\")\\n\\n\\ndef test_bad_rows_are_reported_and_skipped(tmp_path):\\n    text = HEADER + \\"\\\\n\\".join([\\n        \\"A1,Ravi,ATTA-5KG,2,245.50\\",   # line 2 ok\\n        \\"A2,Meena,SALT-1KG,two,28\\",    # line 3 qty not a number\\n        \\"A3,Asha,DAL-1KG,0,90\\",        # line 4 qty zero\\n        \\"A4,Asha,DAL-1KG,-2,90\\",       # line 5 negative\\n        \\"A5,Asha,DAL-1KG,1.5,90\\",      # line 6 not whole\\n        \\"A6,Ravi,OIL-1L,1,abc\\",        # line 7 bad price\\n        \\"A7,Ravi,OIL-1L,1,-10\\",        # line 8 negative price\\n        \\"A8,Ravi,OIL-1L,1,180\\",        # line 9 ok\\n    ]) + \\"\\\\n\\"\\n    orders, errors = import_orders(write(tmp_path, text))\\n    assert [o[\\"order_id\\"] for o in orders] == [\\"A1\\", \\"A8\\"]\\n    assert [line for line, _ in errors] == [3, 4, 5, 6, 7, 8]\\n    assert all(isinstance(reason, str) and reason for _, reason in errors)\\n\\n\\ndef test_missing_values_and_short_rows(tmp_path):\\n    text = HEADER + \\"A1,,ATTA-5KG,2,245.50\\\\n,Ravi,ATTA-5KG,2,245.50\\\\nA3,Ravi,ATTA-5KG\\\\nA4,Ravi,SALT-1KG,1,20\\\\n\\"\\n    orders, errors = import_orders(write(tmp_path, text))\\n    assert [o[\\"order_id\\"] for o in orders] == [\\"A4\\"]\\n    assert [line for line, _ in errors] == [2, 3, 4]\\n\\n\\ndef test_duplicate_order_id(tmp_path):\\n    text = HEADER + \\"A1,Ravi,ATTA-5KG,2,245.50\\\\nA2,Meena,SALT-1KG,1,28\\\\nA1,Ravi,ATTA-5KG,2,245.50\\\\n\\"\\n    orders, errors = import_orders(write(tmp_path, text))\\n    assert [o[\\"order_id\\"] for o in orders] == [\\"A1\\", \\"A2\\"]\\n    assert [line for line, _ in errors] == [4]\\n\\n\\ndef test_quotes_unicode_and_blank_lines(tmp_path):\\n    text = HEADER + \'\\"A1\\",\\"Sharma, Ravi\\",ATTA-5KG,2,245.50\\\\n\\\\nA2,राम किराना,SALT-1KG,1,28\\\\n\'\\n    orders, errors = import_orders(write(tmp_path, text))\\n    assert errors == []\\n    assert [o[\\"customer\\"] for o in orders] == [\\"Sharma, Ravi\\", \\"राम किराना\\"]\\n\\n\\n@pytest.mark.parametrize(\\"text\\", [\\"order_id,customer,sku,qty\\\\nA1,Ravi,X,1\\\\n\\", \\"id;name;sku\\\\n1;2;3\\\\n\\", \\"\\"])\\ndef test_missing_columns_raise_value_error(tmp_path, text):\\n    with pytest.raises(ValueError):\\n        import_orders(write(tmp_path, text))\\n\\n\\ndef test_missing_file(tmp_path):\\n    with pytest.raises(FileNotFoundError):\\n        import_orders(tmp_path / \\"nope.csv\\")\\n\\n\\ndef test_header_only_file(tmp_path):\\n    assert import_orders(write(tmp_path, HEADER)) == ([], [])\\n\\n\\ndef test_no_state_between_calls(tmp_path):\\n    p = write(tmp_path, HEADER + \\"A1,Ravi,ATTA-5KG,2,245.50\\\\n\\")\\n    first = import_orders(p)\\n    second = import_orders(p)\\n    assert first == second\\n    assert second[1] == []  # A1 in the second import is not a duplicate of the first import\\n\\n\\ndef test_large_file_is_fast(tmp_path):\\n    rows = [f\\"O{i},Customer {i % 50},SKU-{i % 300},{1 + i % 5},{i % 1000}.25\\" for i in range(50_000)]\\n    p = write(tmp_path, HEADER + \\"\\\\n\\".join(rows) + \\"\\\\n\\")\\n    start = time.perf_counter()\\n    orders, errors = import_orders(p)\\n    assert time.perf_counter() - start < 3.0\\n    assert len(orders) == 50_000 and errors == []\\n", "id": "03-csv-order-import", "meta": {"difficulty": 2, "files": ["order_import.py"], "kind": "build", "timeout": 60}, "reference": {"order_import.py": "\\"\\"\\"Import orders from a spreadsheet CSV export, collecting bad rows instead of failing.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport csv\\nfrom decimal import Decimal, InvalidOperation\\nfrom os import PathLike\\n\\nREQUIRED = (\\"order_id\\", \\"customer\\", \\"sku\\", \\"qty\\", \\"price\\")\\n\\n\\ndef _parse_row(row: dict[str, str], columns: dict[str, str]) -> dict:\\n    values = {}\\n    for name in REQUIRED:\\n        raw = row.get(columns[name])\\n        if raw is None:\\n            raise ValueError(f\\"missing value for {name}\\")\\n        raw = raw.strip()\\n        if not raw:\\n            raise ValueError(f\\"empty {name}\\")\\n        values[name] = raw\\n    try:\\n        qty = int(values[\\"qty\\"])\\n    except ValueError:\\n        raise ValueError(f\\"qty is not a whole number: {values[\'qty\']!r}\\") from None\\n    if qty <= 0:\\n        raise ValueError(f\\"qty must be positive: {qty}\\")\\n    try:\\n        price = Decimal(values[\\"price\\"])\\n    except InvalidOperation:\\n        raise ValueError(f\\"price is not a number: {values[\'price\']!r}\\") from None\\n    if not price.is_finite() or price < 0:\\n        raise ValueError(f\\"bad price: {values[\'price\']!r}\\")\\n    values[\\"qty\\"] = qty\\n    values[\\"price\\"] = price\\n    return values\\n\\n\\ndef import_orders(path: str | PathLike) -> tuple[list[dict], list[tuple[int, str]]]:\\n    \\"\\"\\"Read orders from the CSV at path. Returns (orders, errors).\\"\\"\\"\\n    orders: list[dict] = []\\n    errors: list[tuple[int, str]] = []\\n    seen: set[str] = set()\\n    # utf-8-sig strips the byte-order mark that Excel puts at the start of CSV exports.\\n    with open(path, newline=\\"\\", encoding=\\"utf-8-sig\\") as fh:\\n        reader = csv.reader(fh)\\n        header = next(reader, None)\\n        if header is None:\\n            raise ValueError(\\"file is empty: no header row\\")\\n        columns: dict[str, str] = {}\\n        normalised = [h.strip().lower() for h in header]\\n        for name in REQUIRED:\\n            if name not in normalised:\\n                raise ValueError(f\\"missing required column {name!r}\\")\\n            columns[name] = name\\n        for row_values in reader:\\n            line = reader.line_num\\n            if not any(v.strip() for v in row_values):\\n                continue\\n            if len(row_values) != len(normalised):\\n                errors.append((line, f\\"expected {len(normalised)} fields, got {len(row_values)}\\"))\\n                continue\\n            row = {}\\n            for key, value in zip(normalised, row_values, strict=True):\\n                row.setdefault(key, value)\\n            try:\\n                order = _parse_row(row, columns)\\n            except ValueError as exc:\\n                errors.append((line, str(exc)))\\n                continue\\n            if order[\\"order_id\\"] in seen:\\n                errors.append((line, f\\"duplicate order_id {order[\'order_id\']!r}\\"))\\n                continue\\n            seen.add(order[\\"order_id\\"])\\n            orders.append(order)\\n    return orders, errors\\n"}, "starter": {}, "task_md": "Our staff keep orders in a spreadsheet and export them as CSV; I want to load those files into the new system. Please write `order_import.py` with `import_orders(path)` that returns a tuple `(orders, errors)`.\\nThe CSV has a header row with the columns `order_id`, `customer`, `sku`, `qty` and `price` (the order of columns can change and extra columns can be ignored). Each good row becomes a dict with exactly those five keys: `qty` as an int, `price` as a `Decimal`, the others as strings.\\nA bad row must not stop the import: add `(line_number, reason)` to `errors` (the header is line 1) and carry on with the next row. An `order_id` that already appeared earlier in the file counts as a bad row too.\\nIf the file doesn\'t have the required columns at all, raise `ValueError`."}'),
    json.loads('{"hidden": "import time\\n\\nimport pytest\\n\\nfrom ratelimit import KeyedLimiter, TokenBucket\\n\\n\\nclass FakeClock:\\n    def __init__(self, now=1000.0):\\n        self.now = now\\n\\n    def __call__(self):\\n        return self.now\\n\\n    def advance(self, seconds):\\n        self.now += seconds\\n\\n\\ndef test_starts_full():\\n    clock = FakeClock()\\n    b = TokenBucket(rate=1, capacity=3, clock=clock)\\n    assert [b.allow() for _ in range(4)] == [True, True, True, False]\\n\\n\\ndef test_refills_over_time():\\n    clock = FakeClock()\\n    b = TokenBucket(rate=2, capacity=4, clock=clock)\\n    for _ in range(4):\\n        assert b.allow()\\n    assert not b.allow()\\n    clock.advance(1)\\n    assert b.allow()\\n    assert b.allow()\\n    assert not b.allow()\\n\\n\\ndef test_fractional_rate():\\n    clock = FakeClock()\\n    b = TokenBucket(rate=0.5, capacity=1, clock=clock)\\n    assert b.allow()\\n    clock.advance(1)\\n    assert not b.allow()\\n    clock.advance(1)\\n    assert b.allow()\\n\\n\\ndef test_never_exceeds_capacity():\\n    clock = FakeClock()\\n    b = TokenBucket(rate=10, capacity=5, clock=clock)\\n    clock.advance(10_000)\\n    assert sum(b.allow() for _ in range(20)) == 5\\n\\n\\ndef test_denied_request_takes_nothing():\\n    clock = FakeClock()\\n    b = TokenBucket(rate=1, capacity=5, clock=clock)\\n    assert b.allow(cost=3)\\n    assert not b.allow(cost=3)\\n    assert b.allow(cost=2)\\n    assert not b.allow()\\n\\n\\ndef test_clock_going_backwards_does_not_drain():\\n    clock = FakeClock()\\n    b = TokenBucket(rate=1, capacity=2, clock=clock)\\n    clock.advance(-5)\\n    assert b.allow()\\n    assert b.allow()\\n    assert not b.allow()\\n    clock.advance(6)\\n    assert b.allow()\\n\\n\\ndef test_uses_injected_clock_only():\\n    clock = FakeClock()\\n    b = TokenBucket(rate=1, capacity=1, clock=clock)\\n    assert b.allow()\\n    time.sleep(0.05)\\n    assert not b.allow()  # real time passed, fake time did not\\n\\n\\n@pytest.mark.parametrize(\\"rate, capacity\\", [(0, 1), (-1, 1), (1, 0), (1, -5), (float(\\"nan\\"), 1), (1, float(\\"inf\\"))])\\ndef test_bad_settings_raise(rate, capacity):\\n    with pytest.raises(ValueError):\\n        TokenBucket(rate=rate, capacity=capacity, clock=FakeClock())\\n    with pytest.raises(ValueError):\\n        KeyedLimiter(rate=rate, capacity=capacity, clock=FakeClock())\\n\\n\\n@pytest.mark.parametrize(\\"cost\\", [0, -1])\\ndef test_bad_cost_raises(cost):\\n    clock = FakeClock()\\n    b = TokenBucket(rate=1, capacity=5, clock=clock)\\n    with pytest.raises(ValueError):\\n        b.allow(cost=cost)\\n    assert b.allow(cost=5)  # the bad call took nothing\\n    k = KeyedLimiter(rate=1, capacity=5, clock=clock)\\n    with pytest.raises(ValueError):\\n        k.allow(\\"98450\\", cost=cost)\\n\\n\\ndef test_buckets_are_independent():\\n    clock = FakeClock()\\n    a = TokenBucket(rate=1, capacity=1, clock=clock)\\n    b = TokenBucket(rate=1, capacity=1, clock=clock)\\n    assert a.allow()\\n    assert b.allow()\\n\\n\\ndef test_keyed_limiter_separate_per_key():\\n    clock = FakeClock()\\n    lim = KeyedLimiter(rate=1, capacity=2, clock=clock)\\n    assert lim.allow(\\"9876543210\\")\\n    assert lim.allow(\\"9876543210\\")\\n    assert not lim.allow(\\"9876543210\\")\\n    assert lim.allow(\\"9123456780\\")\\n    clock.advance(1)\\n    assert lim.allow(\\"9876543210\\")\\n    assert not lim.allow(\\"9876543210\\")\\n\\n\\ndef test_keyed_limiters_do_not_share_state():\\n    clock = FakeClock()\\n    a = KeyedLimiter(rate=1, capacity=1, clock=clock)\\n    b = KeyedLimiter(rate=1, capacity=1, clock=clock)\\n    assert a.allow(\\"k\\")\\n    assert b.allow(\\"k\\")\\n\\n\\ndef test_new_key_bucket_starts_full_even_later():\\n    clock = FakeClock()\\n    lim = KeyedLimiter(rate=1, capacity=3, clock=clock)\\n    clock.advance(100)\\n    assert sum(lim.allow(\\"late\\") for _ in range(5)) == 3\\n\\n\\ndef test_many_calls_are_fast():\\n    clock = FakeClock()\\n    lim = KeyedLimiter(rate=5, capacity=10, clock=clock)\\n    start = time.perf_counter()\\n    allowed = 0\\n    for i in range(100_000):\\n        clock.advance(0.001)\\n        allowed += lim.allow(i % 1000)\\n    assert time.perf_counter() - start < 3.0\\n    assert allowed > 0\\n", "id": "04-rate-limiter", "meta": {"difficulty": 2, "files": ["ratelimit.py"], "kind": "build", "timeout": 60}, "reference": {"ratelimit.py": "\\"\\"\\"Token-bucket rate limiting with an injectable clock.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport threading\\nimport time\\nfrom collections.abc import Callable, Hashable\\n\\nClock = Callable[[], float]\\n\\n\\ndef _positive(name: str, value: float) -> float:\\n    if isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value) and value > 0:\\n        return float(value)\\n    raise ValueError(f\\"{name} must be a positive finite number, got {value!r}\\")\\n\\n\\nclass TokenBucket:\\n    \\"\\"\\"A bucket that refills at `rate` tokens per second up to `capacity` tokens.\\"\\"\\"\\n\\n    def __init__(self, rate: float, capacity: float, clock: Clock = time.monotonic) -> None:\\n        self.rate = _positive(\\"rate\\", rate)\\n        self.capacity = _positive(\\"capacity\\", capacity)\\n        self._clock = clock\\n        self._tokens = self.capacity\\n        self._last = clock()\\n        self._lock = threading.Lock()\\n\\n    def _refill(self) -> None:\\n        now = self._clock()\\n        elapsed = now - self._last\\n        if elapsed > 0:  # a clock that goes backwards must not drain the bucket\\n            self._tokens = min(self.capacity, self._tokens + elapsed * self.rate)\\n        self._last = max(self._last, now)\\n\\n    def allow(self, cost: float = 1) -> bool:\\n        cost = _positive(\\"cost\\", cost)\\n        with self._lock:\\n            self._refill()\\n            if cost <= self._tokens:\\n                self._tokens -= cost\\n                return True\\n            return False\\n\\n\\nclass KeyedLimiter:\\n    \\"\\"\\"One token bucket per key, created on first use.\\"\\"\\"\\n\\n    def __init__(self, rate: float, capacity: float, clock: Clock = time.monotonic) -> None:\\n        self.rate = _positive(\\"rate\\", rate)\\n        self.capacity = _positive(\\"capacity\\", capacity)\\n        self._clock = clock\\n        self._buckets: dict[Hashable, TokenBucket] = {}\\n        self._lock = threading.Lock()\\n\\n    def allow(self, key: Hashable, cost: float = 1) -> bool:\\n        _positive(\\"cost\\", cost)\\n        with self._lock:\\n            bucket = self._buckets.get(key)\\n            if bucket is None:\\n                bucket = self._buckets[key] = TokenBucket(self.rate, self.capacity, self._clock)\\n        return bucket.allow(cost)\\n"}, "starter": {}, "task_md": "We send WhatsApp order updates through a provider that blocks us if we send in bursts, so I need a rate limiter. Please write `ratelimit.py` with a token bucket: `TokenBucket(rate, capacity, clock=time.monotonic)`, where `rate` is tokens added per second and `capacity` is the most it can hold; a new bucket starts full.\\n`allow(cost=1)` returns True and takes the tokens if there are enough, otherwise it returns False and takes nothing.\\nAlso add `KeyedLimiter(rate, capacity, clock=time.monotonic)` whose `allow(key, cost=1)` keeps a separate bucket per key (we use the customer\'s phone number).\\nThe `clock` argument is there so tests can control time. Raise `ValueError` for a rate, capacity or cost that makes no sense."}'),
    json.loads('{"hidden": "import time\\n\\nimport pytest\\n\\nfrom ttlcache import TTLCache\\n\\n\\nclass FakeClock:\\n    def __init__(self, now=500.0):\\n        self.now = now\\n\\n    def __call__(self):\\n        return self.now\\n\\n\\ndef make(maxsize=3, ttl=10):\\n    clock = FakeClock()\\n    return TTLCache(maxsize, ttl, clock=clock), clock\\n\\n\\ndef test_set_and_get():\\n    c, _ = make()\\n    c.set(\\"atta\\", 245)\\n    assert c.get(\\"atta\\") == 245\\n    assert \\"atta\\" in c\\n    assert len(c) == 1\\n\\n\\ndef test_missing_key_returns_default():\\n    c, _ = make()\\n    assert c.get(\\"nope\\") is None\\n    assert c.get(\\"nope\\", \\"dflt\\") == \\"dflt\\"\\n    assert \\"nope\\" not in c\\n\\n\\ndef test_falsy_and_none_values_are_cached():\\n    c, _ = make()\\n    c.set(\\"a\\", None)\\n    c.set(\\"b\\", 0)\\n    assert c.get(\\"a\\", \\"dflt\\") is None\\n    assert c.get(\\"b\\", \\"dflt\\") == 0\\n    assert \\"a\\" in c and len(c) == 2\\n\\n\\ndef test_expiry_boundary():\\n    c, clock = make(ttl=10)\\n    c.set(\\"k\\", 1)\\n    clock.now += 9.999\\n    assert c.get(\\"k\\") == 1\\n    clock.now = 510.0\\n    assert c.get(\\"k\\") is None\\n    assert \\"k\\" not in c\\n    assert len(c) == 0\\n\\n\\ndef test_get_does_not_extend_ttl_but_set_does():\\n    c, clock = make(ttl=10)\\n    c.set(\\"k\\", 1)\\n    clock.now += 6\\n    assert c.get(\\"k\\") == 1\\n    clock.now += 6\\n    assert c.get(\\"k\\") is None\\n    c.set(\\"k\\", 2)\\n    clock.now += 6\\n    c.set(\\"k\\", 3)\\n    clock.now += 6\\n    assert c.get(\\"k\\") == 3\\n\\n\\ndef test_lru_eviction_order():\\n    c, _ = make(maxsize=2)\\n    c.set(\\"a\\", 1)\\n    c.set(\\"b\\", 2)\\n    c.get(\\"a\\")\\n    c.set(\\"c\\", 3)\\n    assert c.get(\\"b\\") is None\\n    assert c.get(\\"a\\") == 1 and c.get(\\"c\\") == 3\\n    assert len(c) == 2\\n\\n\\ndef test_updating_key_does_not_evict_others():\\n    c, _ = make(maxsize=2)\\n    c.set(\\"a\\", 1)\\n    c.set(\\"b\\", 2)\\n    c.set(\\"a\\", 10)\\n    assert len(c) == 2\\n    c.set(\\"c\\", 3)  # a was set most recently, so b goes\\n    assert c.get(\\"b\\") is None\\n    assert c.get(\\"a\\") == 10 and c.get(\\"c\\") == 3\\n\\n\\ndef test_expired_entries_never_push_out_live_ones():\\n    c, clock = make(maxsize=2, ttl=10)\\n    c.set(\\"b\\", \\"old\\")      # expires at 510\\n    clock.now += 5\\n    c.set(\\"a\\", \\"live\\")     # expires at 515\\n    clock.now += 1\\n    c.get(\\"b\\")             # b is now most recently used, a least\\n    clock.now = 512.0      # b expired, a still live: only 1 live entry\\n    c.set(\\"c\\", \\"new\\")\\n    assert c.get(\\"a\\") == \\"live\\"\\n    assert c.get(\\"c\\") == \\"new\\"\\n\\n\\ndef test_len_counts_only_live_entries():\\n    c, clock = make(maxsize=5, ttl=10)\\n    c.set(\\"a\\", 1)\\n    clock.now += 5\\n    c.set(\\"b\\", 2)\\n    clock.now += 6\\n    assert len(c) == 1\\n\\n\\ndef test_delete():\\n    c, _ = make()\\n    c.set(\\"a\\", 1)\\n    c.delete(\\"a\\")\\n    assert c.get(\\"a\\") is None\\n    c.delete(\\"a\\")\\n    c.delete(\\"never\\")\\n    assert len(c) == 0\\n\\n\\n@pytest.mark.parametrize(\\"maxsize, ttl\\", [(0, 10), (-1, 10), (3, 0), (3, -1), (3, float(\\"nan\\"))])\\ndef test_bad_settings_raise(maxsize, ttl):\\n    with pytest.raises(ValueError):\\n        TTLCache(maxsize, ttl, clock=FakeClock())\\n\\n\\ndef test_caches_do_not_share_state():\\n    c1, _ = make()\\n    c2, _ = make()\\n    c1.set(\\"a\\", 1)\\n    assert c2.get(\\"a\\") is None\\n    assert len(c2) == 0\\n\\n\\ndef test_default_clock_works():\\n    c = TTLCache(2, 60)\\n    c.set(\\"a\\", 1)\\n    assert c.get(\\"a\\") == 1\\n\\n\\ndef test_many_operations_are_fast():\\n    clock = FakeClock()\\n    c = TTLCache(1000, 30, clock=clock)\\n    start = time.perf_counter()\\n    for i in range(100_000):\\n        clock.now += 0.001\\n        c.set(i, i)\\n        c.get(i - 500)\\n    assert time.perf_counter() - start < 3.0\\n    assert len(c) == 1000\\n    assert c.get(99_999) == 99_999\\n    assert c.get(0) is None\\n", "id": "05-lru-ttl-cache", "meta": {"difficulty": 3, "files": ["ttlcache.py"], "kind": "build", "timeout": 60}, "reference": {"ttlcache.py": "\\"\\"\\"A least-recently-used cache whose entries expire after a fixed time.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nimport threading\\nimport time\\nfrom collections import OrderedDict\\nfrom collections.abc import Callable, Hashable\\nfrom typing import Any\\n\\n_MISSING = object()\\n\\n\\nclass TTLCache:\\n    \\"\\"\\"LRU cache with per-entry expiry. Each instance has its own storage; thread-safe.\\"\\"\\"\\n\\n    def __init__(self, maxsize: int, ttl: float, clock: Callable[[], float] = time.monotonic) -> None:\\n        if isinstance(maxsize, bool) or not isinstance(maxsize, int) or maxsize < 1:\\n            raise ValueError(f\\"maxsize must be a positive integer, got {maxsize!r}\\")\\n        if isinstance(ttl, bool) or not isinstance(ttl, (int, float)) or not math.isfinite(ttl) or ttl <= 0:\\n            raise ValueError(f\\"ttl must be a positive number of seconds, got {ttl!r}\\")\\n        self.maxsize = maxsize\\n        self.ttl = ttl\\n        self._clock = clock\\n        # Recency order: least recently used first.\\n        self._values: OrderedDict[Hashable, Any] = OrderedDict()\\n        # Expiry order: the ttl is the same for every entry, so the entry set\\n        # longest ago always expires first.\\n        self._expires: OrderedDict[Hashable, float] = OrderedDict()\\n        self._lock = threading.RLock()\\n\\n    def _remove(self, key: Hashable) -> None:\\n        del self._values[key]\\n        del self._expires[key]\\n\\n    def _purge_expired(self, now: float) -> None:\\n        while self._expires:\\n            key, expires_at = next(iter(self._expires.items()))\\n            if expires_at > now:\\n                break\\n            self._remove(key)\\n\\n    def get(self, key: Hashable, default: Any = None) -> Any:\\n        with self._lock:\\n            expires_at = self._expires.get(key, _MISSING)\\n            if expires_at is _MISSING:\\n                return default\\n            if expires_at <= self._clock():\\n                self._remove(key)\\n                return default\\n            self._values.move_to_end(key)\\n            return self._values[key]\\n\\n    def set(self, key: Hashable, value: Any) -> None:\\n        with self._lock:\\n            now = self._clock()\\n            self._purge_expired(now)\\n            if key in self._values:\\n                self._remove(key)\\n            while len(self._values) >= self.maxsize:\\n                oldest = next(iter(self._values))\\n                self._remove(oldest)\\n            self._values[key] = value\\n            self._expires[key] = now + self.ttl\\n\\n    def delete(self, key: Hashable) -> None:\\n        with self._lock:\\n            if key in self._values:\\n                self._remove(key)\\n\\n    def __contains__(self, key: Hashable) -> bool:\\n        with self._lock:\\n            expires_at = self._expires.get(key)\\n            return expires_at is not None and expires_at > self._clock()\\n\\n    def __len__(self) -> int:\\n        with self._lock:\\n            self._purge_expired(self._clock())\\n            return len(self._values)\\n"}, "starter": {}, "task_md": "Our price lookups hit the supplier API too often. Please write `ttlcache.py` with `TTLCache(maxsize, ttl, clock=time.monotonic)`: a cache of at most `maxsize` live entries, where each entry expires `ttl` seconds after it was last set (an entry set at time t is gone at exactly t + ttl).\\nIt needs `get(key, default=None)`, `set(key, value)`, `delete(key)` (deleting a key that isn\'t there is fine), `len(cache)` for the number of live entries and `key in cache`.\\nWhen it is full, drop the least recently used entry (reading or setting a key counts as using it).\\nRaise `ValueError` if `maxsize` or `ttl` is not a positive number. The `clock` argument lets tests control time."}'),
    json.loads('{"hidden": "import re\\nimport time\\n\\nimport pytest\\n\\nfrom slugs import SlugRegistry, slugify\\n\\nSLUG = re.compile(r\\"^[a-z0-9]+(-[a-z0-9]+)*$\\")\\n\\n\\n@pytest.mark.parametrize(\\"text, expected\\", [\\n    (\\"Tata Salt 1kg\\", \\"tata-salt-1kg\\"),\\n    (\\"  Amul -- Butter!!  \\", \\"amul-butter\\"),\\n    (\\"Café Crème\\", \\"cafe-creme\\"),\\n    (\\"Parle-G (Pack of 10)\\", \\"parle-g-pack-of-10\\"),\\n    (\\"MAGGI_Masala/Noodles\\", \\"maggi-masala-noodles\\"),\\n])\\ndef test_basic_slugs(text, expected):\\n    assert slugify(text) == expected\\n\\n\\n@pytest.mark.parametrize(\\"text\\", [\\"\\", \\"   \\", \\"!!!\\", \\"नमक\\", \\"---\\"])\\ndef test_empty_result_falls_back_to_item(text):\\n    assert slugify(text) == \\"item\\"\\n\\n\\ndef test_mixed_script_keeps_latin_part():\\n    assert slugify(\\"Namak नमक 1kg\\") == \\"namak-1kg\\"\\n\\n\\ndef test_max_length_default_and_custom():\\n    assert len(slugify(\\"a\\" * 100)) == 60\\n    assert slugify(\\"abcd efgh\\", max_length=5) == \\"abcd\\"  # no dangling dash after cutting\\n    assert slugify(\\"abcd efgh\\", max_length=6) == \\"abcd-e\\"\\n\\n\\ndef test_slugify_is_idempotent_and_deterministic():\\n    for text in [\\"Tata Salt 1kg\\", \\"Café Crème\\", \\"a\\" * 80, \\"x - y - z\\"]:\\n        once = slugify(text)\\n        assert slugify(once) == once\\n        assert slugify(text) == once\\n\\n\\ndef test_registry_suffixes():\\n    reg = SlugRegistry()\\n    assert [reg.claim(\\"Tata Salt\\") for _ in range(3)] == [\\"tata-salt\\", \\"tata-salt-2\\", \\"tata-salt-3\\"]\\n    assert reg.claim(\\"Amul Butter\\") == \\"amul-butter\\"\\n\\n\\ndef test_registry_avoids_slugs_that_look_like_suffixes():\\n    reg = SlugRegistry()\\n    got = [reg.claim(\\"Salt 2\\"), reg.claim(\\"Salt\\"), reg.claim(\\"Salt\\"), reg.claim(\\"Salt\\"), reg.claim(\\"salt-2\\")]\\n    assert len(set(got)) == len(got)\\n    assert got[0] == \\"salt-2\\" and got[1] == \\"salt\\"\\n    assert all(SLUG.match(s) for s in got)\\n\\n\\ndef test_registry_respects_max_length():\\n    reg = SlugRegistry(max_length=12)\\n    got = [reg.claim(\\"Super Long Product Name\\") for _ in range(15)]\\n    assert len(set(got)) == 15\\n    assert all(len(s) <= 12 for s in got)\\n    assert all(SLUG.match(s) for s in got)\\n\\n\\ndef test_registry_fallback_names_are_unique():\\n    reg = SlugRegistry()\\n    got = [reg.claim(\\"नमक\\") for _ in range(3)] + [reg.claim(\\"\\") for _ in range(2)]\\n    assert got[0] == \\"item\\"\\n    assert len(set(got)) == 5\\n    assert all(SLUG.match(s) for s in got)\\n\\n\\ndef test_registries_do_not_share_state():\\n    a = SlugRegistry()\\n    b = SlugRegistry()\\n    assert a.claim(\\"Tata Salt\\") == \\"tata-salt\\"\\n    assert b.claim(\\"Tata Salt\\") == \\"tata-salt\\"\\n\\n\\ndef test_many_same_names_are_fast():\\n    reg = SlugRegistry()\\n    start = time.perf_counter()\\n    got = [reg.claim(\\"Parle-G\\") for _ in range(10_000)]\\n    assert time.perf_counter() - start < 1.5\\n    assert len(set(got)) == 10_000\\n    assert got[-1] == \\"parle-g-10000\\"\\n", "id": "06-slugify", "meta": {"difficulty": 1, "files": ["slugs.py"], "kind": "build", "timeout": 60}, "reference": {"slugs.py": "\\"\\"\\"URL-safe slugs for product names, with collision handling.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\nimport unicodedata\\n\\nFALLBACK = \\"item\\"\\n_NON_ALNUM = re.compile(r\\"[^a-z0-9]+\\")\\n\\n\\ndef _check_max_length(max_length: int) -> int:\\n    if isinstance(max_length, bool) or not isinstance(max_length, int) or max_length < len(FALLBACK):\\n        raise ValueError(f\\"max_length must be an integer >= {len(FALLBACK)}, got {max_length!r}\\")\\n    return max_length\\n\\n\\ndef _cut(slug: str, max_length: int) -> str:\\n    return slug[:max_length].strip(\\"-\\")\\n\\n\\ndef slugify(text: str, max_length: int = 60) -> str:\\n    \\"\\"\\"Turn text into a lowercase ASCII slug of at most max_length characters.\\"\\"\\"\\n    if not isinstance(text, str):\\n        raise TypeError(f\\"text must be a str, got {type(text).__name__}\\")\\n    _check_max_length(max_length)\\n    ascii_text = unicodedata.normalize(\\"NFKD\\", text).encode(\\"ascii\\", \\"ignore\\").decode(\\"ascii\\")\\n    slug = _NON_ALNUM.sub(\\"-\\", ascii_text.lower()).strip(\\"-\\")\\n    return _cut(slug, max_length) or FALLBACK\\n\\n\\nclass SlugRegistry:\\n    \\"\\"\\"Hands out unique slugs. Each registry keeps its own set of claimed slugs.\\"\\"\\"\\n\\n    def __init__(self, max_length: int = 60) -> None:\\n        self.max_length = _check_max_length(max_length)\\n        self._taken: set[str] = set()\\n        self._next_suffix: dict[str, int] = {}\\n\\n    def claim(self, text: str) -> str:\\n        base = slugify(text, self.max_length)\\n        if base not in self._taken:\\n            self._taken.add(base)\\n            return base\\n        n = self._next_suffix.get(base, 2)\\n        while True:\\n            suffix = f\\"-{n}\\"\\n            room = self.max_length - len(suffix)\\n            candidate = (_cut(base, room) or FALLBACK[:room]) + suffix\\n            n += 1\\n            if candidate not in self._taken:\\n                break\\n        self._next_suffix[base] = n\\n        self._taken.add(candidate)\\n        return candidate\\n"}, "starter": {}, "task_md": "Our online catalogue needs URL-safe ids for products. Please write `slugs.py` with `slugify(text, max_length=60)`: lowercase, accents removed (\\"Café\\" becomes \\"cafe\\"), every run of other characters turned into a single `-`, no `-` at the start or end, and at most `max_length` characters. If nothing is left (for example a name written only in Hindi), use `\\"item\\"`.\\nAlso add a class `SlugRegistry(max_length=60)` with `claim(text)` that returns a slug no one has claimed before from that registry: the first \\"Tata Salt\\" gets `tata-salt`, the next ones `tata-salt-2`, `tata-salt-3` and so on, still within `max_length`.\\nWe have thousands of products with the same name (\\"Parle-G\\" in every pack size), so it has to stay quick."}'),
    json.loads('{"hidden": "import json\\nimport subprocess\\nimport sys\\nimport time\\n\\nimport logsum\\n\\nSAMPLE = \\"\\"\\"2026-10-01 10:00:03 INFO server started\\n2026-10-01 10:05:00 ERROR payment gateway timeout\\n2026-10-01 10:06:00 WARNING low stock ATTA-5KG\\n\\n2026-10-01 10:07:00 ERROR payment gateway timeout\\n2026-10-01 09:59:00 DEBUG cache warm\\n2026-10-01 11:00:00 ERROR sms failed\\nthis line is garbage\\n2026-13-45 10:00:00 ERROR impossible date\\n2026-10-01 12:00:00 FATAL unknown level\\n2026-10-01 12:30:00 ERROR db locked\\n\\"\\"\\"\\n\\n\\ndef run_json(tmp_path, capsys, text=SAMPLE, extra=(), encoding=\\"utf-8\\"):\\n    p = tmp_path / \\"shop.log\\"\\n    if isinstance(text, bytes):\\n        p.write_bytes(text)\\n    else:\\n        p.write_text(text, encoding=encoding)\\n    code = logsum.main([str(p), \\"--json\\", *extra])\\n    out = capsys.readouterr().out\\n    assert code == 0\\n    return json.loads(out)\\n\\n\\ndef test_json_summary(tmp_path, capsys):\\n    s = run_json(tmp_path, capsys)\\n    assert s[\\"total_lines\\"] == 10\\n    assert s[\\"by_level\\"] == {\\"DEBUG\\": 1, \\"INFO\\": 1, \\"WARNING\\": 1, \\"ERROR\\": 4}\\n    assert s[\\"malformed\\"] == 3\\n    assert s[\\"first\\"] == \\"2026-10-01 09:59:00\\"  # earliest, not first in the file\\n    assert s[\\"last\\"] == \\"2026-10-01 12:30:00\\"\\n\\n\\ndef test_top_errors_counts_and_ties(tmp_path, capsys):\\n    s = run_json(tmp_path, capsys)\\n    assert s[\\"top_errors\\"] == [\\n        {\\"message\\": \\"payment gateway timeout\\", \\"count\\": 2},\\n        {\\"message\\": \\"db locked\\", \\"count\\": 1},\\n        {\\"message\\": \\"sms failed\\", \\"count\\": 1},\\n    ]\\n\\n\\ndef test_top_limit(tmp_path, capsys):\\n    s = run_json(tmp_path, capsys, extra=[\\"--top\\", \\"1\\"])\\n    assert s[\\"top_errors\\"] == [{\\"message\\": \\"payment gateway timeout\\", \\"count\\": 2}]\\n\\n\\ndef test_default_top_is_five(tmp_path, capsys):\\n    text = \\"\\".join(f\\"2026-10-01 10:00:0{i} ERROR problem {i}\\\\n\\" for i in range(8))\\n    s = run_json(tmp_path, capsys, text=text)\\n    assert [e[\\"message\\"] for e in s[\\"top_errors\\"]] == [f\\"problem {i}\\" for i in range(5)]\\n\\n\\ndef test_empty_file(tmp_path, capsys):\\n    s = run_json(tmp_path, capsys, text=\\"\\")\\n    assert s == {\\n        \\"total_lines\\": 0,\\n        \\"by_level\\": {\\"DEBUG\\": 0, \\"INFO\\": 0, \\"WARNING\\": 0, \\"ERROR\\": 0},\\n        \\"malformed\\": 0,\\n        \\"first\\": None,\\n        \\"last\\": None,\\n        \\"top_errors\\": [],\\n    }\\n\\n\\ndef test_windows_line_endings(tmp_path, capsys):\\n    text = \\"2026-10-01 10:00:00 ERROR disk full\\\\r\\\\n2026-10-01 10:00:01 ERROR disk full\\\\r\\\\n\\\\r\\\\n\\"\\n    s = run_json(tmp_path, capsys, text=text.encode(\\"utf-8\\"))\\n    assert s[\\"total_lines\\"] == 2\\n    assert s[\\"top_errors\\"] == [{\\"message\\": \\"disk full\\", \\"count\\": 2}]\\n\\n\\ndef test_bad_bytes_do_not_crash(tmp_path, capsys):\\n    data = b\\"2026-10-01 10:00:00 INFO ok\\\\n2026-10-01 10:00:01 ERROR bad \\\\xff\\\\xfe bytes\\\\n\\\\xff\\\\xff\\\\xff\\\\n\\"\\n    s = run_json(tmp_path, capsys, text=data)\\n    assert s[\\"total_lines\\"] == 3\\n    assert s[\\"by_level\\"][\\"ERROR\\"] == 1\\n    assert s[\\"malformed\\"] == 1\\n\\n\\ndef test_unicode_messages(tmp_path, capsys):\\n    s = run_json(tmp_path, capsys, text=\\"2026-10-01 10:00:00 ERROR ग्राहक नहीं मिला\\\\n\\")\\n    assert s[\\"top_errors\\"] == [{\\"message\\": \\"ग्राहक नहीं मिला\\", \\"count\\": 1}]\\n\\n\\ndef test_text_output(tmp_path, capsys):\\n    p = tmp_path / \\"shop.log\\"\\n    p.write_text(SAMPLE, encoding=\\"utf-8\\")\\n    assert logsum.main([str(p)]) == 0\\n    out = capsys.readouterr().out\\n    assert \\"ERROR\\" in out\\n    assert \\"payment gateway timeout\\" in out\\n    assert not out.lstrip().startswith(\\"{\\")\\n\\n\\ndef test_missing_file_is_clean_error(tmp_path, capsys):\\n    code = logsum.main([str(tmp_path / \\"missing.log\\"), \\"--json\\"])\\n    captured = capsys.readouterr()\\n    assert code != 0\\n    assert captured.err.strip()\\n    assert \\"Traceback\\" not in captured.err\\n    assert captured.out.strip() == \\"\\"\\n\\n\\ndef test_directory_instead_of_file_is_clean_error(tmp_path, capsys):\\n    code = logsum.main([str(tmp_path)])\\n    captured = capsys.readouterr()\\n    assert code != 0\\n    assert captured.err.strip()\\n    assert \\"Traceback\\" not in captured.err\\n\\n\\ndef test_negative_top_is_rejected(tmp_path, capsys):\\n    p = tmp_path / \\"shop.log\\"\\n    p.write_text(SAMPLE, encoding=\\"utf-8\\")\\n    try:\\n        code = logsum.main([str(p), \\"--top\\", \\"-1\\"])\\n    except SystemExit as exc:\\n        code = exc.code\\n    assert code not in (0, None)\\n\\n\\ndef test_runs_as_a_script(tmp_path):\\n    p = tmp_path / \\"shop.log\\"\\n    p.write_text(SAMPLE, encoding=\\"utf-8\\")\\n    result = subprocess.run(\\n        [sys.executable, logsum.__file__, str(p), \\"--json\\"], capture_output=True, text=True, timeout=30, check=False\\n    )\\n    assert result.returncode == 0, result.stderr\\n    assert json.loads(result.stdout)[\\"by_level\\"][\\"ERROR\\"] == 4\\n    missing = subprocess.run(\\n        [sys.executable, logsum.__file__, str(tmp_path / \\"nope.log\\")],\\n        capture_output=True, text=True, timeout=30, check=False,\\n    )\\n    assert missing.returncode != 0\\n    assert \\"Traceback\\" not in missing.stderr\\n\\n\\ndef test_does_not_modify_the_log(tmp_path, capsys):\\n    p = tmp_path / \\"shop.log\\"\\n    p.write_text(SAMPLE, encoding=\\"utf-8\\")\\n    logsum.main([str(p), \\"--json\\"])\\n    assert p.read_text(encoding=\\"utf-8\\") == SAMPLE\\n    assert sorted(x.name for x in tmp_path.iterdir()) == [\\"shop.log\\"]\\n\\n\\ndef test_big_log_is_fast(tmp_path, capsys):\\n    levels = [\\"INFO\\", \\"DEBUG\\", \\"WARNING\\", \\"ERROR\\"]\\n    lines = [\\n        f\\"2026-10-01 {i // 3600 % 24:02d}:{i // 60 % 60:02d}:{i % 60:02d} {levels[i % 4]} event {i % 37}\\\\n\\"\\n        for i in range(200_000)\\n    ]\\n    start = time.perf_counter()\\n    s = run_json(tmp_path, capsys, text=\\"\\".join(lines))\\n    assert time.perf_counter() - start < 4.0\\n    assert s[\\"total_lines\\"] == 200_000\\n    assert s[\\"by_level\\"][\\"ERROR\\"] == 50_000\\n", "id": "07-log-summary-cli", "meta": {"difficulty": 3, "files": ["logsum.py"], "kind": "build", "timeout": 60}, "reference": {"logsum.py": "\\"\\"\\"Summarise a shop server log file: counts per level, time range and top errors.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport sys\\nfrom collections import Counter\\nfrom collections.abc import Iterable\\nfrom datetime import datetime\\n\\nLEVELS = (\\"DEBUG\\", \\"INFO\\", \\"WARNING\\", \\"ERROR\\")\\nTIME_FORMAT = \\"%Y-%m-%d %H:%M:%S\\"\\n\\n\\ndef _parse_line(line: str) -> tuple[datetime, str, str] | None:\\n    parts = line.split(None, 3)\\n    if len(parts) < 3:\\n        return None\\n    try:\\n        # The server writes local time without an offset; naive datetimes are only compared with each other.\\n        stamp = datetime.strptime(f\\"{parts[0]} {parts[1]}\\", TIME_FORMAT)  # noqa: DTZ007\\n    except ValueError:\\n        return None\\n    level = parts[2]\\n    if level not in LEVELS:\\n        return None\\n    message = parts[3].strip() if len(parts) == 4 else \\"\\"\\n    return stamp, level, message\\n\\n\\ndef summarise(lines: Iterable[str], top: int = 5) -> dict:\\n    \\"\\"\\"Build the summary dict for an iterable of log lines.\\"\\"\\"\\n    total = 0\\n    malformed = 0\\n    by_level = dict.fromkeys(LEVELS, 0)\\n    errors: Counter[str] = Counter()\\n    first: datetime | None = None\\n    last: datetime | None = None\\n    for raw in lines:\\n        line = raw.strip()\\n        if not line:\\n            continue\\n        total += 1\\n        parsed = _parse_line(line)\\n        if parsed is None:\\n            malformed += 1\\n            continue\\n        stamp, level, message = parsed\\n        by_level[level] += 1\\n        if level == \\"ERROR\\":\\n            errors[message] += 1\\n        if first is None or stamp < first:\\n            first = stamp\\n        if last is None or stamp > last:\\n            last = stamp\\n    top_errors = sorted(errors.items(), key=lambda item: (-item[1], item[0]))[:top]\\n    return {\\n        \\"total_lines\\": total,\\n        \\"by_level\\": by_level,\\n        \\"malformed\\": malformed,\\n        \\"first\\": first.strftime(TIME_FORMAT) if first else None,\\n        \\"last\\": last.strftime(TIME_FORMAT) if last else None,\\n        \\"top_errors\\": [{\\"message\\": m, \\"count\\": c} for m, c in top_errors],\\n    }\\n\\n\\ndef format_text(summary: dict) -> str:\\n    out = [f\\"Lines: {summary[\'total_lines\']} (malformed: {summary[\'malformed\']})\\"]\\n    out.append(\\"By level: \\" + \\", \\".join(f\\"{lvl} {n}\\" for lvl, n in summary[\\"by_level\\"].items()))\\n    if summary[\\"first\\"]:\\n        out.append(f\\"From {summary[\'first\']} to {summary[\'last\']}\\")\\n    else:\\n        out.append(\\"No timestamps found\\")\\n    if summary[\\"top_errors\\"]:\\n        out.append(\\"Top errors:\\")\\n        out.extend(f\\"  {e[\'count\']:>6}  {e[\'message\']}\\" for e in summary[\\"top_errors\\"])\\n    return \\"\\\\n\\".join(out)\\n\\n\\ndef _non_negative_int(text: str) -> int:\\n    try:\\n        value = int(text)\\n    except ValueError:\\n        raise argparse.ArgumentTypeError(f\\"not a whole number: {text!r}\\") from None\\n    if value < 0:\\n        raise argparse.ArgumentTypeError(f\\"must be 0 or more: {value}\\")\\n    return value\\n\\n\\ndef main(argv: list[str] | None = None) -> int:\\n    parser = argparse.ArgumentParser(description=\\"Summarise a shop server log file.\\")\\n    parser.add_argument(\\"path\\", help=\\"log file to read\\")\\n    parser.add_argument(\\"--top\\", type=_non_negative_int, default=5, help=\\"how many ERROR messages to list\\")\\n    parser.add_argument(\\"--json\\", action=\\"store_true\\", help=\\"print JSON instead of text\\")\\n    try:\\n        args = parser.parse_args(argv)\\n    except SystemExit as exc:  # argparse already printed the usage error\\n        return int(exc.code or 0)\\n    try:\\n        with open(args.path, encoding=\\"utf-8\\", errors=\\"replace\\") as fh:\\n            summary = summarise(fh, top=args.top)\\n    except OSError as exc:\\n        print(f\\"logsum: cannot read {args.path}: {exc.strerror or exc}\\", file=sys.stderr)\\n        return 1\\n    if args.json:\\n        print(json.dumps(summary, ensure_ascii=False))\\n    else:\\n        print(format_text(summary))\\n    return 0\\n\\n\\nif __name__ == \\"__main__\\":\\n    sys.exit(main())\\n"}, "starter": {}, "task_md": "Our shop server writes a log file with lines like `2026-10-01 10:00:03 ERROR payment gateway timeout` (date, time, level, message; levels are DEBUG, INFO, WARNING and ERROR). I want a small command-line tool to summarise it.\\nPlease write `logsum.py`, runnable as `python logsum.py PATH [--top N] [--json]`, with a `main(argv=None)` function that returns the exit code so it can be tested.\\nIt should report the total number of log lines (blank lines don\'t count), the count per level, how many lines couldn\'t be understood, the earliest and latest timestamp, and the N most common ERROR messages (default 5; ties sorted alphabetically).\\nWith `--json` it prints one JSON object with the keys `total_lines`, `by_level` (all four levels, 0 if none), `malformed`, `first`, `last` (as `\\"YYYY-MM-DD HH:MM:SS\\"`, or null) and `top_errors` (a list of `{\\"message\\": ..., \\"count\\": ...}`); without it, a short readable summary.\\nIt runs on the server from cron, so it must not blow up with a traceback on a bad file — print a clear message to stderr and return a non-zero exit code."}'),
    json.loads('{"hidden": "import copy\\n\\nimport pytest\\n\\nfrom catalog import paginate, search_products\\n\\nITEMS = [f\\"p{i}\\" for i in range(1, 46)]  # 45 items\\n\\n\\ndef test_first_page_starts_at_first_item():\\n    r = paginate(ITEMS, 1)\\n    assert r[\\"items\\"] == ITEMS[:20]\\n    assert r[\\"page\\"] == 1\\n\\n\\ndef test_middle_page():\\n    assert paginate(ITEMS, 2)[\\"items\\"] == ITEMS[20:40]\\n\\n\\ndef test_last_partial_page_is_shown():\\n    r = paginate(ITEMS, 3)\\n    assert r[\\"items\\"] == ITEMS[40:]\\n    assert r[\\"total_pages\\"] == 3\\n    assert r[\\"has_next\\"] is False\\n    assert paginate(ITEMS, 2)[\\"has_next\\"] is True\\n\\n\\ndef test_every_item_appears_exactly_once():\\n    seen = []\\n    for per_page in (1, 7, 20, 44, 45, 46):\\n        seen = []\\n        page = 1\\n        while True:\\n            r = paginate(ITEMS, page, per_page)\\n            seen.extend(r[\\"items\\"])\\n            if not r[\\"has_next\\"]:\\n                break\\n            page += 1\\n        assert seen == ITEMS, per_page\\n        assert r[\\"total_pages\\"] == page\\n\\n\\ndef test_exact_multiple():\\n    items = list(range(40))\\n    assert paginate(items, 2)[\\"total_pages\\"] == 2\\n    assert paginate(items, 2)[\\"has_next\\"] is False\\n    assert paginate(items, 2)[\\"items\\"] == list(range(20, 40))\\n\\n\\ndef test_empty_list():\\n    r = paginate([], 1)\\n    assert r[\\"items\\"] == []\\n    assert r[\\"total_pages\\"] == 0\\n    assert r[\\"has_next\\"] is False\\n\\n\\ndef test_page_past_the_end_is_empty():\\n    r = paginate(ITEMS, 9)\\n    assert r[\\"items\\"] == []\\n    assert r[\\"has_next\\"] is False\\n\\n\\n@pytest.mark.parametrize(\\"page\\", [0, -1])\\ndef test_bad_page_raises(page):\\n    with pytest.raises(ValueError):\\n        paginate(ITEMS, page)\\n\\n\\n@pytest.mark.parametrize(\\"per_page\\", [0, -5])\\ndef test_bad_page_size_raises(per_page):\\n    with pytest.raises(ValueError):\\n        paginate(ITEMS, 1, per_page)\\n\\n\\ndef test_input_not_modified():\\n    items = list(ITEMS)\\n    r = paginate(items, 1)\\n    r[\\"items\\"].append(\\"x\\")\\n    assert items == ITEMS\\n\\n\\nPRODUCTS = [\\n    {\\"name\\": \\"Aashirvaad Atta 5kg\\"},\\n    {\\"name\\": \\"Tata Salt 1kg\\"},\\n    {\\"name\\": \\"Fortune Chakki ATTA\\"},\\n    {\\"name\\": \\"Amul Butter\\"},\\n]\\n\\n\\n@pytest.mark.parametrize(\\"query\\", [\\"atta\\", \\"Atta\\", \\"ATTA\\", \\"  atta \\"])\\ndef test_search_ignores_case(query):\\n    r = search_products(PRODUCTS, query)\\n    assert [p[\\"name\\"] for p in r[\\"items\\"]] == [\\"Aashirvaad Atta 5kg\\", \\"Fortune Chakki ATTA\\"]\\n\\n\\ndef test_search_pages():\\n    products = [{\\"name\\": f\\"Parle-G {i}\\"} for i in range(25)] + [{\\"name\\": \\"Other\\"}]\\n    before = copy.deepcopy(products)\\n    r1 = search_products(products, \\"parle\\", page=1)\\n    r2 = search_products(products, \\"parle\\", page=2)\\n    assert len(r1[\\"items\\"]) == 20 and len(r2[\\"items\\"]) == 5\\n    assert r1[\\"total_pages\\"] == 2\\n    assert products == before\\n\\n\\ndef test_search_no_match():\\n    r = search_products(PRODUCTS, \\"ghee\\")\\n    assert r[\\"items\\"] == [] and r[\\"total_pages\\"] == 0\\n", "id": "08-pagination-bug", "meta": {"difficulty": 1, "files": ["catalog.py"], "kind": "bugfix", "timeout": 60}, "reference": {"catalog.py": "\\"\\"\\"Product listing helpers for the shop app.\\"\\"\\"\\n\\nimport math\\n\\n\\ndef paginate(items, page, per_page=20):\\n    \\"\\"\\"Return one page of items.\\n\\n    page is 1-based. The result is a dict with the page\'s items, the page\\n    number, the total number of pages and whether there is a next page.\\n    \\"\\"\\"\\n    if page < 1:\\n        raise ValueError(\\"page must be 1 or more\\")\\n    if per_page < 1:\\n        raise ValueError(\\"per_page must be 1 or more\\")\\n    total_pages = math.ceil(len(items) / per_page)\\n    start = (page - 1) * per_page\\n    end = start + per_page\\n    return {\\n        \\"items\\": list(items[start:end]),\\n        \\"page\\": page,\\n        \\"total_pages\\": total_pages,\\n        \\"has_next\\": page < total_pages,\\n    }\\n\\n\\ndef search_products(products, query, page=1, per_page=20):\\n    \\"\\"\\"Products whose name contains query (ignoring case), one page at a time.\\"\\"\\"\\n    q = query.strip().casefold()\\n    matches = [p for p in products if q in p[\\"name\\"].casefold()]\\n    return paginate(matches, page, per_page)\\n"}, "starter": {"catalog.py": "\\"\\"\\"Product listing helpers for the shop app.\\"\\"\\"\\n\\n\\ndef paginate(items, page, per_page=20):\\n    \\"\\"\\"Return one page of items.\\n\\n    page is 1-based. The result is a dict with the page\'s items, the page\\n    number, the total number of pages and whether there is a next page.\\n    \\"\\"\\"\\n    if page < 1:\\n        raise ValueError(\\"page must be 1 or more\\")\\n    total_pages = len(items) // per_page\\n    start = page * per_page\\n    end = start + per_page\\n    return {\\n        \\"items\\": items[start:end],\\n        \\"page\\": page,\\n        \\"total_pages\\": total_pages,\\n        \\"has_next\\": page < total_pages,\\n    }\\n\\n\\ndef search_products(products, query, page=1, per_page=20):\\n    \\"\\"\\"Products whose name contains query (ignoring case), one page at a time.\\"\\"\\"\\n    q = query.strip().lower()\\n    matches = [p for p in products if q in p[\\"name\\"]]\\n    return paginate(matches, page, per_page)\\n", "test_catalog.py": "from catalog import paginate\\n\\n\\ndef test_page_has_requested_size():\\n    items = list(range(100))\\n    assert len(paginate(items, 2)[\\"items\\"]) == 20\\n"}, "task_md": "Customers say the product list on our app is broken: page 1 never shows the first products, and the last few products never show up on any page. Searching is also odd: searching \\"atta\\" doesn\'t find \\"Aashirvaad Atta 5kg\\", though searching \\"Atta\\" does.\\nThe code is in `catalog.py` (`paginate(items, page, per_page=20)` and `search_products(products, query, page=1, per_page=20)`). Please fix it."}'),
    json.loads('{"hidden": "from datetime import date, datetime, timedelta, timezone\\n\\nimport pytest\\n\\nfrom delivery import promised_date\\n\\nUTC = timezone.utc\\nIST = timezone(timedelta(hours=5, minutes=30))\\n\\n\\ndef ist(*args):\\n    return datetime(*args, tzinfo=IST).astimezone(UTC)\\n\\n\\ndef test_reported_case_saturday_afternoon():\\n    # Sat 3 Oct 2026, 15:10 IST -> dispatch Sun 4 Oct -> deliver Mon 5 Oct\\n    assert promised_date(ist(2026, 10, 3, 15, 10)) == date(2026, 10, 5)\\n\\n\\ndef test_saturday_morning_skips_sunday():\\n    # dispatch Sat 3 Oct, next day is Sunday -> Monday\\n    assert promised_date(ist(2026, 10, 3, 9, 0)) == date(2026, 10, 5)\\n\\n\\ndef test_weekday_before_cutoff():\\n    assert promised_date(ist(2026, 10, 6, 13, 59)) == date(2026, 10, 7)\\n\\n\\ndef test_cutoff_is_two_pm_india_time():\\n    # 14:00 IST is 08:30 UTC: the UTC hour is below 14 but the order is after the cutoff\\n    assert promised_date(ist(2026, 10, 6, 14, 0)) == date(2026, 10, 8)\\n    assert promised_date(ist(2026, 10, 6, 19, 0)) == date(2026, 10, 8)\\n\\n\\ndef test_late_night_order_uses_india_date():\\n    # 00:30 IST on Wed 7 Oct is still Tue 6 Oct in UTC\\n    t = ist(2026, 10, 7, 0, 30)\\n    assert t.date() == date(2026, 10, 6)\\n    assert promised_date(t) == date(2026, 10, 8)\\n    # 23:45 IST Tue 6 Oct -> after cutoff -> dispatch Wed -> deliver Thu\\n    assert promised_date(ist(2026, 10, 6, 23, 45)) == date(2026, 10, 8)\\n\\n\\ndef test_any_timezone_is_accepted():\\n    t = datetime(2026, 10, 6, 12, 0, tzinfo=timezone(timedelta(hours=-5)))  # 22:30 IST\\n    assert promised_date(t) == date(2026, 10, 8)\\n\\n\\ndef test_holidays_are_skipped():\\n    holidays = {date(2026, 10, 7)}\\n    assert promised_date(ist(2026, 10, 6, 10, 0), holidays) == date(2026, 10, 8)\\n\\n\\ndef test_holiday_next_to_sunday():\\n    # dispatch Fri 2 Oct; Sat 3 holiday, Sun 4 Sunday, Mon 5 holiday -> Tue 6\\n    holidays = [date(2026, 10, 3), date(2026, 10, 5)]\\n    assert promised_date(ist(2026, 10, 2, 11, 0), holidays) == date(2026, 10, 6)\\n\\n\\ndef test_holidays_as_generator_or_tuple():\\n    gen = (d for d in [date(2026, 10, 8), date(2026, 10, 7)])\\n    assert promised_date(ist(2026, 10, 6, 10, 0), gen) == date(2026, 10, 9)\\n    assert promised_date(ist(2026, 10, 6, 10, 0), (date(2026, 10, 7),)) == date(2026, 10, 8)\\n\\n\\ndef test_never_promises_a_sunday():\\n    start = ist(2026, 1, 1, 0, 0)\\n    for hours in range(0, 24 * 60, 5):\\n        d = promised_date(start + timedelta(hours=hours))\\n        assert d.weekday() != 6\\n\\n\\ndef test_naive_datetime_is_rejected():\\n    with pytest.raises(ValueError):\\n        promised_date(datetime(2026, 10, 6, 10, 0))\\n\\n\\ndef test_returns_date_and_does_not_depend_on_today():\\n    d = promised_date(ist(2020, 2, 28, 10, 0))  # leap year: Fri 28 Feb -> Sat 29 Feb\\n    assert d == date(2020, 2, 29)\\n    assert type(d) is date\\n\\n\\ndef test_year_end():\\n    # Thu 31 Dec 2026 15:00 IST -> dispatch Fri 1 Jan -> deliver Sat 2 Jan 2027\\n    assert promised_date(ist(2026, 12, 31, 15, 0)) == date(2027, 1, 2)\\n", "id": "09-delivery-date-bug", "meta": {"difficulty": 2, "files": ["delivery.py"], "kind": "bugfix", "timeout": 60}, "reference": {"delivery.py": "\\"\\"\\"Delivery promises shown to customers at checkout.\\"\\"\\"\\n\\nfrom datetime import timedelta, timezone\\n\\nIST = timezone(timedelta(hours=5, minutes=30), \\"IST\\")\\nCUTOFF_HOUR = 14  # orders placed before 2 pm India time go out the same day\\nSUNDAY = 6  # date.weekday(): Monday is 0, Sunday is 6\\n\\n\\ndef promised_date(order_time, holidays=()):\\n    \\"\\"\\"Return the date we promise to deliver an order.\\n\\n    order_time must be a timezone-aware datetime (the server stores UTC).\\n    Orders placed before 2 pm India time are dispatched that day, later\\n    orders the next day. Delivery is on the first day after dispatch that\\n    is not a Sunday or one of the given holidays (dates).\\n    \\"\\"\\"\\n    if order_time.tzinfo is None or order_time.utcoffset() is None:\\n        raise ValueError(\\"order_time must be timezone-aware\\")\\n    local = order_time.astimezone(IST)\\n    holidays = frozenset(holidays)\\n    dispatch = local.date()\\n    if local.hour >= CUTOFF_HOUR:\\n        dispatch += timedelta(days=1)\\n    day = dispatch + timedelta(days=1)\\n    while day.weekday() == SUNDAY or day in holidays:\\n        day += timedelta(days=1)\\n    return day\\n"}, "starter": {"delivery.py": "\\"\\"\\"Delivery promises shown to customers at checkout.\\"\\"\\"\\n\\nfrom datetime import timedelta, timezone\\n\\nIST = timezone(timedelta(hours=5, minutes=30), \\"IST\\")\\nCUTOFF_HOUR = 14  # orders placed before 2 pm India time go out the same day\\n\\n\\ndef promised_date(order_time, holidays=()):\\n    \\"\\"\\"Return the date we promise to deliver an order.\\n\\n    order_time must be a timezone-aware datetime (the server stores UTC).\\n    Orders placed before 2 pm India time are dispatched that day, later\\n    orders the next day. Delivery is on the first day after dispatch that\\n    is not a Sunday or one of the given holidays (dates).\\n    \\"\\"\\"\\n    dispatch = order_time.date()\\n    if order_time.hour >= CUTOFF_HOUR:\\n        dispatch += timedelta(days=1)\\n    day = dispatch + timedelta(days=1)\\n    while day.weekday() == 7 or day in holidays:\\n        day += timedelta(days=1)\\n    return day\\n"}, "task_md": "Bug report from the shop: a customer ordered at 3:10 pm on Saturday 3 October (India time) and the app promised delivery on Sunday 4 October. We never deliver on Sundays, and an order after 2 pm should only be dispatched the next day anyway. A few late-night orders also seem to get the wrong date.\\nThe logic is `promised_date(order_time, holidays=())` in `delivery.py`; our server stores times in UTC. Please fix it."}'),
    json.loads('{"hidden": "import time\\nfrom decimal import Decimal\\n\\nimport pytest\\n\\nfrom billing import apply_discount, cart_total, split_bill\\n\\n\\ndef D(s):\\n    return Decimal(s)\\n\\n\\ndef test_cart_total_is_exact():\\n    assert cart_total([D(\\"0.10\\"), D(\\"0.20\\")]) == D(\\"0.30\\")\\n    assert str(cart_total([D(\\"0.10\\"), D(\\"0.20\\")])) in (\\"0.30\\", \\"0.3\\")\\n\\n\\ndef test_cart_total_empty():\\n    assert cart_total([]) == 0\\n    assert isinstance(cart_total([]), Decimal)\\n\\n\\ndef test_cart_total_accepts_generator():\\n    assert cart_total(D(\\"1.10\\") for _ in range(3)) == D(\\"3.30\\")\\n\\n\\ndef test_cart_total_many_items_exact_and_fast():\\n    prices = [D(\\"0.10\\")] * 100_000\\n    start = time.perf_counter()\\n    assert cart_total(prices) == D(\\"10000.00\\")\\n    assert time.perf_counter() - start < 2.0\\n\\n\\ndef test_cart_total_large_amounts_keep_paise():\\n    assert cart_total([D(\\"9999999999999.99\\"), D(\\"0.01\\")]) == D(\\"10000000000000.00\\")\\n\\n\\n@pytest.mark.parametrize(\\"price, percent, expected\\", [\\n    (\\"2.25\\", 10, \\"2.03\\"),     # 2.025 -> half up\\n    (\\"100.00\\", 0, \\"100.00\\"),\\n    (\\"100.00\\", 100, \\"0.00\\"),\\n    (\\"199.99\\", 15, \\"169.99\\"),  # 169.9915\\n    (\\"1.05\\", 50, \\"0.53\\"),      # 0.525 -> half up\\n    (\\"80.00\\", D(\\"12.5\\"), \\"70.00\\"),\\n])\\ndef test_discount_rounds_half_up(price, percent, expected):\\n    result = apply_discount(D(price), percent)\\n    assert isinstance(result, Decimal)\\n    assert result == D(expected)\\n\\n\\n@pytest.mark.parametrize(\\"percent\\", [-1, 101, 150])\\ndef test_discount_out_of_range(percent):\\n    with pytest.raises(ValueError):\\n        apply_discount(D(\\"10\\"), percent)\\n\\n\\ndef test_split_reported_case():\\n    assert split_bill(D(\\"100\\"), 3) == [D(\\"33.34\\"), D(\\"33.33\\"), D(\\"33.33\\")]\\n\\n\\ndef test_split_always_adds_up():\\n    for total in [\\"0.01\\", \\"0.05\\", \\"1.00\\", \\"10.01\\", \\"99.99\\", \\"100.00\\", \\"12345.67\\"]:\\n        for people in range(1, 12):\\n            shares = split_bill(D(total), people)\\n            assert len(shares) == people\\n            assert sum(shares) == D(total), (total, people)\\n            assert max(shares) - min(shares) <= D(\\"0.01\\")\\n            assert shares == sorted(shares, reverse=True)  # extra paise go to the first people\\n            assert all(isinstance(s, Decimal) for s in shares)\\n\\n\\ndef test_split_less_than_one_paisa_each():\\n    assert split_bill(D(\\"0.02\\"), 3) == [D(\\"0.01\\"), D(\\"0.01\\"), D(\\"0.00\\")]\\n\\n\\ndef test_split_one_person():\\n    assert split_bill(D(\\"57.50\\"), 1) == [D(\\"57.50\\")]\\n\\n\\n@pytest.mark.parametrize(\\"people\\", [0, -2])\\ndef test_split_bad_people(people):\\n    with pytest.raises(ValueError):\\n        split_bill(D(\\"100\\"), people)\\n\\n", "id": "10-money-rounding-bug", "meta": {"difficulty": 2, "files": ["billing.py"], "kind": "bugfix", "timeout": 60}, "reference": {"billing.py": "\\"\\"\\"Money helpers for the billing counter. All amounts are Decimal rupees.\\"\\"\\"\\n\\nfrom decimal import ROUND_HALF_UP, Decimal\\n\\nPAISA = Decimal(\\"0.01\\")\\n\\n\\ndef _to_decimal(value):\\n    if isinstance(value, float):\\n        # str() gives the shortest repr, e.g. 0.1 -> \\"0.1\\", not 0.1000000000000000055...\\n        return Decimal(str(value))\\n    return Decimal(value)\\n\\n\\ndef cart_total(prices):\\n    \\"\\"\\"Sum of the item prices.\\"\\"\\"\\n    return sum((_to_decimal(p) for p in prices), Decimal(0))\\n\\n\\ndef apply_discount(price, percent):\\n    \\"\\"\\"Price after a percentage discount, rounded half up to the paisa.\\n\\n    percent is a number from 0 to 100; anything else raises ValueError.\\n    \\"\\"\\"\\n    percent = _to_decimal(percent)\\n    if not percent.is_finite() or percent < 0 or percent > 100:\\n        raise ValueError(\\"percent must be between 0 and 100\\")\\n    result = _to_decimal(price) * (100 - percent) / 100\\n    return result.quantize(PAISA, rounding=ROUND_HALF_UP)\\n\\n\\ndef split_bill(total, people):\\n    \\"\\"\\"Split total between people.\\n\\n    Returns one share per person. The shares add up exactly to the total;\\n    when it does not divide evenly, the first people pay one paisa more.\\n    \\"\\"\\"\\n    if isinstance(people, bool) or not isinstance(people, int) or people < 1:\\n        raise ValueError(\\"people must be a whole number of at least 1\\")\\n    total_paise = int((_to_decimal(total) / PAISA).to_integral_value(rounding=ROUND_HALF_UP))\\n    base, extra = divmod(total_paise, people)\\n    return [Decimal(base + (1 if i < extra else 0)) * PAISA for i in range(people)]\\n"}, "starter": {"billing.py": "\\"\\"\\"Money helpers for the billing counter. All amounts are Decimal rupees.\\"\\"\\"\\n\\nfrom decimal import Decimal\\n\\n\\ndef cart_total(prices):\\n    \\"\\"\\"Sum of the item prices.\\"\\"\\"\\n    return Decimal(str(sum(float(p) for p in prices)))\\n\\n\\ndef apply_discount(price, percent):\\n    \\"\\"\\"Price after a percentage discount, rounded half up to the paisa.\\n\\n    percent is a number from 0 to 100; anything else raises ValueError.\\n    \\"\\"\\"\\n    if percent < 0 or percent > 100:\\n        raise ValueError(\\"percent must be between 0 and 100\\")\\n    result = float(price) * (1 - percent / 100)\\n    return Decimal(str(round(result, 2)))\\n\\n\\ndef split_bill(total, people):\\n    \\"\\"\\"Split total between people.\\n\\n    Returns one share per person. The shares add up exactly to the total;\\n    when it does not divide evenly, the first people pay one paisa more.\\n    \\"\\"\\"\\n    share = round(float(total) / people, 2)\\n    return [Decimal(str(share))] * people\\n"}, "task_md": "Three complaints from the counter this week, all in `billing.py`:\\na cart with items of ₹0.10 and ₹0.20 shows a total of ₹0.30000000000000004; splitting a ₹100 bill between 3 friends gives ₹33.33 each, so we are a paisa short; and a 10% discount on ₹2.25 comes out as ₹2.02, but our rule is to round half up, so it should be ₹2.03.\\nPlease fix these. Callers pass and expect `Decimal` amounts."}'),
    json.loads('{"hidden": "import copy\\nfrom decimal import Decimal\\n\\nimport pytest\\n\\nfrom cart import Cart\\n\\n\\ndef D(s):\\n    return Decimal(s)\\n\\n\\ndef test_new_carts_start_empty():\\n    a = Cart(\\"ravi\\")\\n    a.add(\\"ATTA-5KG\\", 1, D(\\"245\\"))\\n    b = Cart(\\"meena\\")\\n    assert b.lines() == []\\n    assert b.total() == 0\\n\\n\\ndef test_carts_are_independent():\\n    a = Cart(\\"ravi\\")\\n    b = Cart(\\"meena\\")\\n    a.add(\\"SALT-1KG\\", 2, D(\\"28\\"))\\n    b.add(\\"OIL-1L\\", 1, D(\\"180\\"))\\n    assert [line[\\"sku\\"] for line in a.lines()] == [\\"SALT-1KG\\"]\\n    assert [line[\\"sku\\"] for line in b.lines()] == [\\"OIL-1L\\"]\\n\\n\\ndef test_starting_items_are_not_shared_with_caller():\\n    start = [{\\"sku\\": \\"DAL-1KG\\", \\"qty\\": 1, \\"price\\": D(\\"120\\")}]\\n    before = copy.deepcopy(start)\\n    a = Cart(\\"ravi\\", start)\\n    b = Cart(\\"asha\\", start)\\n    a.add(\\"SALT-1KG\\", 1, D(\\"28\\"))\\n    a.add(\\"DAL-1KG\\", 1, D(\\"120\\"))\\n    assert start == before\\n    assert b.total() == D(\\"120\\")\\n    assert a.total() == D(\\"268\\")\\n\\n\\ndef test_same_product_merges_into_one_line():\\n    c = Cart(\\"ravi\\")\\n    c.add(\\"ATTA-5KG\\", 1, D(\\"245\\"))\\n    c.add(\\"SALT-1KG\\", 1, D(\\"28\\"))\\n    c.add(\\"ATTA-5KG\\", 2, D(\\"245\\"))\\n    lines = c.lines()\\n    assert [(line[\\"sku\\"], line[\\"qty\\"]) for line in lines] == [(\\"ATTA-5KG\\", 3), (\\"SALT-1KG\\", 1)]\\n    assert c.total() == D(\\"763\\")\\n\\n\\ndef test_lines_cannot_change_the_cart():\\n    c = Cart(\\"ravi\\")\\n    c.add(\\"A\\", 1, D(\\"10\\"))\\n    lines = c.lines()\\n    lines.append({\\"sku\\": \\"B\\", \\"qty\\": 1, \\"price\\": D(\\"99\\")})\\n    assert c.total() == D(\\"10\\")\\n    assert len(c.lines()) == 1\\n\\n\\ndef test_remove_and_missing_remove():\\n    c = Cart(\\"ravi\\")\\n    c.add(\\"A\\", 2, D(\\"10\\"))\\n    c.add(\\"B\\", 1, D(\\"5\\"))\\n    c.remove(\\"A\\")\\n    assert [line[\\"sku\\"] for line in c.lines()] == [\\"B\\"]\\n    with pytest.raises(KeyError):\\n        c.remove(\\"A\\")\\n\\n\\ndef test_add_after_remove_starts_fresh():\\n    c = Cart(\\"ravi\\")\\n    c.add(\\"A\\", 2, D(\\"10\\"))\\n    c.remove(\\"A\\")\\n    c.add(\\"A\\", 1, D(\\"10\\"))\\n    assert [(line[\\"sku\\"], line[\\"qty\\"]) for line in c.lines()] == [(\\"A\\", 1)]\\n\\n\\n@pytest.mark.parametrize(\\"qty\\", [0, -1])\\ndef test_bad_qty_still_rejected(qty):\\n    c = Cart(\\"ravi\\")\\n    c.add(\\"A\\", 1, D(\\"10\\"))\\n    with pytest.raises(ValueError):\\n        c.add(\\"A\\", qty, D(\\"10\\"))\\n    assert c.lines()[0][\\"qty\\"] == 1\\n\\n\\ndef test_price_accepts_strings():\\n    c = Cart(\\"ravi\\")\\n    c.add(\\"A\\", 3, \\"19.99\\")\\n    assert c.total() == D(\\"59.97\\")\\n", "id": "11-cart-shared-state-bug", "meta": {"difficulty": 1, "files": ["cart.py"], "kind": "bugfix", "timeout": 60}, "reference": {"cart.py": "\\"\\"\\"Shopping cart for the shop app.\\"\\"\\"\\n\\nfrom decimal import Decimal\\n\\n\\nclass Cart:\\n    def __init__(self, customer, items=None):\\n        \\"\\"\\"items: optional starting lines, each {\\"sku\\", \\"qty\\", \\"price\\"}.\\"\\"\\"\\n        self.customer = customer\\n        self.items = []\\n        for line in items or ():\\n            self.add(line[\\"sku\\"], line[\\"qty\\"], line[\\"price\\"])\\n\\n    def add(self, sku, qty=1, price=Decimal(0)):\\n        \\"\\"\\"Add qty units of sku at the given unit price. qty must be at least 1.\\"\\"\\"\\n        if qty < 1:\\n            raise ValueError(\\"qty must be at least 1\\")\\n        for line in self.items:\\n            if line[\\"sku\\"] == sku:\\n                line[\\"qty\\"] += qty\\n                line[\\"price\\"] = Decimal(price)\\n                return\\n        self.items.append({\\"sku\\": sku, \\"qty\\": qty, \\"price\\": Decimal(price)})\\n\\n    def remove(self, sku):\\n        \\"\\"\\"Remove the line for sku. Raises KeyError if it is not in the cart.\\"\\"\\"\\n        for line in self.items:\\n            if line[\\"sku\\"] == sku:\\n                self.items.remove(line)\\n                return\\n        raise KeyError(sku)\\n\\n    def lines(self):\\n        \\"\\"\\"The cart lines, in the order they were first added.\\"\\"\\"\\n        return [dict(line) for line in self.items]\\n\\n    def total(self):\\n        return sum((line[\\"price\\"] * line[\\"qty\\"] for line in self.items), Decimal(0))\\n"}, "starter": {"cart.py": "\\"\\"\\"Shopping cart for the shop app.\\"\\"\\"\\n\\nfrom decimal import Decimal\\n\\n\\nclass Cart:\\n    def __init__(self, customer, items=[]):\\n        \\"\\"\\"items: optional starting lines, each {\\"sku\\", \\"qty\\", \\"price\\"}.\\"\\"\\"\\n        self.customer = customer\\n        self.items = items\\n\\n    def add(self, sku, qty=1, price=Decimal(\\"0\\")):\\n        \\"\\"\\"Add qty units of sku at the given unit price. qty must be at least 1.\\"\\"\\"\\n        if qty < 1:\\n            raise ValueError(\\"qty must be at least 1\\")\\n        self.items.append({\\"sku\\": sku, \\"qty\\": qty, \\"price\\": Decimal(price)})\\n\\n    def remove(self, sku):\\n        \\"\\"\\"Remove the line for sku. Raises KeyError if it is not in the cart.\\"\\"\\"\\n        for line in self.items:\\n            if line[\\"sku\\"] == sku:\\n                self.items.remove(line)\\n                return\\n        raise KeyError(sku)\\n\\n    def lines(self):\\n        \\"\\"\\"The cart lines, in the order they were first added.\\"\\"\\"\\n        return self.items\\n\\n    def total(self):\\n        return sum((line[\\"price\\"] * line[\\"qty\\"] for line in self.items), Decimal(\\"0\\"))\\n", "test_cart.py": "from decimal import Decimal\\n\\nfrom cart import Cart\\n\\n\\ndef test_total():\\n    c = Cart(\\"ravi\\")\\n    c.add(\\"ATTA-5KG\\", 2, Decimal(\\"245\\"))\\n    assert c.total() == Decimal(\\"490\\")\\n"}, "task_md": "Two problems with the cart in our shop app (`cart.py`, class `Cart`). Sometimes a new customer\'s cart already has products in it that someone else added. And when a customer adds the same product twice, the cart shows two lines instead of one line with the quantities added together.\\nPlease fix both."}'),
    json.loads('{"hidden": "import contextlib\\nimport importlib\\nimport os\\nimport urllib.parse\\nfrom pathlib import Path\\n\\nimport pytest\\n\\nimport customers\\n\\nSECRET = \\"sk_live_51HxQe2kL9vR7tYb3\\"\\n\\n\\n@pytest.fixture\\ndef store(tmp_path):\\n    return customers.CustomerStore(str(tmp_path / \\"shop.db\\"))\\n\\n\\ndef test_add_and_get_round_trip(store):\\n    cid = store.add(\\"Ravi\\", \\"9876543210\\", \\"Pune\\")\\n    assert isinstance(cid, int)\\n    assert store.get_by_phone(\\"9876543210\\") == {\\"id\\": cid, \\"name\\": \\"Ravi\\", \\"phone\\": \\"9876543210\\", \\"city\\": \\"Pune\\"}\\n\\n\\ndef test_missing_city_stays_empty(store):\\n    store.add(\\"Meena\\", \\"9000000001\\")\\n    assert store.get_by_phone(\\"9000000001\\")[\\"city\\"] is None\\n\\n\\ndef test_names_with_quotes_work(store):\\n    cid = store.add(\\"Joseph D\'Souza\\", \\"9000000002\\", \\"Panaji\\")\\n    assert cid is not None\\n    assert store.get_by_phone(\\"9000000002\\")[\\"name\\"] == \\"Joseph D\'Souza\\"\\n    assert [c[\\"name\\"] for c in store.search(\\"d\'souza\\")] == [\\"Joseph D\'Souza\\"]\\n\\n\\ndef test_search_cannot_be_injected(store):\\n    store.add(\\"Ravi\\", \\"9876543210\\")\\n    store.add(\\"Asha\\", \\"9876543211\\")\\n    assert store.search(\\"\' OR \'1\'=\'1\\") == []\\n    assert store.search(\\"%\' OR 1=1 --\\") == []\\n    assert store.get_by_phone(\\"\' OR \'1\'=\'1\\") is None\\n    assert store.get_by_phone(\\"x\' OR phone LIKE \'%\\") is None\\n    with contextlib.suppress(Exception):\\n        store.search(\\"x\'; DROP TABLE customers; --\\")\\n    assert len(store.search(\\"\\")) == 2  # table is still there\\n\\n\\ndef test_search_ignores_case_and_sorts(store):\\n    store.add(\\"ravi kumar\\", \\"1\\")\\n    store.add(\\"Asha\\", \\"2\\")\\n    store.add(\\"Ravindra\\", \\"3\\")\\n    assert [c[\\"name\\"] for c in store.search(\\"RAVI\\")] == [\\"Ravindra\\", \\"ravi kumar\\"]\\n\\n\\ndef test_duplicate_phone_raises(store):\\n    store.add(\\"Ravi\\", \\"9876543210\\")\\n    with pytest.raises(customers.DuplicateCustomer):\\n        store.add(\\"Someone Else\\", \\"9876543210\\")\\n    assert store.get_by_phone(\\"9876543210\\")[\\"name\\"] == \\"Ravi\\"\\n\\n\\ndef test_other_errors_are_not_hidden(store):\\n    with pytest.raises(Exception) as info:\\n        store.add(None, \\"9000000003\\")\\n    assert not isinstance(info.value, customers.DuplicateCustomer)\\n    assert store.get_by_phone(\\"9000000003\\") is None\\n\\n\\ndef test_import_many_success(tmp_path):\\n    db = str(tmp_path / \\"shop.db\\")\\n    s = customers.CustomerStore(db)\\n    n = s.import_many([{\\"name\\": \\"A\\", \\"phone\\": \\"1\\"}, {\\"name\\": \\"B\\", \\"phone\\": \\"2\\", \\"city\\": \\"Nagpur\\"}])\\n    assert n == 2\\n    reopened = customers.CustomerStore(db)\\n    assert reopened.get_by_phone(\\"2\\")[\\"city\\"] == \\"Nagpur\\"\\n\\n\\ndef test_import_many_is_all_or_nothing(store):\\n    store.add(\\"Existing\\", \\"5\\")\\n    with pytest.raises(customers.DuplicateCustomer):\\n        store.import_many([{\\"name\\": \\"A\\", \\"phone\\": \\"1\\"}, {\\"name\\": \\"B\\", \\"phone\\": \\"5\\"}, {\\"name\\": \\"C\\", \\"phone\\": \\"3\\"}])\\n    assert store.get_by_phone(\\"1\\") is None\\n    assert store.get_by_phone(\\"3\\") is None\\n    with pytest.raises(customers.DuplicateCustomer):\\n        store.import_many([{\\"name\\": \\"A\\", \\"phone\\": \\"7\\"}, {\\"name\\": \\"A again\\", \\"phone\\": \\"7\\"}])\\n    assert store.get_by_phone(\\"7\\") is None\\n    assert [c[\\"phone\\"] for c in store.search(\\"\\")] == [\\"5\\"]\\n\\n\\ndef test_data_is_saved(tmp_path):\\n    db = str(tmp_path / \\"shop.db\\")\\n    customers.CustomerStore(db).add(\\"Ravi\\", \\"9876543210\\")\\n    assert customers.CustomerStore(db).get_by_phone(\\"9876543210\\")[\\"name\\"] == \\"Ravi\\"\\n\\n\\ndef _source_files():\\n    root = Path.cwd()\\n    this_test = Path(__file__).resolve()\\n    for path in root.rglob(\\"*\\"):\\n        parts = set(path.relative_to(root).parts)\\n        if path.is_file() and path.resolve() != this_test and not parts & {\\"hidden\\", \\".git\\", \\"__pycache__\\"}:\\n            yield path\\n\\n\\ndef test_no_api_key_in_the_code():\\n    for path in _source_files():\\n        assert SECRET not in path.read_text(encoding=\\"utf-8\\", errors=\\"ignore\\"), path\\n\\n\\nclass FakeOpener:\\n    def __init__(self):\\n        self.calls = []\\n\\n    def __call__(self, request, *args, **kwargs):\\n        if isinstance(request, str):\\n            url, data = request, None\\n        else:\\n            url, data = request.full_url, request.data\\n        params = urllib.parse.parse_qs(urllib.parse.urlsplit(url).query)\\n        if data:\\n            params.update(urllib.parse.parse_qs(data.decode() if isinstance(data, bytes) else data))\\n        self.calls.append(params)\\n        return \\"sent\\"\\n\\n\\ndef test_sms_uses_key_from_environment_and_encodes_text(monkeypatch):\\n    monkeypatch.setenv(\\"SMS_API_KEY\\", \\"test-key-123\\")\\n    mod = importlib.reload(customers)\\n    opener = FakeOpener()\\n    mod.send_welcome_sms(\\"+919876543210\\", \\"Ravi & Sons #1\\", opener=opener)\\n    assert len(opener.calls) == 1\\n    params = opener.calls[0]\\n    assert params[\\"key\\"] == [\\"test-key-123\\"]\\n    assert params[\\"to\\"] == [\\"+919876543210\\"]\\n    assert \\"Ravi & Sons #1\\" in params[\\"msg\\"][0]\\n\\n\\ndef test_sms_without_key_is_not_sent(monkeypatch):\\n    monkeypatch.delenv(\\"SMS_API_KEY\\", raising=False)\\n    mod = importlib.reload(customers)\\n    opener = FakeOpener()\\n    with contextlib.suppress(Exception):\\n        mod.send_welcome_sms(\\"9876543210\\", \\"Ravi\\", opener=opener)\\n    assert opener.calls == [] or all(p.get(\\"key\\", [\\"\\"])[0] not in (\\"\\", \\"None\\", SECRET) for p in opener.calls)\\n    assert \\"SMS_API_KEY\\" not in os.environ\\n", "id": "12-review-customer-store", "meta": {"difficulty": 3, "files": ["customers.py"], "kind": "review", "timeout": 60}, "reference": {"customers.py": "\\"\\"\\"Customer records for the shop app, stored in SQLite.\\"\\"\\"\\n\\nimport os\\nimport sqlite3\\nimport urllib.parse\\nimport urllib.request\\n\\nSMS_URL = \\"https://sms.example.in/send\\"\\nSMS_KEY_ENV = \\"SMS_API_KEY\\"  # provided by ops in the server environment; never commit the key\\n\\n\\nclass DuplicateCustomer(Exception):\\n    \\"\\"\\"A customer with this phone number is already registered.\\"\\"\\"\\n\\n\\ndef _row_to_dict(row):\\n    return {\\"id\\": row[0], \\"name\\": row[1], \\"phone\\": row[2], \\"city\\": row[3]}\\n\\n\\nclass CustomerStore:\\n    def __init__(self, path):\\n        self.conn = sqlite3.connect(path)\\n        with self.conn:\\n            self.conn.execute(\\n                \\"CREATE TABLE IF NOT EXISTS customers (\\"\\n                \\"id INTEGER PRIMARY KEY, name TEXT NOT NULL, phone TEXT NOT NULL UNIQUE, city TEXT)\\"\\n            )\\n\\n    def close(self):\\n        self.conn.close()\\n\\n    def _insert(self, name, phone, city):\\n        try:\\n            cur = self.conn.execute(\\n                \\"INSERT INTO customers (name, phone, city) VALUES (?, ?, ?)\\", (name, phone, city)\\n            )\\n        except sqlite3.IntegrityError as exc:\\n            if phone is not None and self.get_by_phone(phone) is not None:\\n                raise DuplicateCustomer(f\\"phone {phone} is already registered\\") from exc\\n            raise\\n        return cur.lastrowid\\n\\n    def add(self, name, phone, city=None):\\n        \\"\\"\\"Add a customer and return the new id.\\n\\n        Raises DuplicateCustomer if the phone number is already registered.\\n        \\"\\"\\"\\n        with self.conn:  # commits on success, rolls back on error\\n            return self._insert(name, phone, city)\\n\\n    def get_by_phone(self, phone):\\n        \\"\\"\\"The customer with this phone number as a dict, or None.\\"\\"\\"\\n        row = self.conn.execute(\\n            \\"SELECT id, name, phone, city FROM customers WHERE phone = ?\\", (phone,)\\n        ).fetchone()\\n        return None if row is None else _row_to_dict(row)\\n\\n    def search(self, text):\\n        \\"\\"\\"Customers whose name contains text (ignoring case), sorted by name.\\"\\"\\"\\n        pattern = \\"%\\" + text.replace(\\"\\\\\\\\\\", \\"\\\\\\\\\\\\\\\\\\").replace(\\"%\\", \\"\\\\\\\\%\\").replace(\\"_\\", \\"\\\\\\\\_\\") + \\"%\\"\\n        rows = self.conn.execute(\\n            \\"SELECT id, name, phone, city FROM customers WHERE name LIKE ? ESCAPE \'\\\\\\\\\' ORDER BY name\\",\\n            (pattern,),\\n        ).fetchall()\\n        return [_row_to_dict(r) for r in rows]\\n\\n    def import_many(self, customers):\\n        \\"\\"\\"Add many customers (dicts with name, phone and optional city).\\n\\n        All or nothing: if any customer cannot be added, none are added and\\n        the error is raised. Returns the number of customers added.\\n        \\"\\"\\"\\n        added = 0\\n        with self.conn:  # one transaction: rolled back if any insert fails\\n            for c in customers:\\n                self._insert(c[\\"name\\"], c[\\"phone\\"], c.get(\\"city\\"))\\n                added += 1\\n        return added\\n\\n\\ndef send_welcome_sms(phone, name, opener=urllib.request.urlopen):\\n    \\"\\"\\"Send the welcome SMS through the gateway.\\"\\"\\"\\n    key = os.environ.get(SMS_KEY_ENV)\\n    if not key:\\n        raise RuntimeError(f\\"{SMS_KEY_ENV} is not set; cannot send SMS\\")\\n    query = urllib.parse.urlencode(\\n        {\\"key\\": key, \\"to\\": phone, \\"msg\\": f\\"Welcome to Sharma Kirana, {name}!\\"}\\n    )\\n    return opener(f\\"{SMS_URL}?{query}\\")\\n"}, "starter": {"customers.py": "\\"\\"\\"Customer records for the shop app, stored in SQLite.\\"\\"\\"\\n\\nimport sqlite3\\nimport urllib.request\\n\\n# SMS gateway credentials. On the server, ops provide SMS_API_KEY in the environment.\\nSMS_API_KEY = \\"sk_live_51HxQe2kL9vR7tYb3\\"\\nSMS_URL = \\"https://sms.example.in/send\\"\\n\\n\\nclass DuplicateCustomer(Exception):\\n    \\"\\"\\"A customer with this phone number is already registered.\\"\\"\\"\\n\\n\\nclass CustomerStore:\\n    def __init__(self, path):\\n        self.conn = sqlite3.connect(path)\\n        self.conn.execute(\\n            \\"CREATE TABLE IF NOT EXISTS customers (\\"\\n            \\"id INTEGER PRIMARY KEY, name TEXT NOT NULL, phone TEXT NOT NULL UNIQUE, city TEXT)\\"\\n        )\\n        self.conn.commit()\\n\\n    def add(self, name, phone, city=None):\\n        \\"\\"\\"Add a customer and return the new id.\\n\\n        Raises DuplicateCustomer if the phone number is already registered.\\n        \\"\\"\\"\\n        try:\\n            cur = self.conn.execute(\\n                f\\"INSERT INTO customers (name, phone, city) VALUES (\'{name}\', \'{phone}\', \'{city}\')\\"\\n            )\\n            self.conn.commit()\\n            return cur.lastrowid\\n        except:  # noqa: E722\\n            return None\\n\\n    def get_by_phone(self, phone):\\n        \\"\\"\\"The customer with this phone number as a dict, or None.\\"\\"\\"\\n        row = self.conn.execute(\\n            f\\"SELECT id, name, phone, city FROM customers WHERE phone = \'{phone}\'\\"\\n        ).fetchone()\\n        if row is None:\\n            return None\\n        return {\\"id\\": row[0], \\"name\\": row[1], \\"phone\\": row[2], \\"city\\": row[3]}\\n\\n    def search(self, text):\\n        \\"\\"\\"Customers whose name contains text (ignoring case), sorted by name.\\"\\"\\"\\n        rows = self.conn.execute(\\n            f\\"SELECT id, name, phone, city FROM customers WHERE name LIKE \'%{text}%\' ORDER BY name\\"\\n        ).fetchall()\\n        return [{\\"id\\": r[0], \\"name\\": r[1], \\"phone\\": r[2], \\"city\\": r[3]} for r in rows]\\n\\n    def import_many(self, customers):\\n        \\"\\"\\"Add many customers (dicts with name, phone and optional city).\\n\\n        All or nothing: if any customer cannot be added, none are added and\\n        the error is raised. Returns the number of customers added.\\n        \\"\\"\\"\\n        added = 0\\n        for c in customers:\\n            if self.add(c[\\"name\\"], c[\\"phone\\"], c.get(\\"city\\")) is not None:\\n                added += 1\\n        return added\\n\\n\\ndef send_welcome_sms(phone, name, opener=urllib.request.urlopen):\\n    \\"\\"\\"Send the welcome SMS through the gateway.\\"\\"\\"\\n    url = f\\"{SMS_URL}?key={SMS_API_KEY}&to={phone}&msg=Welcome to Sharma Kirana, {name}!\\"\\n    return opener(url)\\n", "test_customers.py": "from customers import CustomerStore\\n\\n\\ndef test_add_and_get(tmp_path):\\n    store = CustomerStore(tmp_path / \\"shop.db\\")\\n    cid = store.add(\\"Ravi\\", \\"9876543210\\", \\"Pune\\")\\n    assert store.get_by_phone(\\"9876543210\\")[\\"id\\"] == cid\\n"}, "task_md": "We are about to put `customers.py` live on the shop server; a contractor wrote it. Please review it before release and fix whatever is wrong, without changing how the rest of the app calls it (`CustomerStore(path)` with `add`, `get_by_phone`, `search` and `import_many`, the `DuplicateCustomer` exception, and `send_welcome_sms`)."}'),
    json.loads('{"hidden": "import contextlib\\nimport gzip\\nimport hashlib\\nimport logging\\nimport random\\nimport re\\nfrom pathlib import Path\\n\\nimport pytest\\n\\nimport invoices\\n\\nPASSWORD = \\"Gup!t-Pa55word\\"\\nUSERS = {\\"ravi\\": hashlib.sha256(PASSWORD.encode()).hexdigest()}\\n\\n\\n@pytest.fixture\\ndef shop(tmp_path):\\n    base = tmp_path / \\"invoices\\"\\n    (base / \\"2026-10\\").mkdir(parents=True)\\n    (base / \\"2026-10\\" / \\"INV-0001.txt\\").write_bytes(b\\"Invoice 1: Atta 5kg x2 = 490.00\\\\n\\")\\n    (base / \\"INV 0002.txt\\").write_bytes(b\\"Invoice 2\\\\n\\")\\n    (tmp_path / \\"secret.txt\\").write_bytes(b\\"DB_PASSWORD=hunter2\\\\n\\")\\n    return base\\n\\n\\ndef test_read_invoice_in_month_folder(shop):\\n    assert invoices.read_invoice(str(shop), \\"2026-10/INV-0001.txt\\") == b\\"Invoice 1: Atta 5kg x2 = 490.00\\\\n\\"\\n    assert invoices.read_invoice(shop, \\"INV 0002.txt\\") == b\\"Invoice 2\\\\n\\"\\n\\n\\ndef test_missing_invoice_raises(shop):\\n    with pytest.raises(FileNotFoundError):\\n        invoices.read_invoice(str(shop), \\"2026-10/INV-9999.txt\\")\\n\\n\\n@pytest.mark.parametrize(\\"name\\", [\\"../secret.txt\\", \\"2026-10/../../secret.txt\\", \\"..\\\\\\\\secret.txt\\"])\\ndef test_read_cannot_escape_folder(shop, name):\\n    try:\\n        data = invoices.read_invoice(str(shop), name)\\n    except Exception:\\n        return\\n    assert b\\"hunter2\\" not in data\\n\\n\\ndef test_read_rejects_absolute_path(shop):\\n    secret = shop.parent / \\"secret.txt\\"\\n    try:\\n        data = invoices.read_invoice(str(shop), str(secret))\\n    except Exception:\\n        return\\n    assert b\\"hunter2\\" not in data\\n\\n\\ndef test_compress_works_and_keeps_original(shop):\\n    out = Path(invoices.compress_invoice(str(shop), \\"2026-10/INV-0001.txt\\"))\\n    assert out == shop / \\"2026-10\\" / \\"INV-0001.txt.gz\\"\\n    assert gzip.decompress(out.read_bytes()) == b\\"Invoice 1: Atta 5kg x2 = 490.00\\\\n\\"\\n    assert (shop / \\"2026-10\\" / \\"INV-0001.txt\\").exists()\\n\\n\\ndef test_compress_name_with_space(shop):\\n    out = Path(invoices.compress_invoice(str(shop), \\"INV 0002.txt\\"))\\n    assert gzip.decompress(out.read_bytes()) == b\\"Invoice 2\\\\n\\"\\n\\n\\n@pytest.mark.parametrize(\\"name\\", [\\n    \\"INV 0002.txt; touch pwned\\",\\n    \\"INV-0001.txt && touch pwned\\",\\n    \\"$(touch pwned)\\",\\n    \\"`touch pwned`\\",\\n])\\ndef test_compress_cannot_run_commands(shop, monkeypatch, name):\\n    monkeypatch.chdir(shop)\\n    with contextlib.suppress(Exception):\\n        invoices.compress_invoice(str(shop), name)\\n    assert not list(shop.parent.rglob(\\"pwned*\\"))\\n\\n\\ndef test_compress_cannot_write_outside(shop):\\n    with contextlib.suppress(Exception):\\n        invoices.compress_invoice(str(shop), \\"../secret.txt\\")\\n    assert not (shop.parent / \\"secret.txt.gz\\").exists()\\n\\n\\ndef test_login_ok_and_failed():\\n    token = invoices.login(\\"ravi\\", PASSWORD, USERS)\\n    assert isinstance(token, str) and token\\n    assert invoices.login(\\"ravi\\", \\"wrong\\", USERS) is None\\n    assert invoices.login(\\"nobody\\", PASSWORD, USERS) is None\\n\\n\\ndef test_password_never_logged(caplog):\\n    caplog.set_level(logging.DEBUG)\\n    invoices.login(\\"ravi\\", PASSWORD, USERS)\\n    invoices.login(\\"ravi\\", PASSWORD + \\"x\\", USERS)\\n    text = \\"\\\\n\\".join(r.getMessage() for r in caplog.records)\\n    assert PASSWORD not in text\\n    assert \\"ravi\\" in text  # still useful logs\\n\\n\\ndef test_tokens_are_url_safe_and_long():\\n    for _ in range(50):\\n        t = invoices.new_download_token()\\n        assert re.fullmatch(r\\"[A-Za-z0-9_\\\\-]+\\", t)\\n        assert len(t) >= 22\\n\\n\\ndef test_tokens_are_unique():\\n    assert len({invoices.new_download_token() for _ in range(2000)}) == 2000\\n\\n\\ndef test_tokens_cannot_be_predicted_from_random_seed():\\n    random.seed(1234)\\n    a = invoices.new_download_token()\\n    random.seed(1234)\\n    b = invoices.new_download_token()\\n    assert a != b\\n", "id": "13-review-invoice-downloads", "meta": {"difficulty": 2, "files": ["invoices.py"], "kind": "review", "timeout": 60}, "reference": {"invoices.py": "\\"\\"\\"Invoice downloads for the shop\'s customer portal.\\"\\"\\"\\n\\nimport gzip\\nimport hashlib\\nimport hmac\\nimport logging\\nimport secrets\\nimport shutil\\nfrom pathlib import Path\\n\\nlog = logging.getLogger(\\"invoices\\")\\n\\n\\ndef _safe_path(base_dir, name):\\n    \\"\\"\\"Resolve name inside base_dir; refuse anything that points outside it.\\"\\"\\"\\n    base = Path(base_dir).resolve()\\n    candidate = (base / name).resolve()\\n    if not candidate.is_relative_to(base) or candidate == base:\\n        raise ValueError(f\\"invalid invoice name: {name!r}\\")\\n    return candidate\\n\\n\\ndef read_invoice(base_dir, name):\\n    \\"\\"\\"Return the bytes of invoice `name` stored under base_dir.\\n\\n    name comes from the download link and may include the month folder,\\n    e.g. \\"2026-10/INV-0001.txt\\".\\n    \\"\\"\\"\\n    return _safe_path(base_dir, name).read_bytes()\\n\\n\\ndef compress_invoice(base_dir, name):\\n    \\"\\"\\"Write a gzip copy of the invoice next to it (name + \\".gz\\").\\n\\n    The original file is kept. Returns the path of the .gz file.\\n    \\"\\"\\"\\n    path = _safe_path(base_dir, name)\\n    target = path.with_name(path.name + \\".gz\\")\\n    with open(path, \\"rb\\") as src, gzip.open(target, \\"wb\\") as dst:\\n        shutil.copyfileobj(src, dst)\\n    return str(target)\\n\\n\\ndef new_download_token():\\n    \\"\\"\\"A random token for a one-time download link (URL-safe).\\"\\"\\"\\n    return secrets.token_urlsafe(32)\\n\\n\\ndef login(username, password, users):\\n    \\"\\"\\"Check a password; users maps username -> sha256 hex digest of the password.\\n\\n    Returns a new download token on success, None otherwise.\\n    \\"\\"\\"\\n    log.info(\\"login attempt user=%s\\", username)\\n    expected = users.get(username)\\n    actual = hashlib.sha256(password.encode()).hexdigest()\\n    if expected and hmac.compare_digest(actual, expected):\\n        log.info(\\"login ok for %s\\", username)\\n        return new_download_token()\\n    log.warning(\\"failed login for %s\\", username)\\n    return None\\n"}, "starter": {"invoices.py": "\\"\\"\\"Invoice downloads for the shop\'s customer portal.\\"\\"\\"\\n\\nimport hashlib\\nimport logging\\nimport os\\nimport random\\nimport string\\nimport subprocess\\n\\nlog = logging.getLogger(\\"invoices\\")\\n\\n\\ndef read_invoice(base_dir, name):\\n    \\"\\"\\"Return the bytes of invoice `name` stored under base_dir.\\n\\n    name comes from the download link and may include the month folder,\\n    e.g. \\"2026-10/INV-0001.txt\\".\\n    \\"\\"\\"\\n    with open(os.path.join(base_dir, name), \\"rb\\") as f:\\n        return f.read()\\n\\n\\ndef compress_invoice(base_dir, name):\\n    \\"\\"\\"Write a gzip copy of the invoice next to it (name + \\".gz\\").\\n\\n    The original file is kept. Returns the path of the .gz file.\\n    \\"\\"\\"\\n    path = os.path.join(base_dir, name)\\n    subprocess.run(f\\"gzip -kf {path}\\", shell=True, check=True)\\n    return path + \\".gz\\"\\n\\n\\ndef new_download_token():\\n    \\"\\"\\"A random token for a one-time download link (URL-safe).\\"\\"\\"\\n    return \\"\\".join(random.choice(string.ascii_letters + string.digits) for _ in range(32))\\n\\n\\ndef login(username, password, users):\\n    \\"\\"\\"Check a password; users maps username -> sha256 hex digest of the password.\\n\\n    Returns a new download token on success, None otherwise.\\n    \\"\\"\\"\\n    log.info(\\"login attempt user=%s password=%s\\", username, password)\\n    expected = users.get(username)\\n    if expected and hashlib.sha256(password.encode()).hexdigest() == expected:\\n        log.info(\\"login ok for %s\\", username)\\n        return new_download_token()\\n    log.warning(\\"failed login for %s\\", username)\\n    return None\\n"}, "task_md": "`invoices.py` is the part of our customer portal that lets people log in and download their invoices. It goes on the public internet next week. Please review it before release and fix what\'s wrong; other code calls `read_invoice`, `compress_invoice`, `new_download_token` and `login`, so keep those working the same way."}'),
    json.loads('{"hidden": "import copy\\nimport math\\nfrom decimal import Decimal\\n\\nimport pytest\\n\\nfrom shipping import shipping_fee\\n\\n\\ndef legacy_fee(order):\\n    \\"\\"\\"The fee rules exactly as they were before the clean-up (test oracle).\\"\\"\\"\\n    rates = {\\"local\\": (30, 10, 8, 499), \\"state\\": (50, 15, 12, 999), \\"national\\": (80, 25, 20, 1499)}\\n    zone = order[\\"zone\\"].lower()\\n    if zone not in rates:\\n        raise ValueError(zone)\\n    w = order[\\"weight_kg\\"]\\n    if w <= 0 or w > 30:\\n        raise ValueError(w)\\n    first, to5, above5, free_from = rates[zone]\\n    if order[\\"subtotal\\"] >= free_from:\\n        fee = 0\\n    elif w <= 1:\\n        fee = first\\n    elif w <= 5:\\n        fee = first + math.ceil(w - 1) * to5\\n    else:\\n        fee = first + 4 * to5 + math.ceil(w - 5) * above5\\n    if order.get(\\"cod\\"):\\n        fee += max(25, math.ceil(order[\\"subtotal\\"] * 2 / 100))\\n    return fee\\n\\n\\nWEIGHTS = [0.2, 1, 1.01, 2.5, 4.999, 5, 5.0001, 5.5, 12, 29.9, 30]\\nSUBTOTALS = [0, 100, 498.99, 499, 998, 999, 1250, 1498.5, 1499, 1998, 1999, 2500, 10000]\\n\\n\\n@pytest.mark.parametrize(\\"zone\\", [\\"local\\", \\"state\\", \\"national\\"])\\ndef test_existing_zones_unchanged(zone):\\n    for w in WEIGHTS:\\n        for s in SUBTOTALS:\\n            for cod in (False, True):\\n                order = {\\"zone\\": zone, \\"weight_kg\\": w, \\"subtotal\\": s, \\"cod\\": cod}\\n                assert shipping_fee(order) == legacy_fee(order), order\\n\\n\\n@pytest.mark.parametrize(\\"order, fee\\", [\\n    ({\\"zone\\": \\"local\\", \\"weight_kg\\": 0.5, \\"subtotal\\": 200}, 30),\\n    ({\\"zone\\": \\"local\\", \\"weight_kg\\": 3, \\"subtotal\\": 200}, 50),\\n    ({\\"zone\\": \\"state\\", \\"weight_kg\\": 7.2, \\"subtotal\\": 500}, 146),\\n    ({\\"zone\\": \\"national\\", \\"weight_kg\\": 5, \\"subtotal\\": 1000, \\"cod\\": True}, 205),\\n    ({\\"zone\\": \\"national\\", \\"weight_kg\\": 5, \\"subtotal\\": 1499, \\"cod\\": True}, 30),\\n])\\ndef test_known_fees(order, fee):\\n    assert shipping_fee(order) == fee\\n\\n\\ndef test_cod_defaults_to_false_and_zone_case():\\n    assert shipping_fee({\\"zone\\": \\"Local\\", \\"weight_kg\\": 1, \\"subtotal\\": 100}) == 30\\n    assert shipping_fee({\\"zone\\": \\"STATE\\", \\"weight_kg\\": 1, \\"subtotal\\": 100, \\"cod\\": False}) == 50\\n\\n\\ndef test_decimal_subtotal():\\n    assert shipping_fee({\\"zone\\": \\"local\\", \\"weight_kg\\": 2, \\"subtotal\\": Decimal(\\"498.99\\"), \\"cod\\": True}) == 40 + 25\\n    assert shipping_fee({\\"zone\\": \\"local\\", \\"weight_kg\\": 2, \\"subtotal\\": Decimal(\\"499.00\\")}) == 0\\n    assert shipping_fee({\\"zone\\": \\"state\\", \\"weight_kg\\": 2, \\"subtotal\\": Decimal(\\"1300.50\\"), \\"cod\\": True}) == 27\\n\\n\\ndef test_fee_is_an_int():\\n    for zone in (\\"local\\", \\"state\\", \\"national\\", \\"northeast\\"):\\n        fee = shipping_fee({\\"zone\\": zone, \\"weight_kg\\": 2.5, \\"subtotal\\": 300.0, \\"cod\\": True})\\n        assert isinstance(fee, int) and not isinstance(fee, bool)\\n\\n\\n@pytest.mark.parametrize(\\"zone\\", [\\"local\\", \\"state\\", \\"national\\", \\"northeast\\"])\\n@pytest.mark.parametrize(\\"weight\\", [0, -1, 30.01, 45])\\ndef test_bad_weight_raises(zone, weight):\\n    with pytest.raises(ValueError):\\n        shipping_fee({\\"zone\\": zone, \\"weight_kg\\": weight, \\"subtotal\\": 100})\\n\\n\\n@pytest.mark.parametrize(\\"zone\\", [\\"mars\\", \\"\\", \\"north east\\"])\\ndef test_unknown_zone_raises(zone):\\n    with pytest.raises(ValueError):\\n        shipping_fee({\\"zone\\": zone, \\"weight_kg\\": 1, \\"subtotal\\": 100})\\n\\n\\n@pytest.mark.parametrize(\\n    \\"weight, fee\\", [(0.5, 120), (1, 120), (1.2, 155), (3, 190), (5, 260), (5.1, 290), (7.2, 350), (30, 1010)]\\n)\\ndef test_northeast_rates(weight, fee):\\n    assert shipping_fee({\\"zone\\": \\"northeast\\", \\"weight_kg\\": weight, \\"subtotal\\": 500}) == fee\\n\\n\\ndef test_northeast_free_shipping_and_cod():\\n    assert shipping_fee({\\"zone\\": \\"northeast\\", \\"weight_kg\\": 3, \\"subtotal\\": 1998.99}) == 190\\n    assert shipping_fee({\\"zone\\": \\"northeast\\", \\"weight_kg\\": 3, \\"subtotal\\": 1999}) == 0\\n    assert shipping_fee({\\"zone\\": \\"northeast\\", \\"weight_kg\\": 3, \\"subtotal\\": 1999, \\"cod\\": True}) == 40\\n    assert shipping_fee({\\"zone\\": \\"Northeast\\", \\"weight_kg\\": 3, \\"subtotal\\": 100, \\"cod\\": True}) == 215\\n\\n\\ndef test_order_is_not_modified():\\n    order = {\\"zone\\": \\"northeast\\", \\"weight_kg\\": 3, \\"subtotal\\": 100, \\"cod\\": True}\\n    before = copy.deepcopy(order)\\n    shipping_fee(order)\\n    shipping_fee(order)\\n    assert order == before\\n", "id": "14-refactor-shipping-fees", "meta": {"difficulty": 2, "files": ["shipping.py"], "kind": "refactor", "timeout": 60}, "reference": {"shipping.py": "\\"\\"\\"Courier shipping fees per zone.\\n\\nTo add or change a zone, edit ZONES: every zone uses the same rule\\n(a base fee for the first kg, a per-kg rate up to 5 kg, another per-kg rate\\nabove that, and a subtotal from which shipping is free).\\n\\"\\"\\"\\n\\nimport math\\nfrom dataclasses import dataclass\\n\\nMAX_WEIGHT_KG = 30\\nCOD_PERCENT = 2\\nCOD_MINIMUM = 25\\n\\n\\n@dataclass(frozen=True)\\nclass ZoneRates:\\n    first_kg: int          # fee for the first kg\\n    per_kg_to_5: int       # each further kg (or part of one) up to 5 kg\\n    per_kg_above_5: int    # each kg (or part of one) above 5 kg\\n    free_from: int         # subtotal from which shipping is free\\n\\n    def fee(self, weight_kg):\\n        if weight_kg <= 1:\\n            return self.first_kg\\n        if weight_kg <= 5:\\n            return self.first_kg + math.ceil(weight_kg - 1) * self.per_kg_to_5\\n        return self.first_kg + 4 * self.per_kg_to_5 + math.ceil(weight_kg - 5) * self.per_kg_above_5\\n\\n\\nZONES = {\\n    \\"local\\": ZoneRates(first_kg=30, per_kg_to_5=10, per_kg_above_5=8, free_from=499),\\n    \\"state\\": ZoneRates(first_kg=50, per_kg_to_5=15, per_kg_above_5=12, free_from=999),\\n    \\"national\\": ZoneRates(first_kg=80, per_kg_to_5=25, per_kg_above_5=20, free_from=1499),\\n    \\"northeast\\": ZoneRates(first_kg=120, per_kg_to_5=35, per_kg_above_5=30, free_from=1999),\\n}\\n\\n\\ndef cod_charge(subtotal):\\n    \\"\\"\\"Cash-on-delivery charge: 2% of the subtotal, rounded up, at least 25.\\"\\"\\"\\n    return max(COD_MINIMUM, math.ceil(subtotal * COD_PERCENT / 100))\\n\\n\\ndef shipping_fee(order):\\n    \\"\\"\\"Shipping fee in whole rupees.\\n\\n    order is a dict: zone (a key of ZONES), weight_kg, subtotal (rupees) and\\n    optionally cod (cash on delivery, default False).\\n    \\"\\"\\"\\n    zone = order[\\"zone\\"].lower()\\n    rates = ZONES.get(zone)\\n    if rates is None:\\n        raise ValueError(\\"unknown zone \\" + zone)\\n    weight = order[\\"weight_kg\\"]\\n    if weight <= 0:\\n        raise ValueError(\\"weight must be positive\\")\\n    if weight > MAX_WEIGHT_KG:\\n        raise ValueError(\\"too heavy for courier, use freight\\")\\n    subtotal = order[\\"subtotal\\"]\\n    fee = 0 if subtotal >= rates.free_from else rates.fee(weight)\\n    if order.get(\\"cod\\"):\\n        fee += cod_charge(subtotal)\\n    return fee\\n"}, "starter": {"shipping.py": "import math\\n\\n\\ndef shipping_fee(order):\\n    \\"\\"\\"Shipping fee in whole rupees.\\n\\n    order is a dict: zone (\\"local\\", \\"state\\" or \\"national\\"), weight_kg,\\n    subtotal (rupees) and optionally cod (cash on delivery, default False).\\n    \\"\\"\\"\\n    z = order[\\"zone\\"].lower()\\n    w = order[\\"weight_kg\\"]\\n    if z == \\"local\\":\\n        if w <= 0:\\n            raise ValueError(\\"weight must be positive\\")\\n        if w > 30:\\n            raise ValueError(\\"too heavy for courier, use freight\\")\\n        if order[\\"subtotal\\"] >= 499:\\n            fee = 0\\n        else:\\n            if w <= 1:\\n                fee = 30\\n            elif w <= 5:\\n                fee = 30 + math.ceil(w - 1) * 10\\n            else:\\n                fee = 30 + 4 * 10 + math.ceil(w - 5) * 8\\n        if order.get(\\"cod\\"):\\n            c = math.ceil(order[\\"subtotal\\"] * 2 / 100)\\n            if c < 25:\\n                c = 25\\n            fee = fee + c\\n        return fee\\n    elif z == \\"state\\":\\n        if w <= 0:\\n            raise ValueError(\\"weight must be positive\\")\\n        if w > 30:\\n            raise ValueError(\\"too heavy for courier, use freight\\")\\n        if order[\\"subtotal\\"] >= 999:\\n            fee = 0\\n        else:\\n            if w <= 1:\\n                fee = 50\\n            elif w <= 5:\\n                fee = 50 + math.ceil(w - 1) * 15\\n            else:\\n                fee = 50 + 4 * 15 + math.ceil(w - 5) * 12\\n        if order.get(\\"cod\\"):\\n            c = math.ceil(order[\\"subtotal\\"] * 2 / 100)\\n            if c < 25:\\n                c = 25\\n            fee = fee + c\\n        return fee\\n    elif z == \\"national\\":\\n        if w <= 0:\\n            raise ValueError(\\"weight must be positive\\")\\n        if w > 30:\\n            raise ValueError(\\"too heavy for courier, use freight\\")\\n        if order[\\"subtotal\\"] >= 1499:\\n            fee = 0\\n        else:\\n            if w <= 1:\\n                fee = 80\\n            elif w <= 5:\\n                fee = 80 + math.ceil(w - 1) * 25\\n            else:\\n                fee = 80 + 4 * 25 + math.ceil(w - 5) * 20\\n        if order.get(\\"cod\\"):\\n            c = math.ceil(order[\\"subtotal\\"] * 2 / 100)\\n            if c < 25:\\n                c = 25\\n            fee = fee + c\\n        return fee\\n    else:\\n        raise ValueError(\\"unknown zone \\" + z)\\n"}, "task_md": "`shipping.py` works, but every time the courier changes a rate someone has to dig through that big if/else in `shipping_fee(order)`, and we keep making mistakes. Please clean it up so that adding or changing a zone is easy, without changing any fee it calculates today (same function, same inputs, same errors).\\nWhile you\'re in there, add our new zone `\\"northeast\\"`: ₹120 for the first kg, ₹35 for each further kg up to 5 kg, then ₹30 per kg above 5 kg, and free shipping from a subtotal of ₹1,999. Otherwise it works like the other zones (COD charge, weight limits and so on)."}'),
    json.loads('{"hidden": "import copy\\nimport csv\\nimport io\\nimport random\\nfrom decimal import Decimal as D\\n\\nimport pytest\\n\\nfrom sales_report import build_report\\n\\nSALES = [\\n    {\\"order_id\\": \\"A1\\", \\"date\\": \\"2026-10-02\\", \\"sku\\": \\"ATTA-5KG\\", \\"qty\\": 2, \\"amount\\": D(\\"490.00\\")},\\n    {\\"order_id\\": \\"A1\\", \\"date\\": \\"2026-10-02\\", \\"sku\\": \\"SALT-1KG\\", \\"qty\\": 1, \\"amount\\": D(\\"28.00\\")},\\n    {\\"order_id\\": \\"A2\\", \\"date\\": \\"2026-10-01\\", \\"sku\\": \\"OIL-1L\\", \\"qty\\": 3, \\"amount\\": D(\\"540.50\\")},\\n    {\\"order_id\\": \\"A3\\", \\"date\\": \\"2026-10-02\\", \\"sku\\": \\"SALT-1KG\\", \\"qty\\": 4, \\"amount\\": D(\\"112\\")},\\n    {\\"order_id\\": \\"A4\\", \\"date\\": \\"2026-10-03\\", \\"sku\\": \\"DAL-1KG\\", \\"qty\\": 2, \\"amount\\": D(\\"240.25\\")},\\n    {\\"order_id\\": \\"A4\\", \\"date\\": \\"2026-10-04\\", \\"sku\\": \\"ATTA-5KG\\", \\"qty\\": 1, \\"amount\\": D(\\"245.00\\")},\\n    {\\"order_id\\": \\"A5\\", \\"date\\": \\"2026-10-03\\", \\"sku\\": \\"OIL-1L\\", \\"qty\\": 1, \\"amount\\": D(\\"180.5\\")},\\n]\\n\\nTEXT = (\\n    \\"Sales report\\\\n\\"\\n    \\"============\\\\n\\"\\n    \\"2026-10-01  orders: 1  items: 3  amount: 540.50\\\\n\\"\\n    \\"2026-10-02  orders: 2  items: 7  amount: 630.00\\\\n\\"\\n    \\"2026-10-03  orders: 2  items: 3  amount: 420.75\\\\n\\"\\n    \\"2026-10-04  orders: 1  items: 1  amount: 245.00\\\\n\\"\\n    \\"Total  orders: 5  items: 14  amount: 1836.25\\\\n\\"\\n    \\"\\\\n\\"\\n    \\"Top products:\\\\n\\"\\n    \\"  1. SALT-1KG  5 items  140.00\\\\n\\"\\n    \\"  2. OIL-1L  4 items  721.00\\\\n\\"\\n    \\"  3. ATTA-5KG  3 items  735.00\\\\n\\"\\n)\\n\\nTIES = [\\n    {\\"order_id\\": \\"B1\\", \\"date\\": \\"2026-11-01\\", \\"sku\\": \\"TEA-250G\\", \\"qty\\": 2, \\"amount\\": D(\\"150\\")},\\n    {\\"order_id\\": \\"B2\\", \\"date\\": \\"2026-11-01\\", \\"sku\\": \\"COFFEE-100G\\", \\"qty\\": 2, \\"amount\\": D(\\"199.99\\")},\\n    {\\"order_id\\": \\"B3\\", \\"date\\": \\"2026-11-01\\", \\"sku\\": \\"SUGAR-1KG\\", \\"qty\\": 2, \\"amount\\": D(\\"45\\")},\\n    {\\"order_id\\": \\"B3\\", \\"date\\": \\"2026-11-01\\", \\"sku\\": \\"BISCUIT\\", \\"qty\\": 2, \\"amount\\": D(\\"20\\")},\\n    {\\"order_id\\": \\"B4\\", \\"date\\": \\"2026-11-01\\", \\"sku\\": \\"ATTA-5KG\\", \\"qty\\": 1, \\"amount\\": D(\\"245\\")},\\n]\\n\\nTIES_TEXT = (\\n    \\"Sales report\\\\n\\"\\n    \\"============\\\\n\\"\\n    \\"2026-11-01  orders: 4  items: 9  amount: 659.99\\\\n\\"\\n    \\"Total  orders: 4  items: 9  amount: 659.99\\\\n\\"\\n    \\"\\\\n\\"\\n    \\"Top products:\\\\n\\"\\n    \\"  1. BISCUIT  2 items  20.00\\\\n\\"\\n    \\"  2. COFFEE-100G  2 items  199.99\\\\n\\"\\n    \\"  3. SUGAR-1KG  2 items  45.00\\\\n\\"\\n)\\n\\n\\ndef parse_csv(text):\\n    return list(csv.reader(io.StringIO(text)))\\n\\n\\ndef test_text_report_unchanged():\\n    assert build_report(SALES) == TEXT\\n    assert build_report(SALES, fmt=\\"text\\") == TEXT\\n    assert build_report(SALES, \\"text\\") == TEXT\\n\\n\\ndef test_text_report_ties_unchanged():\\n    assert build_report(TIES) == TIES_TEXT\\n\\n\\ndef test_text_report_empty_unchanged():\\n    assert build_report([]) == \\"Sales report\\\\n============\\\\nNo sales.\\\\n\\"\\n\\n\\ndef test_text_report_single_sale():\\n    one = [{\\"order_id\\": \\"Z\\", \\"date\\": \\"2026-10-09\\", \\"sku\\": \\"PEN\\", \\"qty\\": 1, \\"amount\\": D(\\"10\\")}]\\n    assert build_report(one) == (\\n        \\"Sales report\\\\n============\\\\n\\"\\n        \\"2026-10-09  orders: 1  items: 1  amount: 10.00\\\\n\\"\\n        \\"Total  orders: 1  items: 1  amount: 10.00\\\\n\\"\\n        \\"\\\\nTop products:\\\\n  1. PEN  1 items  10.00\\\\n\\"\\n    )\\n\\n\\ndef test_order_of_input_does_not_matter():\\n    shuffled = list(SALES)\\n    random.Random(7).shuffle(shuffled)\\n    assert build_report(shuffled) == TEXT\\n    assert build_report(shuffled, fmt=\\"csv\\") == build_report(SALES, fmt=\\"csv\\")\\n\\n\\ndef test_csv_report():\\n    rows = parse_csv(build_report(SALES, fmt=\\"csv\\"))\\n    assert rows == [\\n        [\\"date\\", \\"orders\\", \\"items\\", \\"amount\\"],\\n        [\\"2026-10-01\\", \\"1\\", \\"3\\", \\"540.50\\"],\\n        [\\"2026-10-02\\", \\"2\\", \\"7\\", \\"630.00\\"],\\n        [\\"2026-10-03\\", \\"2\\", \\"3\\", \\"420.75\\"],\\n        [\\"2026-10-04\\", \\"1\\", \\"1\\", \\"245.00\\"],\\n    ]\\n\\n\\ndef test_csv_report_is_a_string():\\n    out = build_report(SALES, fmt=\\"csv\\")\\n    assert isinstance(out, str)\\n    assert out.splitlines()[0] == \\"date,orders,items,amount\\"\\n\\n\\ndef test_csv_report_empty():\\n    rows = parse_csv(build_report([], fmt=\\"csv\\"))\\n    assert rows in ([], [[\\"date\\", \\"orders\\", \\"items\\", \\"amount\\"]])\\n\\n\\n@pytest.mark.parametrize(\\"fmt\\", [\\"pdf\\", \\"CSV \\", \\"\\", None, \\"json\\"])\\ndef test_unknown_format_raises(fmt):\\n    with pytest.raises(ValueError):\\n        build_report(SALES, fmt=fmt)\\n\\n\\ndef test_input_not_modified_and_repeatable():\\n    sales = copy.deepcopy(SALES)\\n    first = build_report(sales)\\n    second = build_report(sales)\\n    build_report(sales, fmt=\\"csv\\")\\n    assert first == second == TEXT\\n    assert sales == SALES\\n", "id": "15-refactor-sales-report", "meta": {"difficulty": 3, "files": ["sales_report.py"], "kind": "refactor", "timeout": 60}, "reference": {"sales_report.py": "\\"\\"\\"Daily sales report in several output formats.\\n\\nTo add a format, write a function that takes a Summary and returns text,\\nand register it in FORMATTERS.\\n\\"\\"\\"\\n\\nimport csv\\nimport io\\nfrom collections import defaultdict\\nfrom dataclasses import dataclass, field\\nfrom decimal import Decimal\\n\\nPAISA = Decimal(\\"0.01\\")\\nTOP_PRODUCTS = 3\\n\\n\\ndef _money(amount):\\n    return str(amount.quantize(PAISA))\\n\\n\\n@dataclass\\nclass Totals:\\n    orders: set = field(default_factory=set)\\n    items: int = 0\\n    amount: Decimal = Decimal(0)\\n\\n    def add(self, sale):\\n        self.orders.add(sale[\\"order_id\\"])\\n        self.items += sale[\\"qty\\"]\\n        self.amount += sale[\\"amount\\"]\\n\\n\\n@dataclass\\nclass Summary:\\n    days: list          # [(date, Totals)] sorted by date\\n    total: Totals\\n    top_products: list  # [(sku, items, amount)] most items first, then by sku\\n\\n\\ndef summarise(sales):\\n    by_day = defaultdict(Totals)\\n    by_sku = defaultdict(Totals)\\n    total = Totals()\\n    for sale in sales:\\n        by_day[sale[\\"date\\"]].add(sale)\\n        by_sku[sale[\\"sku\\"]].add(sale)\\n        total.add(sale)\\n    products = sorted(\\n        ((sku, t.items, t.amount) for sku, t in by_sku.items()),\\n        key=lambda p: (-p[1], p[0]),\\n    )\\n    return Summary(days=sorted(by_day.items()), total=total, top_products=products[:TOP_PRODUCTS])\\n\\n\\ndef _totals_text(t):\\n    return f\\"orders: {len(t.orders)}  items: {t.items}  amount: {_money(t.amount)}\\"\\n\\n\\ndef format_text(summary):\\n    lines = [\\"Sales report\\", \\"============\\"]\\n    if not summary.days:\\n        lines.append(\\"No sales.\\")\\n        return \\"\\\\n\\".join(lines) + \\"\\\\n\\"\\n    lines.extend(f\\"{day}  {_totals_text(t)}\\" for day, t in summary.days)\\n    lines.append(f\\"Total  {_totals_text(summary.total)}\\")\\n    lines.append(\\"\\")\\n    lines.append(\\"Top products:\\")\\n    lines.extend(\\n        f\\"  {rank}. {sku}  {items} items  {_money(amount)}\\"\\n        for rank, (sku, items, amount) in enumerate(summary.top_products, start=1)\\n    )\\n    return \\"\\\\n\\".join(lines) + \\"\\\\n\\"\\n\\n\\ndef format_csv(summary):\\n    out = io.StringIO()\\n    writer = csv.writer(out, lineterminator=\\"\\\\n\\")\\n    writer.writerow([\\"date\\", \\"orders\\", \\"items\\", \\"amount\\"])\\n    for day, t in summary.days:\\n        writer.writerow([day, len(t.orders), t.items, _money(t.amount)])\\n    return out.getvalue()\\n\\n\\nFORMATTERS = {\\"text\\": format_text, \\"csv\\": format_csv}\\n\\n\\ndef build_report(sales, fmt=\\"text\\"):\\n    \\"\\"\\"Sales report for sales (dicts with order_id, date, sku, qty, amount) in the given format.\\"\\"\\"\\n    try:\\n        formatter = FORMATTERS[fmt]\\n    except (KeyError, TypeError):\\n        raise ValueError(f\\"unknown report format {fmt!r}; choose from {\', \'.join(FORMATTERS)}\\") from None\\n    return formatter(summarise(sales))\\n"}, "starter": {"sales_report.py": "from decimal import Decimal\\n\\n\\ndef build_report(sales):\\n    # sales = list of dicts: order_id, date (\\"YYYY-MM-DD\\"), sku, qty (int), amount (Decimal)\\n    s = \\"Sales report\\\\n\\"\\n    s = s + \\"============\\\\n\\"\\n    if len(sales) == 0:\\n        s = s + \\"No sales.\\\\n\\"\\n        return s\\n    days = []\\n    for x in sales:\\n        if x[\\"date\\"] not in days:\\n            days.append(x[\\"date\\"])\\n    days.sort()\\n    tot_orders = set()\\n    tot_items = 0\\n    tot_amount = Decimal(\\"0\\")\\n    for d in days:\\n        o = set()\\n        i = 0\\n        a = Decimal(\\"0\\")\\n        for x in sales:\\n            if x[\\"date\\"] == d:\\n                o.add(x[\\"order_id\\"])\\n                i = i + x[\\"qty\\"]\\n                a = a + x[\\"amount\\"]\\n        s = s + d + \\"  orders: \\" + str(len(o)) + \\"  items: \\" + str(i) + \\"  amount: \\" + str(a.quantize(Decimal(\\"0.01\\"))) + \\"\\\\n\\"\\n        for oo in o:\\n            tot_orders.add(oo)\\n        tot_items = tot_items + i\\n        tot_amount = tot_amount + a\\n    s = s + \\"Total  orders: \\" + str(len(tot_orders)) + \\"  items: \\" + str(tot_items) + \\"  amount: \\" + str(tot_amount.quantize(Decimal(\\"0.01\\"))) + \\"\\\\n\\"\\n    prods = {}\\n    for x in sales:\\n        if x[\\"sku\\"] in prods:\\n            prods[x[\\"sku\\"]][0] = prods[x[\\"sku\\"]][0] + x[\\"qty\\"]\\n            prods[x[\\"sku\\"]][1] = prods[x[\\"sku\\"]][1] + x[\\"amount\\"]\\n        else:\\n            prods[x[\\"sku\\"]] = [x[\\"qty\\"], x[\\"amount\\"]]\\n    l = []\\n    for k in prods:\\n        l.append((k, prods[k][0], prods[k][1]))\\n    l.sort(key=lambda t: (-t[1], t[0]))\\n    s = s + \\"\\\\nTop products:\\\\n\\"\\n    n = 1\\n    for t in l[:3]:\\n        s = s + \\"  \\" + str(n) + \\". \\" + t[0] + \\"  \\" + str(t[1]) + \\" items  \\" + str(t[2].quantize(Decimal(\\"0.01\\"))) + \\"\\\\n\\"\\n        n = n + 1\\n    return s\\n"}, "task_md": "`sales_report.py` builds the daily sales report the shop owner reads every evening. It works, but the code is hard to follow and nobody wants to touch it. Please clean it up and make it easy to add new output formats, without changing the text report at all.\\nThen add the first new format: our accountant wants CSV. `build_report(sales, fmt=\\"text\\")` should keep returning the text report by default, and with `fmt=\\"csv\\"` return CSV text with the header `date,orders,items,amount` and one row per day with the same numbers as the text report (nothing else). Any other `fmt` should raise `ValueError`."}'),
    json.loads('{"hidden": "import http.client\\nimport json\\nimport threading\\n\\nimport pytest\\n\\nfrom order_api import OrderAPI, make_server\\n\\n\\ndef body(obj):\\n    return json.dumps(obj).encode(\\"utf-8\\")\\n\\n\\ndef create(api, item=\\"Toor dal 1kg\\", qty=2, price_paise=14500):\\n    return api.handle(\\"POST\\", \\"/orders\\", body({\\"item\\": item, \\"qty\\": qty, \\"price_paise\\": price_paise}))\\n\\n\\ndef test_create_returns_201_with_id_and_total():\\n    api = OrderAPI()\\n    status, resp = create(api, qty=3, price_paise=4250)\\n    assert status == 201\\n    assert resp[\\"qty\\"] == 3 and resp[\\"price_paise\\"] == 4250\\n    assert resp[\\"total_paise\\"] == 12750\\n    assert isinstance(resp[\\"id\\"], int)\\n    json.dumps(resp)  # must be JSON-ready\\n\\n\\ndef test_get_one_and_list_oldest_first():\\n    api = OrderAPI()\\n    _, a = create(api, item=\\"Atta 5kg\\")\\n    _, b = create(api, item=\\"Sugar 1kg\\")\\n    status, got = api.handle(\\"GET\\", f\\"/orders/{b[\'id\']}\\")\\n    assert status == 200 and got[\\"item\\"] == \\"Sugar 1kg\\"\\n    status, listing = api.handle(\\"GET\\", \\"/orders\\")\\n    assert status == 200\\n    assert [o[\\"id\\"] for o in listing[\\"orders\\"]] == [a[\\"id\\"], b[\\"id\\"]]\\n    assert a[\\"id\\"] != b[\\"id\\"]\\n\\n\\ndef test_empty_list():\\n    status, listing = OrderAPI().handle(\\"GET\\", \\"/orders\\")\\n    assert status == 200 and listing == {\\"orders\\": []}\\n\\n\\ndef test_unknown_order_is_404():\\n    api = OrderAPI()\\n    create(api)\\n    status, resp = api.handle(\\"GET\\", \\"/orders/999\\")\\n    assert status == 404 and \\"error\\" in resp\\n\\n\\n@pytest.mark.parametrize(\\"path\\", [\\"/orders/abc\\", \\"/orders/1/extra\\", \\"/nothing\\", \\"/\\"])\\ndef test_bad_paths_are_404_not_crash(path):\\n    api = OrderAPI()\\n    create(api)\\n    status, resp = api.handle(\\"GET\\", path)\\n    assert status == 404\\n    assert \\"error\\" in resp\\n\\n\\n@pytest.mark.parametrize(\\n    \\"raw\\",\\n    [\\n        b\\"\\",\\n        b\\"{not json\\",\\n        b\\"[1, 2, 3]\\",\\n        b\'\\"just a string\\"\',\\n        b\\"\\\\xff\\\\xfe\\\\x00garbage\\",\\n    ],\\n)\\ndef test_unparseable_body_is_400(raw):\\n    status, resp = OrderAPI().handle(\\"POST\\", \\"/orders\\", raw)\\n    assert status == 400\\n    assert isinstance(resp.get(\\"error\\"), str) and resp[\\"error\\"]\\n\\n\\n@pytest.mark.parametrize(\\n    \\"payload\\",\\n    [\\n        {\\"qty\\": 1, \\"price_paise\\": 100},\\n        {\\"item\\": \\"\\", \\"qty\\": 1, \\"price_paise\\": 100},\\n        {\\"item\\": \\"   \\", \\"qty\\": 1, \\"price_paise\\": 100},\\n        {\\"item\\": 42, \\"qty\\": 1, \\"price_paise\\": 100},\\n        {\\"item\\": \\"Rice\\", \\"price_paise\\": 100},\\n        {\\"item\\": \\"Rice\\", \\"qty\\": 0, \\"price_paise\\": 100},\\n        {\\"item\\": \\"Rice\\", \\"qty\\": -3, \\"price_paise\\": 100},\\n        {\\"item\\": \\"Rice\\", \\"qty\\": 2.5, \\"price_paise\\": 100},\\n        {\\"item\\": \\"Rice\\", \\"qty\\": \\"2\\", \\"price_paise\\": 100},\\n        {\\"item\\": \\"Rice\\", \\"qty\\": True, \\"price_paise\\": 100},\\n        {\\"item\\": \\"Rice\\", \\"qty\\": None, \\"price_paise\\": 100},\\n        {\\"item\\": \\"Rice\\", \\"qty\\": 1},\\n        {\\"item\\": \\"Rice\\", \\"qty\\": 1, \\"price_paise\\": -1},\\n        {\\"item\\": \\"Rice\\", \\"qty\\": 1, \\"price_paise\\": 99.5},\\n    ],\\n)\\ndef test_invalid_order_is_400(payload):\\n    api = OrderAPI()\\n    status, resp = api.handle(\\"POST\\", \\"/orders\\", body(payload))\\n    assert status == 400\\n    assert isinstance(resp.get(\\"error\\"), str) and resp[\\"error\\"]\\n    # nothing was stored\\n    assert api.handle(\\"GET\\", \\"/orders\\")[1] == {\\"orders\\": []}\\n\\n\\ndef test_free_item_price_zero_is_allowed():\\n    status, resp = create(OrderAPI(), item=\\"Free carry bag\\", qty=1, price_paise=0)\\n    assert status == 201 and resp[\\"total_paise\\"] == 0\\n\\n\\n@pytest.mark.parametrize(\\"method,path\\", [(\\"DELETE\\", \\"/orders\\"), (\\"PUT\\", \\"/orders\\"), (\\"POST\\", \\"/orders/1\\")])\\ndef test_wrong_method_is_405(method, path):\\n    api = OrderAPI()\\n    create(api)\\n    status, resp = api.handle(method, path, b\\"\\")\\n    assert status == 405\\n    assert \\"error\\" in resp\\n\\n\\ndef test_returned_order_cannot_corrupt_store():\\n    api = OrderAPI()\\n    _, order = create(api, qty=2, price_paise=100)\\n    order[\\"total_paise\\"] = 1\\n    _, listing = api.handle(\\"GET\\", \\"/orders\\")\\n    listing[\\"orders\\"].clear()\\n    status, again = api.handle(\\"GET\\", f\\"/orders/{order[\'id\']}\\")\\n    assert status == 200 and again[\\"total_paise\\"] == 200\\n\\n\\ndef test_concurrent_creates_get_unique_ids():\\n    api = OrderAPI()\\n    threads_n, per_thread = 8, 50\\n    barrier = threading.Barrier(threads_n)\\n    ids = []\\n    ids_lock = threading.Lock()\\n    errors = []\\n\\n    def worker(n):\\n        try:\\n            barrier.wait(timeout=5)\\n            for i in range(per_thread):\\n                status, resp = create(api, item=f\\"item-{n}-{i}\\", qty=1, price_paise=1)\\n                assert status == 201\\n                with ids_lock:\\n                    ids.append(resp[\\"id\\"])\\n        except Exception as exc:  # pragma: no cover - reported below\\n            errors.append(exc)\\n\\n    threads = [threading.Thread(target=worker, args=(n,)) for n in range(threads_n)]\\n    for t in threads:\\n        t.start()\\n    for t in threads:\\n        t.join(timeout=10)\\n    assert not errors\\n    assert len(ids) == threads_n * per_thread\\n    assert len(set(ids)) == len(ids)\\n    assert len(api.handle(\\"GET\\", \\"/orders\\")[1][\\"orders\\"]) == len(ids)\\n\\n\\ndef test_real_server_roundtrip():\\n    api = OrderAPI()\\n    server = make_server(api, host=\\"127.0.0.1\\", port=0)\\n    thread = threading.Thread(target=server.serve_forever, daemon=True)\\n    thread.start()\\n    try:\\n        host, port = server.server_address[:2]\\n        conn = http.client.HTTPConnection(host, port, timeout=5)\\n        payload = body({\\"item\\": \\"Haldi 200g\\", \\"qty\\": 2, \\"price_paise\\": 3000})\\n        conn.request(\\"POST\\", \\"/orders\\", body=payload, headers={\\"Content-Type\\": \\"application/json\\"})\\n        resp = conn.getresponse()\\n        data = json.loads(resp.read())\\n        assert resp.status == 201\\n        assert \\"application/json\\" in resp.getheader(\\"Content-Type\\", \\"\\")\\n        assert data[\\"total_paise\\"] == 6000\\n\\n        conn.request(\\"POST\\", \\"/orders\\", body=b\\"{oops\\", headers={\\"Content-Type\\": \\"application/json\\"})\\n        resp = conn.getresponse()\\n        assert resp.status == 400\\n        assert \\"error\\" in json.loads(resp.read())\\n\\n        conn.request(\\"GET\\", \\"/orders/12345\\")\\n        resp = conn.getresponse()\\n        assert resp.status == 404\\n        resp.read()\\n        conn.close()\\n    finally:\\n        server.shutdown()\\n        server.server_close()\\n        thread.join(timeout=5)\\n", "id": "16-order-http-api", "meta": {"difficulty": 3, "files": ["order_api.py"], "kind": "build", "timeout": 60}, "reference": {"order_api.py": "\\"\\"\\"A tiny JSON API for shop orders, standard library only.\\n\\n`OrderAPI.handle` does all routing and validation and can be tested without a socket;\\n`make_server` wraps it in a `ThreadingHTTPServer`.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport re\\nimport threading\\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\\n\\nMAX_BODY_BYTES = 64 * 1024\\nMAX_ITEM_LENGTH = 200\\n_ORDER_PATH = re.compile(r\\"/orders/([0-9]+)\\")\\n\\n\\nclass BadRequest(Exception):\\n    \\"\\"\\"The client sent something we cannot accept (HTTP 400).\\"\\"\\"\\n\\n\\ndef _require_int(data: dict, field: str, minimum: int) -> int:\\n    if field not in data:\\n        raise BadRequest(f\\"missing field: {field}\\")\\n    value = data[field]\\n    # bool is a subclass of int; true/false is never a valid quantity or price.\\n    if isinstance(value, bool) or not isinstance(value, int):\\n        raise BadRequest(f\\"{field} must be a whole number\\")\\n    if value < minimum:\\n        raise BadRequest(f\\"{field} must be at least {minimum}\\")\\n    return value\\n\\n\\ndef _parse_order(body: bytes) -> dict:\\n    if not body:\\n        raise BadRequest(\\"request body is empty\\")\\n    if len(body) > MAX_BODY_BYTES:\\n        raise BadRequest(\\"request body is too large\\")\\n    try:\\n        data = json.loads(body.decode(\\"utf-8\\"))\\n    except (UnicodeDecodeError, json.JSONDecodeError):\\n        raise BadRequest(\\"request body is not valid JSON\\") from None\\n    if not isinstance(data, dict):\\n        raise BadRequest(\\"request body must be a JSON object\\")\\n    item = data.get(\\"item\\")\\n    if not isinstance(item, str) or not item.strip():\\n        raise BadRequest(\\"item must be a non-empty string\\")\\n    if len(item) > MAX_ITEM_LENGTH:\\n        raise BadRequest(\\"item is too long\\")\\n    qty = _require_int(data, \\"qty\\", 1)\\n    price = _require_int(data, \\"price_paise\\", 0)\\n    return {\\"item\\": item.strip(), \\"qty\\": qty, \\"price_paise\\": price}\\n\\n\\nclass OrderAPI:\\n    \\"\\"\\"In-memory order store with JSON-style request handling. Safe to share between threads.\\"\\"\\"\\n\\n    def __init__(self) -> None:\\n        self._lock = threading.Lock()\\n        self._orders: dict[int, dict] = {}\\n        self._next_id = 1\\n\\n    def handle(self, method: str, path: str, body: bytes = b\\"\\") -> tuple[int, dict]:\\n        method = (method or \\"\\").upper()\\n        path = (path or \\"\\").split(\\"?\\", 1)[0]\\n        if len(path) > 1:\\n            path = path.rstrip(\\"/\\")\\n        try:\\n            if path == \\"/orders\\":\\n                return self._collection(method, body or b\\"\\")\\n            if path.startswith(\\"/orders/\\"):\\n                return self._one(method, path)\\n        except BadRequest as exc:\\n            return 400, {\\"error\\": str(exc)}\\n        return 404, {\\"error\\": \\"not found\\"}\\n\\n    def _collection(self, method: str, body: bytes) -> tuple[int, dict]:\\n        if method == \\"GET\\":\\n            return 200, {\\"orders\\": self._list()}\\n        if method == \\"POST\\":\\n            return 201, self._create(_parse_order(body))\\n        return 405, {\\"error\\": \\"method not allowed\\"}\\n\\n    def _one(self, method: str, path: str) -> tuple[int, dict]:\\n        if method != \\"GET\\":\\n            return 405, {\\"error\\": \\"method not allowed\\"}\\n        match = _ORDER_PATH.fullmatch(path)\\n        order = self._get(int(match.group(1))) if match else None\\n        if order is None:\\n            return 404, {\\"error\\": \\"order not found\\"}\\n        return 200, order\\n\\n    def _create(self, fields: dict) -> dict:\\n        with self._lock:\\n            order_id = self._next_id\\n            self._next_id += 1\\n            order = {\\"id\\": order_id, **fields, \\"total_paise\\": fields[\\"qty\\"] * fields[\\"price_paise\\"]}\\n            self._orders[order_id] = order\\n            return dict(order)\\n\\n    def _get(self, order_id: int) -> dict | None:\\n        with self._lock:\\n            order = self._orders.get(order_id)\\n            return dict(order) if order is not None else None\\n\\n    def _list(self) -> list[dict]:\\n        with self._lock:\\n            return [dict(self._orders[k]) for k in sorted(self._orders)]\\n\\n\\ndef make_server(api: OrderAPI, host: str = \\"127.0.0.1\\", port: int = 8000) -> ThreadingHTTPServer:\\n    \\"\\"\\"Return a threading HTTP server that serves `api`. The caller runs `serve_forever()`.\\"\\"\\"\\n\\n    class Handler(BaseHTTPRequestHandler):\\n        def _dispatch(self) -> None:\\n            body = b\\"\\"\\n            if self.command in (\\"POST\\", \\"PUT\\", \\"PATCH\\"):\\n                try:\\n                    length = int(self.headers.get(\\"Content-Length\\") or 0)\\n                except ValueError:\\n                    self._send(400, {\\"error\\": \\"bad Content-Length\\"})\\n                    return\\n                if length < 0:\\n                    self._send(400, {\\"error\\": \\"bad Content-Length\\"})\\n                    return\\n                if length > MAX_BODY_BYTES:\\n                    self._send(413, {\\"error\\": \\"request body is too large\\"})\\n                    return\\n                body = self.rfile.read(length)\\n            status, payload = api.handle(self.command, self.path, body)\\n            self._send(status, payload)\\n\\n        def _send(self, status: int, payload: dict) -> None:\\n            data = json.dumps(payload).encode(\\"utf-8\\")\\n            self.send_response(status)\\n            self.send_header(\\"Content-Type\\", \\"application/json; charset=utf-8\\")\\n            self.send_header(\\"Content-Length\\", str(len(data)))\\n            self.end_headers()\\n            self.wfile.write(data)\\n\\n        do_GET = do_POST = do_PUT = do_PATCH = do_DELETE = _dispatch\\n\\n        def log_message(self, *args) -> None:\\n            pass  # keep test and server output quiet\\n\\n    server = ThreadingHTTPServer((host, port), Handler)\\n    server.daemon_threads = True\\n    return server\\n"}, "starter": {}, "task_md": "Our shop\'s mobile app needs a small JSON API for orders, and we can\'t add any dependencies, so standard library only.\\nPlease write `order_api.py` with a class `OrderAPI` whose method `handle(method: str, path: str, body: bytes = b\\"\\") -> tuple[int, dict]` does the routing and returns the HTTP status code and the JSON-ready response body, so we can test it without opening a socket.\\nAlso add `make_server(api: OrderAPI, host: str = \\"127.0.0.1\\", port: int = 8000)` that returns an `http.server.ThreadingHTTPServer` serving that API (we call `serve_forever()` ourselves).\\nRoutes: `POST /orders` with `{\\"item\\": str, \\"qty\\": int, \\"price_paise\\": int}` creates an order and returns it with a new `id` and `total_paise`; `GET /orders/<id>` returns one order; `GET /orders` returns `{\\"orders\\": [...]}`, oldest first.\\nQuantity is a whole number of at least 1, and the price can\'t be negative.\\nThe app team expects the usual status codes: 201 on create, 200 for reads, 400 when the request is bad (with an `\\"error\\"` message in the body), 404 for an unknown order or path, 405 for a wrong method.\\nSeveral cashiers\' phones hit this at the same time."}'),
    json.loads('{"hidden": "import threading\\n\\nimport pytest\\n\\nfrom jobs import JobQueue\\n\\n\\nclass FakeSleep:\\n    def __init__(self):\\n        self.calls = []\\n\\n    def __call__(self, seconds):\\n        self.calls.append(seconds)\\n\\n\\nclass Flaky:\\n    \\"\\"\\"Fails `failures` times, then returns `value`.\\"\\"\\"\\n\\n    def __init__(self, failures, value=\\"sent\\"):\\n        self.failures = failures\\n        self.value = value\\n        self.calls = 0\\n\\n    def __call__(self, *args, **kwargs):\\n        self.calls += 1\\n        if self.calls <= self.failures:\\n            raise ConnectionError(f\\"gateway down (call {self.calls})\\")\\n        return self.value\\n\\n\\ndef test_success_first_try_no_sleep():\\n    sleep = FakeSleep()\\n    q = JobQueue(sleep=sleep)\\n    job_id = q.submit(lambda: \\"ok\\")\\n    assert q.get(job_id).status == \\"pending\\"\\n    q.run_pending()\\n    job = q.get(job_id)\\n    assert job.status == \\"done\\"\\n    assert job.result == \\"ok\\"\\n    assert job.error is None\\n    assert job.attempts == 1\\n    assert sleep.calls == []\\n\\n\\ndef test_args_and_kwargs_are_passed():\\n    q = JobQueue(sleep=FakeSleep())\\n    job_id = q.submit(lambda phone, text, prefix=\\"\\": f\\"{prefix}{phone}:{text}\\", \\"98xxxxxx01\\", \\"Order packed\\", prefix=\\"+91-\\")\\n    q.run_pending()\\n    assert q.get(job_id).result == \\"+91-98xxxxxx01:Order packed\\"\\n\\n\\ndef test_retries_then_succeeds_with_exponential_backoff():\\n    sleep = FakeSleep()\\n    q = JobQueue(max_attempts=3, base_delay=1.0, sleep=sleep)\\n    func = Flaky(failures=2)\\n    job_id = q.submit(func)\\n    q.run_pending()\\n    job = q.get(job_id)\\n    assert job.status == \\"done\\"\\n    assert job.result == \\"sent\\"\\n    assert job.attempts == 3\\n    assert func.calls == 3\\n    assert sleep.calls == [1.0, 2.0]\\n\\n\\ndef test_gives_up_after_max_attempts_without_sleeping_after_last_try():\\n    sleep = FakeSleep()\\n    q = JobQueue(max_attempts=4, base_delay=0.5, sleep=sleep)\\n    func = Flaky(failures=100)\\n    job_id = q.submit(func)\\n    q.run_pending()\\n    job = q.get(job_id)\\n    assert job.status == \\"failed\\"\\n    assert job.attempts == 4\\n    assert func.calls == 4\\n    assert isinstance(job.error, ConnectionError)\\n    assert sleep.calls == [0.5, 1.0, 2.0]\\n\\n\\ndef test_backoff_is_capped_at_max_delay():\\n    sleep = FakeSleep()\\n    q = JobQueue(max_attempts=6, base_delay=10, max_delay=25, sleep=sleep)\\n    q.submit(Flaky(failures=100))\\n    q.run_pending()\\n    assert sleep.calls == [10, 20, 25, 25, 25]\\n\\n\\ndef test_failed_job_does_not_stop_the_others_and_order_is_fifo():\\n    order = []\\n    q = JobQueue(max_attempts=2, sleep=FakeSleep())\\n\\n    def record(name):\\n        order.append(name)\\n        return name\\n\\n    def broken():\\n        order.append(\\"broken\\")\\n        raise RuntimeError(\\"bad template\\")\\n\\n    a = q.submit(record, \\"a\\")\\n    b = q.submit(broken)\\n    c = q.submit(record, \\"c\\")\\n    q.run_pending()\\n    assert order == [\\"a\\", \\"broken\\", \\"broken\\", \\"c\\"]\\n    assert q.get(a).status == \\"done\\"\\n    assert q.get(b).status == \\"failed\\"\\n    assert q.get(c).status == \\"done\\"\\n\\n\\ndef test_run_pending_twice_does_not_rerun_finished_jobs():\\n    q = JobQueue(max_attempts=2, sleep=FakeSleep())\\n    ok = Flaky(failures=0)\\n    bad = Flaky(failures=100)\\n    q.submit(ok)\\n    q.submit(bad)\\n    q.run_pending()\\n    q.run_pending()\\n    assert ok.calls == 1\\n    assert bad.calls == 2\\n\\n\\ndef test_jobs_submitted_later_run_on_next_call():\\n    q = JobQueue(sleep=FakeSleep())\\n    first = q.submit(lambda: 1)\\n    q.run_pending()\\n    second = q.submit(lambda: 2)\\n    assert q.get(second).status == \\"pending\\"\\n    q.run_pending()\\n    assert q.get(first).result == 1\\n    assert q.get(second).result == 2\\n\\n\\ndef test_unknown_job_id_raises_key_error():\\n    q = JobQueue(sleep=FakeSleep())\\n    with pytest.raises(KeyError):\\n        q.get(12345)\\n\\n\\ndef test_submit_rejects_non_callable():\\n    q = JobQueue(sleep=FakeSleep())\\n    with pytest.raises(TypeError):\\n        q.submit(\\"send_sms\\")\\n\\n\\n@pytest.mark.parametrize(\\"kwargs\\", [{\\"max_attempts\\": 0}, {\\"max_attempts\\": -1}, {\\"base_delay\\": -1}])\\ndef test_bad_settings_raise_value_error(kwargs):\\n    with pytest.raises(ValueError):\\n        JobQueue(sleep=FakeSleep(), **kwargs)\\n\\n\\ndef test_keyboard_interrupt_is_not_swallowed_or_retried():\\n    sleep = FakeSleep()\\n    q = JobQueue(max_attempts=5, sleep=sleep)\\n    calls = []\\n\\n    def interrupted():\\n        calls.append(1)\\n        raise KeyboardInterrupt\\n\\n    q.submit(interrupted)\\n    with pytest.raises(KeyboardInterrupt):\\n        q.run_pending()\\n    assert len(calls) == 1\\n    assert sleep.calls == []\\n\\n\\ndef test_concurrent_submits_get_unique_ids_and_all_run():\\n    q = JobQueue(sleep=FakeSleep())\\n    n_threads, per_thread = 8, 100\\n    barrier = threading.Barrier(n_threads)\\n    ids, lock = [], threading.Lock()\\n\\n    def worker():\\n        barrier.wait(timeout=5)\\n        for _ in range(per_thread):\\n            job_id = q.submit(lambda: None)\\n            with lock:\\n                ids.append(job_id)\\n\\n    threads = [threading.Thread(target=worker) for _ in range(n_threads)]\\n    for t in threads:\\n        t.start()\\n    for t in threads:\\n        t.join(timeout=10)\\n    assert len(ids) == n_threads * per_thread\\n    assert len(set(ids)) == len(ids)\\n    q.run_pending()\\n    assert all(q.get(i).status == \\"done\\" for i in ids)\\n\\n\\ndef test_submit_is_not_blocked_while_a_job_is_running():\\n    q = JobQueue(sleep=FakeSleep())\\n    started, proceed = threading.Event(), threading.Event()\\n\\n    def slow_job():\\n        started.set()\\n        assert proceed.wait(timeout=5)\\n        return \\"slow done\\"\\n\\n    slow_id = q.submit(slow_job)\\n    runner = threading.Thread(target=q.run_pending)\\n    runner.start()\\n    try:\\n        assert started.wait(timeout=5)\\n        submitted = []\\n        submitter = threading.Thread(target=lambda: submitted.append(q.submit(lambda: \\"quick\\")))\\n        submitter.start()\\n        submitter.join(timeout=2)\\n        assert submitted, \\"submit() blocked while run_pending() was running a job\\"\\n    finally:\\n        proceed.set()\\n        runner.join(timeout=5)\\n    assert not runner.is_alive()\\n    q.run_pending()\\n    assert q.get(slow_id).result == \\"slow done\\"\\n    assert q.get(submitted[0]).result == \\"quick\\"\\n", "id": "17-message-job-queue", "meta": {"difficulty": 2, "files": ["jobs.py"], "kind": "build", "timeout": 60}, "reference": {"jobs.py": "\\"\\"\\"A small in-process job queue with retries and exponential backoff.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport threading\\nimport time\\nfrom collections import deque\\nfrom collections.abc import Callable\\nfrom dataclasses import dataclass, field\\nfrom typing import Any\\n\\nPENDING = \\"pending\\"\\nDONE = \\"done\\"\\nFAILED = \\"failed\\"\\n\\n\\n@dataclass\\nclass Job:\\n    id: int\\n    func: Callable[..., Any] = field(repr=False)\\n    args: tuple = field(default=(), repr=False)\\n    kwargs: dict = field(default_factory=dict, repr=False)\\n    status: str = PENDING\\n    result: Any = None\\n    error: BaseException | None = None\\n    attempts: int = 0\\n\\n\\nclass JobQueue:\\n    def __init__(\\n        self,\\n        max_attempts: int = 3,\\n        base_delay: float = 1.0,\\n        max_delay: float = 30.0,\\n        sleep: Callable[[float], None] = time.sleep,\\n    ) -> None:\\n        if isinstance(max_attempts, bool) or not isinstance(max_attempts, int) or max_attempts < 1:\\n            raise ValueError(\\"max_attempts must be a whole number >= 1\\")\\n        if base_delay < 0 or max_delay < 0:\\n            raise ValueError(\\"delays must not be negative\\")\\n        if not callable(sleep):\\n            raise TypeError(\\"sleep must be callable\\")\\n        self.max_attempts = max_attempts\\n        self.base_delay = base_delay\\n        self.max_delay = max_delay\\n        self._sleep = sleep\\n        self._lock = threading.Lock()\\n        self._run_lock = threading.Lock()\\n        self._jobs: dict[int, Job] = {}\\n        self._waiting: deque[Job] = deque()\\n        self._next_id = 1\\n\\n    def submit(self, func: Callable[..., Any], *args: Any, **kwargs: Any) -> int:\\n        if not callable(func):\\n            raise TypeError(\\"func must be callable\\")\\n        with self._lock:\\n            job = Job(self._next_id, func, args, kwargs)\\n            self._next_id += 1\\n            self._jobs[job.id] = job\\n            self._waiting.append(job)\\n            return job.id\\n\\n    def get(self, job_id: int) -> Job:\\n        with self._lock:\\n            try:\\n                return self._jobs[job_id]\\n            except KeyError:\\n                raise KeyError(f\\"unknown job id: {job_id!r}\\") from None\\n\\n    def _delay(self, failed_attempts: int) -> float:\\n        return min(self.base_delay * (2 ** (failed_attempts - 1)), self.max_delay)\\n\\n    def run_pending(self) -> int:\\n        \\"\\"\\"Run every waiting job (FIFO). Returns how many jobs were processed.\\"\\"\\"\\n        processed = 0\\n        with self._run_lock:  # one runner at a time; submit() stays free\\n            while True:\\n                with self._lock:\\n                    if not self._waiting:\\n                        return processed\\n                    job = self._waiting.popleft()\\n                self._run_one(job)\\n                processed += 1\\n\\n    def _run_one(self, job: Job) -> None:\\n        while True:\\n            job.attempts += 1\\n            try:\\n                result = job.func(*job.args, **job.kwargs)\\n            except Exception as exc:  # noqa: BLE001 - any job error is retried; Ctrl+C etc. propagate\\n                job.error = exc\\n                if job.attempts >= self.max_attempts:\\n                    job.status = FAILED\\n                    return\\n                self._sleep(self._delay(job.attempts))\\n            else:\\n                job.result = result\\n                job.error = None\\n                job.status = DONE\\n                return\\n"}, "starter": {}, "task_md": "We send order confirmations over an SMS gateway that fails now and then, so I want a small in-process job queue in `jobs.py`.\\n`JobQueue(max_attempts=3, base_delay=1.0, max_delay=30.0, sleep=time.sleep)`: `submit(func, *args, **kwargs) -> int` adds a job and returns its id, and `run_pending()` runs every waiting job in the order they were submitted.\\n`get(job_id)` returns the job; it should have `.status` (`\\"pending\\"`, `\\"done\\"` or `\\"failed\\"`), `.result`, `.error` (the last exception) and `.attempts`.\\nWhen a job raises, try it again after waiting `base_delay` seconds, doubling the wait each time but never more than `max_delay`, until it has been tried `max_attempts` times; then mark it failed and move on to the next job.\\nThe `sleep` function is passed in so our tests don\'t really wait.\\nJobs get submitted from the web request threads while a worker calls `run_pending()`."}'),
    json.loads('{"hidden": "import json\\n\\nimport pytest\\n\\nfrom shopconfig import ConfigError, load_config\\n\\nSECRET = \\"sk_live_9f8e7d6c5b4a\\"\\n\\n\\ndef write_json(tmp_path, data, name=\\"shop.json\\"):\\n    p = tmp_path / name\\n    p.write_text(json.dumps(data, ensure_ascii=False), encoding=\\"utf-8\\")\\n    return p\\n\\n\\ndef test_defaults_with_only_the_name():\\n    cfg = load_config(env={\\"SHOP_NAME\\": \\"Sharma Kirana\\"})\\n    assert cfg.name == \\"Sharma Kirana\\"\\n    assert cfg.port == 8080\\n    assert cfg.debug is False\\n    assert cfg.low_stock_threshold == 5\\n    assert cfg.sms_api_key is None\\n\\n\\ndef test_file_overrides_defaults(tmp_path):\\n    p = write_json(tmp_path, {\\"name\\": \\"शर्मा किराना\\", \\"port\\": 9000, \\"debug\\": True, \\"low_stock_threshold\\": 2})\\n    cfg = load_config(p, env={})\\n    assert cfg.name == \\"शर्मा किराना\\"\\n    assert (cfg.port, cfg.debug, cfg.low_stock_threshold) == (9000, True, 2)\\n\\n\\ndef test_env_overrides_file(tmp_path):\\n    p = write_json(tmp_path, {\\"name\\": \\"File Name\\", \\"port\\": 9000, \\"debug\\": True})\\n    cfg = load_config(str(p), env={\\"SHOP_PORT\\": \\"9100\\", \\"SHOP_DEBUG\\": \\"false\\", \\"SHOP_SMS_API_KEY\\": SECRET})\\n    assert cfg.name == \\"File Name\\"\\n    assert cfg.port == 9100\\n    assert cfg.debug is False\\n    assert cfg.sms_api_key == SECRET\\n\\n\\n@pytest.mark.parametrize(\\"text,expected\\", [(\\"true\\", True), (\\"TRUE\\", True), (\\"1\\", True), (\\"false\\", False),\\n                                           (\\"False\\", False), (\\"0\\", False)])\\ndef test_env_booleans(text, expected):\\n    cfg = load_config(env={\\"SHOP_NAME\\": \\"x\\", \\"SHOP_DEBUG\\": text})\\n    assert cfg.debug is expected\\n\\n\\ndef test_uses_os_environ_by_default(monkeypatch):\\n    monkeypatch.setenv(\\"SHOP_NAME\\", \\"Env Shop\\")\\n    monkeypatch.setenv(\\"SHOP_LOW_STOCK_THRESHOLD\\", \\"12\\")\\n    cfg = load_config()\\n    assert cfg.name == \\"Env Shop\\"\\n    assert cfg.low_stock_threshold == 12\\n\\n\\ndef test_does_not_modify_given_env():\\n    env = {\\"SHOP_NAME\\": \\"x\\", \\"SHOP_PORT\\": \\"9001\\"}\\n    before = dict(env)\\n    load_config(env=env)\\n    assert env == before\\n\\n\\n@pytest.mark.parametrize(\\n    \\"env\\",\\n    [\\n        {},\\n        {\\"SHOP_NAME\\": \\"\\"},\\n        {\\"SHOP_NAME\\": \\"   \\"},\\n        {\\"SHOP_NAME\\": \\"x\\", \\"SHOP_PORT\\": \\"abc\\"},\\n        {\\"SHOP_NAME\\": \\"x\\", \\"SHOP_PORT\\": \\"0\\"},\\n        {\\"SHOP_NAME\\": \\"x\\", \\"SHOP_PORT\\": \\"70000\\"},\\n        {\\"SHOP_NAME\\": \\"x\\", \\"SHOP_PORT\\": \\"-5\\"},\\n        {\\"SHOP_NAME\\": \\"x\\", \\"SHOP_PORT\\": \\"80.5\\"},\\n        {\\"SHOP_NAME\\": \\"x\\", \\"SHOP_DEBUG\\": \\"maybe\\"},\\n        {\\"SHOP_NAME\\": \\"x\\", \\"SHOP_LOW_STOCK_THRESHOLD\\": \\"-1\\"},\\n        {\\"SHOP_NAME\\": \\"x\\", \\"SHOP_LOW_STOCK_THRESHOLD\\": \\"few\\"},\\n    ],\\n)\\ndef test_bad_env_values_raise_config_error(env):\\n    with pytest.raises(ConfigError):\\n        load_config(env=env)\\n\\n\\n@pytest.mark.parametrize(\\n    \\"data\\",\\n    [\\n        {\\"port\\": 9000},\\n        {\\"name\\": \\"x\\", \\"port\\": True},\\n        {\\"name\\": \\"x\\", \\"port\\": 0},\\n        {\\"name\\": \\"x\\", \\"port\\": \\"lots\\"},\\n        {\\"name\\": \\"x\\", \\"debug\\": \\"sometimes\\"},\\n        {\\"name\\": \\"x\\", \\"low_stock_threshold\\": 2.5},\\n        {\\"name\\": 123},\\n    ],\\n)\\ndef test_bad_file_values_raise_config_error(tmp_path, data):\\n    with pytest.raises(ConfigError):\\n        load_config(write_json(tmp_path, data), env={})\\n\\n\\ndef test_missing_file_raises_config_error(tmp_path):\\n    with pytest.raises(ConfigError):\\n        load_config(tmp_path / \\"nope.json\\", env={\\"SHOP_NAME\\": \\"x\\"})\\n\\n\\n@pytest.mark.parametrize(\\"content\\", [\\"{not json\\", \\"\\", \\"[1, 2]\\", \'\\"shop\\"\'])\\ndef test_broken_file_raises_config_error(tmp_path, content):\\n    p = tmp_path / \\"shop.json\\"\\n    p.write_text(content, encoding=\\"utf-8\\")\\n    with pytest.raises(ConfigError):\\n        load_config(p, env={\\"SHOP_NAME\\": \\"x\\"})\\n\\n\\ndef test_config_error_is_an_exception():\\n    assert issubclass(ConfigError, Exception)\\n\\n\\ndef test_config_is_read_only():\\n    cfg = load_config(env={\\"SHOP_NAME\\": \\"x\\"})\\n    with pytest.raises((AttributeError, TypeError)):\\n        cfg.port = 1\\n    assert cfg.port == 8080\\n\\n\\ndef test_secret_is_not_shown_when_printed(tmp_path):\\n    p = write_json(tmp_path, {\\"name\\": \\"Sharma Kirana\\", \\"sms_api_key\\": SECRET})\\n    cfg = load_config(p, env={})\\n    assert cfg.sms_api_key == SECRET\\n    assert SECRET not in repr(cfg)\\n    assert SECRET not in str(cfg)\\n    assert SECRET not in f\\"{cfg}\\"\\n    assert \\"Sharma Kirana\\" in repr(cfg)  # the rest is still useful in the log\\n\\n", "id": "18-shop-config-loader", "meta": {"difficulty": 2, "files": ["shopconfig.py"], "kind": "build", "timeout": 60}, "reference": {"shopconfig.py": "\\"\\"\\"Load the shop server\'s settings: defaults < JSON file < SHOP_* environment variables.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nfrom collections.abc import Mapping\\nfrom dataclasses import dataclass, field\\nfrom typing import Any\\n\\nENV_PREFIX = \\"SHOP_\\"\\n_TRUE = {\\"1\\", \\"true\\", \\"yes\\", \\"on\\"}\\n_FALSE = {\\"0\\", \\"false\\", \\"no\\", \\"off\\"}\\n\\n\\nclass ConfigError(Exception):\\n    \\"\\"\\"The configuration is missing something or has an invalid value.\\"\\"\\"\\n\\n\\n@dataclass(frozen=True)\\nclass Config:\\n    name: str\\n    port: int = 8080\\n    debug: bool = False\\n    low_stock_threshold: int = 5\\n    sms_api_key: str | None = field(default=None, repr=False)\\n\\n    def __repr__(self) -> str:\\n        key = \\"None\\" if self.sms_api_key is None else \\"\'***\'\\"\\n        return (\\n            f\\"Config(name={self.name!r}, port={self.port!r}, debug={self.debug!r}, \\"\\n            f\\"low_stock_threshold={self.low_stock_threshold!r}, sms_api_key={key})\\"\\n        )\\n\\n    __str__ = __repr__\\n\\n\\n_DEFAULTS: dict[str, Any] = {\\n    \\"port\\": 8080,\\n    \\"debug\\": False,\\n    \\"low_stock_threshold\\": 5,\\n    \\"sms_api_key\\": None,\\n}\\n_FIELDS = (\\"name\\", \\"port\\", \\"debug\\", \\"low_stock_threshold\\", \\"sms_api_key\\")\\n\\n\\ndef _read_file(path: str | os.PathLike) -> dict[str, Any]:\\n    try:\\n        with open(path, encoding=\\"utf-8\\") as fh:\\n            data = json.load(fh)\\n    except FileNotFoundError:\\n        raise ConfigError(f\\"config file not found: {os.fspath(path)}\\") from None\\n    except (OSError, UnicodeDecodeError) as exc:\\n        raise ConfigError(f\\"cannot read config file {os.fspath(path)}: {exc}\\") from None\\n    except json.JSONDecodeError as exc:\\n        raise ConfigError(f\\"config file {os.fspath(path)} is not valid JSON: {exc}\\") from None\\n    if not isinstance(data, dict):\\n        raise ConfigError(f\\"config file {os.fspath(path)} must contain a JSON object\\")\\n    unknown = sorted(set(data) - set(_FIELDS))\\n    if unknown:\\n        raise ConfigError(f\\"unknown setting(s) in {os.fspath(path)}: {\', \'.join(unknown)}\\")\\n    return data\\n\\n\\ndef _env_name(key: str) -> str:\\n    return ENV_PREFIX + key.upper()\\n\\n\\ndef _as_int(key: str, value: Any, source: str) -> int:\\n    if isinstance(value, bool):\\n        raise ConfigError(f\\"{key} from {source} must be a whole number, got {value!r}\\")\\n    if isinstance(value, int):\\n        return value\\n    if isinstance(value, str):\\n        try:\\n            return int(value.strip())\\n        except ValueError:\\n            pass\\n    raise ConfigError(f\\"{key} from {source} must be a whole number, got {value!r}\\")\\n\\n\\ndef _as_bool(key: str, value: Any, source: str) -> bool:\\n    if isinstance(value, bool):\\n        return value\\n    if isinstance(value, str):\\n        text = value.strip().lower()\\n        if text in _TRUE:\\n            return True\\n        if text in _FALSE:\\n            return False\\n    raise ConfigError(f\\"{key} from {source} must be true or false, got {value!r}\\")\\n\\n\\ndef load_config(path: str | os.PathLike | None = None, env: Mapping[str, str] | None = None) -> Config:\\n    if env is None:\\n        env = os.environ\\n    raw: dict[str, tuple[Any, str]] = {k: (v, \\"defaults\\") for k, v in _DEFAULTS.items()}\\n    if path is not None:\\n        source = f\\"file {os.fspath(path)}\\"\\n        for key, value in _read_file(path).items():\\n            raw[key] = (value, source)\\n    for key in _FIELDS:\\n        name = _env_name(key)\\n        if name in env:\\n            raw[key] = (env[name], f\\"environment variable {name}\\")\\n\\n    if \\"name\\" not in raw:\\n        raise ConfigError(f\\"name is required: set it in the config file or {_env_name(\'name\')}\\")\\n    name, source = raw[\\"name\\"]\\n    if not isinstance(name, str) or not name.strip():\\n        raise ConfigError(f\\"name from {source} must be a non-empty string\\")\\n\\n    port = _as_int(\\"port\\", *raw[\\"port\\"])\\n    if not 1 <= port <= 65535:\\n        raise ConfigError(f\\"port from {raw[\'port\'][1]} must be between 1 and 65535, got {port}\\")\\n\\n    threshold = _as_int(\\"low_stock_threshold\\", *raw[\\"low_stock_threshold\\"])\\n    if threshold < 0:\\n        raise ConfigError(f\\"low_stock_threshold from {raw[\'low_stock_threshold\'][1]} must not be negative\\")\\n\\n    debug = _as_bool(\\"debug\\", *raw[\\"debug\\"])\\n\\n    key, source = raw[\\"sms_api_key\\"]\\n    if key is not None and not isinstance(key, str):\\n        raise ConfigError(f\\"sms_api_key from {source} must be a string\\")\\n    if isinstance(key, str) and not key.strip():\\n        key = None\\n\\n    return Config(name=name.strip(), port=port, debug=debug, low_stock_threshold=threshold, sms_api_key=key)\\n"}, "starter": {}, "task_md": "Our shop server reads its settings from a JSON file and from environment variables, and I want one place that does it.\\nPlease write `shopconfig.py` with `load_config(path=None, env=None) -> Config`: `path` is the optional JSON file and `env` is a mapping of environment variables that defaults to `os.environ`.\\nSettings are `name` (str, required), `port` (int, default 8080), `debug` (bool, default False), `low_stock_threshold` (int, default 5) and `sms_api_key` (str or None, default None).\\nThe environment variables are `SHOP_` plus the upper-case setting name (`SHOP_PORT=9000`, `SHOP_DEBUG=true`, ...) and they win over the file, which wins over the defaults.\\nIf the configuration is wrong in any way, raise `ConfigError` (define it in the module) with a message that tells whoever is deploying what to fix; last month a server started on port 0 because of a typo and nobody noticed.\\nThe returned `Config` should be read-only, and keep in mind `sms_api_key` is a secret while our startup log prints the config."}'),
    json.loads('{"hidden": "import threading\\n\\nimport pytest\\n\\nfrom metrics import MetricsRegistry\\n\\n\\ndef parse(text):\\n    rows = []\\n    for line in text.splitlines():\\n        if not line.strip():\\n            continue\\n        name, value = line.split()\\n        rows.append((name, float(value)))\\n    return rows\\n\\n\\ndef run_threads(target, n, *args):\\n    threads = [threading.Thread(target=target, args=args) for _ in range(n)]\\n    for t in threads:\\n        t.start()\\n    for t in threads:\\n        t.join(timeout=20)\\n    assert not any(t.is_alive() for t in threads)\\n\\n\\ndef test_counter_starts_at_zero_and_counts():\\n    reg = MetricsRegistry()\\n    c = reg.counter(\\"orders_placed_total\\")\\n    assert c.value == 0\\n    c.inc()\\n    c.inc(4)\\n    assert c.value == 5\\n\\n\\ndef test_same_name_same_counter():\\n    reg = MetricsRegistry()\\n    reg.counter(\\"sms_sent_total\\").inc()\\n    reg.counter(\\"sms_sent_total\\").inc(2)\\n    assert reg.counter(\\"sms_sent_total\\") is reg.counter(\\"sms_sent_total\\")\\n    assert reg.counter(\\"sms_sent_total\\").value == 3\\n\\n\\ndef test_counter_cannot_go_down():\\n    c = MetricsRegistry().counter(\\"refunds_total\\")\\n    c.inc(2)\\n    with pytest.raises(ValueError):\\n        c.inc(-1)\\n    assert c.value == 2\\n\\n\\ndef test_counter_rejects_non_numbers():\\n    c = MetricsRegistry().counter(\\"x_total\\")\\n    with pytest.raises((TypeError, ValueError)):\\n        c.inc(\\"5\\")\\n    assert c.value == 0\\n\\n\\ndef test_gauge_up_down_and_set():\\n    g = MetricsRegistry().gauge(\\"open_carts\\")\\n    g.inc()\\n    g.inc(3)\\n    g.dec(5)\\n    assert g.value == -1\\n    g.set(2.5)\\n    assert g.value == 2.5\\n\\n\\n@pytest.mark.parametrize(\\"name\\", [\\"\\", \\"Orders\\", \\"1orders\\", \\"_orders\\", \\"orders-total\\", \\"orders total\\", \\"ऑर्डर\\",\\n                                  \\"orders.total\\"])\\ndef test_bad_names_rejected(name):\\n    reg = MetricsRegistry()\\n    with pytest.raises(ValueError):\\n        reg.counter(name)\\n    with pytest.raises(ValueError):\\n        reg.gauge(name)\\n    assert reg.snapshot() == {}\\n\\n\\ndef test_kind_conflict_rejected():\\n    reg = MetricsRegistry()\\n    reg.counter(\\"orders_total\\")\\n    reg.gauge(\\"queue_depth\\")\\n    with pytest.raises(ValueError):\\n        reg.gauge(\\"orders_total\\")\\n    with pytest.raises(ValueError):\\n        reg.counter(\\"queue_depth\\")\\n\\n\\ndef test_snapshot_is_a_copy():\\n    reg = MetricsRegistry()\\n    reg.counter(\\"a_total\\").inc(2)\\n    snap = reg.snapshot()\\n    assert snap == {\\"a_total\\": 2}\\n    snap[\\"a_total\\"] = 100\\n    snap[\\"fake\\"] = 1\\n    assert reg.snapshot() == {\\"a_total\\": 2}\\n\\n\\ndef test_render_sorted_name_value_lines():\\n    reg = MetricsRegistry()\\n    reg.gauge(\\"temperature_c\\").set(2.5)\\n    reg.counter(\\"b_total\\").inc(3)\\n    reg.counter(\\"a_total\\").inc()\\n    assert parse(reg.render()) == [(\\"a_total\\", 1.0), (\\"b_total\\", 3.0), (\\"temperature_c\\", 2.5)]\\n\\n\\ndef test_render_empty_registry():\\n    assert parse(MetricsRegistry().render()) == []\\n\\n\\ndef test_concurrent_increments_are_not_lost():\\n    reg = MetricsRegistry()\\n    n_threads, per_thread = 8, 5000\\n    barrier = threading.Barrier(n_threads)\\n\\n    def worker():\\n        barrier.wait(timeout=5)\\n        for _ in range(per_thread):\\n            reg.counter(\\"hits_total\\").inc()\\n            reg.gauge(\\"in_flight\\").inc()\\n            reg.gauge(\\"in_flight\\").dec()\\n\\n    run_threads(worker, n_threads)\\n    assert reg.counter(\\"hits_total\\").value == n_threads * per_thread\\n    assert reg.gauge(\\"in_flight\\").value == 0\\n\\n\\ndef test_render_while_new_metrics_are_created():\\n    reg = MetricsRegistry()\\n    stop = threading.Event()\\n    errors = []\\n\\n    def writer():\\n        for i in range(20000):\\n            reg.counter(f\\"sku_{i}_sold_total\\").inc()\\n        stop.set()\\n\\n    def reader():\\n        try:\\n            while not stop.is_set():\\n                reg.render()\\n                reg.snapshot()\\n        except Exception as exc:\\n            errors.append(exc)\\n\\n    readers = [threading.Thread(target=reader) for _ in range(2)]\\n    w = threading.Thread(target=writer)\\n    for t in readers:\\n        t.start()\\n    w.start()\\n    w.join(timeout=20)\\n    stop.set()\\n    for t in readers:\\n        t.join(timeout=20)\\n    assert errors == []\\n    assert len(reg.snapshot()) == 20000\\n", "id": "19-metrics-registry", "meta": {"difficulty": 1, "files": ["metrics.py"], "kind": "build", "timeout": 60}, "reference": {"metrics.py": "\\"\\"\\"Thread-safe counters and gauges with a plain-text exporter.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\nimport threading\\nfrom numbers import Real\\n\\n_NAME = re.compile(r\\"[a-z][a-z0-9_]*\\")\\n\\n\\ndef _check_amount(amount: object) -> None:\\n    if isinstance(amount, bool) or not isinstance(amount, Real):\\n        raise TypeError(f\\"amount must be a number, got {amount!r}\\")\\n\\n\\nclass Counter:\\n    \\"\\"\\"A value that only goes up.\\"\\"\\"\\n\\n    def __init__(self, name: str) -> None:\\n        self.name = name\\n        self._lock = threading.Lock()\\n        self._value: float = 0\\n\\n    def inc(self, amount: float = 1) -> None:\\n        _check_amount(amount)\\n        if amount < 0:\\n            raise ValueError(\\"a counter can only go up\\")\\n        with self._lock:\\n            self._value += amount\\n\\n    @property\\n    def value(self) -> float:\\n        with self._lock:\\n            return self._value\\n\\n\\nclass Gauge:\\n    \\"\\"\\"A value that can go up and down.\\"\\"\\"\\n\\n    def __init__(self, name: str) -> None:\\n        self.name = name\\n        self._lock = threading.Lock()\\n        self._value: float = 0\\n\\n    def set(self, value: float) -> None:\\n        _check_amount(value)\\n        with self._lock:\\n            self._value = value\\n\\n    def inc(self, amount: float = 1) -> None:\\n        _check_amount(amount)\\n        with self._lock:\\n            self._value += amount\\n\\n    def dec(self, amount: float = 1) -> None:\\n        _check_amount(amount)\\n        with self._lock:\\n            self._value -= amount\\n\\n    @property\\n    def value(self) -> float:\\n        with self._lock:\\n            return self._value\\n\\n\\nclass MetricsRegistry:\\n    def __init__(self) -> None:\\n        self._lock = threading.Lock()\\n        self._metrics: dict[str, Counter | Gauge] = {}\\n\\n    def _get_or_create(self, name: str, kind: type) -> Counter | Gauge:\\n        if not isinstance(name, str) or not _NAME.fullmatch(name):\\n            raise ValueError(f\\"invalid metric name: {name!r}\\")\\n        with self._lock:\\n            metric = self._metrics.get(name)\\n            if metric is None:\\n                metric = kind(name)\\n                self._metrics[name] = metric\\n            elif not isinstance(metric, kind):\\n                raise ValueError(f\\"{name!r} is already a {type(metric).__name__.lower()}\\")\\n            return metric\\n\\n    def counter(self, name: str) -> Counter:\\n        return self._get_or_create(name, Counter)\\n\\n    def gauge(self, name: str) -> Gauge:\\n        return self._get_or_create(name, Gauge)\\n\\n    def snapshot(self) -> dict[str, float]:\\n        with self._lock:\\n            metrics = list(self._metrics.items())\\n        return {name: metric.value for name, metric in sorted(metrics)}\\n\\n    def render(self) -> str:\\n        return \\"\\".join(f\\"{name} {value}\\\\n\\" for name, value in self.snapshot().items())\\n"}, "starter": {}, "task_md": "We want basic metrics for the shop server without pulling in prometheus_client, so please write `metrics.py` with a `MetricsRegistry` class.\\n`counter(name)` returns a counter (the same object every time for the same name) with `inc(amount=1)` and `.value`; counters only ever go up.\\n`gauge(name)` returns a gauge with `set(value)`, `inc(amount=1)`, `dec(amount=1)` and `.value`.\\n`snapshot()` returns a dict `{name: value}` of everything, and `render()` returns text with one `name value` line per metric, sorted by name, for our monitoring scraper.\\nNames look like `orders_placed_total`: lowercase letters, digits and underscores, starting with a letter. Raise `ValueError` for any other name, and also when a name that is already a counter is asked for as a gauge or the other way round.\\nEvery request thread updates these at the same time, and the scraper calls `render()` while that is happening."}'),
    json.loads('{"hidden": "import threading\\n\\nimport pytest\\n\\nfrom order_status import InvalidTransition, Order\\n\\n\\nclass FakeClock:\\n    def __init__(self, start=1000.0):\\n        self.now = start\\n\\n    def __call__(self):\\n        return self.now\\n\\n\\ndef make(order_id=\\"ORD-1001\\"):\\n    return Order(order_id, clock=FakeClock(1001.0))\\n\\n\\ndef test_happy_path_with_history():\\n    clock = FakeClock(1001.0)\\n    o = Order(\\"ORD-1001\\", clock=clock)\\n    assert o.status == \\"placed\\"\\n    clock.now = 1002.0\\n    o.pack()\\n    clock.now = 1003.0\\n    o.ship(\\"DTDC123456\\")\\n    clock.now = 1004.0\\n    o.deliver()\\n    assert o.status == \\"delivered\\"\\n    assert list(o.history) == [(\\"placed\\", 1001.0), (\\"packed\\", 1002.0), (\\"shipped\\", 1003.0), (\\"delivered\\", 1004.0)]\\n\\n\\n@pytest.mark.parametrize(\\"steps\\", [[], [\\"pack\\"]])\\ndef test_cancel_before_shipping(steps):\\n    o = make()\\n    for step in steps:\\n        getattr(o, step)()\\n    o.cancel(\\"customer changed mind\\")\\n    assert o.status == \\"cancelled\\"\\n    assert o.history[-1][0] == \\"cancelled\\"\\n\\n\\ndef advance(o, upto):\\n    if upto in (\\"packed\\", \\"shipped\\", \\"delivered\\"):\\n        o.pack()\\n    if upto in (\\"shipped\\", \\"delivered\\"):\\n        o.ship(\\"TRK-1\\")\\n    if upto == \\"delivered\\":\\n        o.deliver()\\n\\n\\n@pytest.mark.parametrize(\\n    \\"state,action\\",\\n    [\\n        (\\"placed\\", \\"deliver\\"),\\n        (\\"placed\\", \\"ship\\"),\\n        (\\"packed\\", \\"pack\\"),\\n        (\\"packed\\", \\"deliver\\"),\\n        (\\"shipped\\", \\"cancel\\"),\\n        (\\"shipped\\", \\"pack\\"),\\n        (\\"shipped\\", \\"ship\\"),\\n        (\\"delivered\\", \\"cancel\\"),\\n        (\\"delivered\\", \\"deliver\\"),\\n        (\\"delivered\\", \\"pack\\"),\\n    ],\\n)\\ndef test_illegal_moves_raise_and_change_nothing(state, action):\\n    o = make()\\n    advance(o, state)\\n    before_history = list(o.history)\\n    args = (\\"TRK-2\\",) if action == \\"ship\\" else ()\\n    with pytest.raises(InvalidTransition):\\n        getattr(o, action)(*args)\\n    assert o.status == state\\n    assert list(o.history) == before_history\\n\\n\\n@pytest.mark.parametrize(\\"action\\", [\\"pack\\", \\"deliver\\", \\"cancel\\"])\\ndef test_cancelled_is_final(action):\\n    o = make()\\n    o.cancel()\\n    with pytest.raises(InvalidTransition):\\n        getattr(o, action)()\\n    with pytest.raises((InvalidTransition, ValueError)):\\n        o.ship(\\"TRK-3\\")\\n    assert o.status == \\"cancelled\\"\\n\\n\\n@pytest.mark.parametrize(\\"tracking\\", [\\"\\", \\"   \\", None])\\ndef test_ship_needs_tracking_id(tracking):\\n    o = make()\\n    o.pack()\\n    with pytest.raises(ValueError):\\n        o.ship(tracking)\\n    assert o.status == \\"packed\\"\\n    assert [s for s, _ in o.history] == [\\"placed\\", \\"packed\\"]\\n\\n\\ndef test_invalid_transition_is_an_exception():\\n    assert issubclass(InvalidTransition, Exception)\\n    o = make()\\n    with pytest.raises(InvalidTransition):\\n        o.deliver()\\n\\n\\ndef test_status_cannot_be_set_directly():\\n    o = make()\\n    try:\\n        o.status = \\"delivered\\"\\n    except (AttributeError, TypeError):\\n        pass\\n    assert o.status == \\"placed\\"\\n\\n\\ndef test_history_cannot_be_changed_from_outside():\\n    o = make()\\n    h = o.history\\n    try:\\n        h.append((\\"delivered\\", 0))\\n    except (AttributeError, TypeError):\\n        pass\\n    assert list(o.history) == [(\\"placed\\", 1001.0)]\\n\\n\\ndef test_orders_are_independent():\\n    a, b = make(\\"A\\"), make(\\"B\\")\\n    a.pack()\\n    assert b.status == \\"placed\\"\\n    assert len(b.history) == 1\\n\\n\\ndef test_only_one_of_many_concurrent_packs_wins():\\n    o = make()\\n    n = 8\\n    barrier = threading.Barrier(n)\\n    results = []\\n    lock = threading.Lock()\\n\\n    def worker():\\n        barrier.wait(timeout=5)\\n        try:\\n            o.pack()\\n            outcome = \\"ok\\"\\n        except InvalidTransition:\\n            outcome = \\"rejected\\"\\n        with lock:\\n            results.append(outcome)\\n\\n    threads = [threading.Thread(target=worker) for _ in range(n)]\\n    for t in threads:\\n        t.start()\\n    for t in threads:\\n        t.join(timeout=5)\\n    assert sorted(results) == [\\"ok\\"] + [\\"rejected\\"] * (n - 1)\\n    assert [s for s, _ in o.history] == [\\"placed\\", \\"packed\\"]\\n", "id": "20-order-status-machine", "meta": {"difficulty": 1, "files": ["order_status.py"], "kind": "build", "timeout": 60}, "reference": {"order_status.py": "\\"\\"\\"Order status rules: placed -> packed -> shipped -> delivered, cancel only before shipping.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport threading\\nimport time\\nfrom collections.abc import Callable\\n\\nPLACED = \\"placed\\"\\nPACKED = \\"packed\\"\\nSHIPPED = \\"shipped\\"\\nDELIVERED = \\"delivered\\"\\nCANCELLED = \\"cancelled\\"\\n\\n_ALLOWED: dict[str, frozenset[str]] = {\\n    PLACED: frozenset({PACKED, CANCELLED}),\\n    PACKED: frozenset({SHIPPED, CANCELLED}),\\n    SHIPPED: frozenset({DELIVERED}),\\n    DELIVERED: frozenset(),\\n    CANCELLED: frozenset(),\\n}\\n\\n\\nclass InvalidTransition(Exception):\\n    \\"\\"\\"The requested status change is not allowed from the current status.\\"\\"\\"\\n\\n\\nclass Order:\\n    def __init__(self, order_id: str, clock: Callable[[], float] = time.time) -> None:\\n        if not isinstance(order_id, str) or not order_id.strip():\\n            raise ValueError(\\"order_id must be a non-empty string\\")\\n        self._order_id = order_id\\n        self._clock = clock\\n        self._lock = threading.Lock()\\n        self._status = PLACED\\n        self._history: list[tuple[str, float]] = [(PLACED, clock())]\\n        self._tracking_id: str | None = None\\n        self._cancel_reason: str | None = None\\n\\n    @property\\n    def order_id(self) -> str:\\n        return self._order_id\\n\\n    @property\\n    def status(self) -> str:\\n        return self._status\\n\\n    @property\\n    def history(self) -> list[tuple[str, float]]:\\n        with self._lock:\\n            return list(self._history)\\n\\n    @property\\n    def tracking_id(self) -> str | None:\\n        return self._tracking_id\\n\\n    @property\\n    def cancel_reason(self) -> str | None:\\n        return self._cancel_reason\\n\\n    def _move(self, new_status: str) -> None:\\n        # caller holds self._lock\\n        if new_status not in _ALLOWED[self._status]:\\n            raise InvalidTransition(f\\"order {self._order_id}: cannot go from {self._status} to {new_status}\\")\\n        self._status = new_status\\n        self._history.append((new_status, self._clock()))\\n\\n    def pack(self) -> None:\\n        with self._lock:\\n            self._move(PACKED)\\n\\n    def ship(self, tracking_id: str) -> None:\\n        if not isinstance(tracking_id, str) or not tracking_id.strip():\\n            raise ValueError(\\"a non-empty tracking id is required to ship\\")\\n        with self._lock:\\n            self._move(SHIPPED)\\n            self._tracking_id = tracking_id.strip()\\n\\n    def deliver(self) -> None:\\n        with self._lock:\\n            self._move(DELIVERED)\\n\\n    def cancel(self, reason: str = \\"\\") -> None:\\n        with self._lock:\\n            self._move(CANCELLED)\\n            self._cancel_reason = reason\\n\\n    def __repr__(self) -> str:\\n        return f\\"Order({self._order_id!r}, status={self._status!r})\\"\\n"}, "starter": {}, "task_md": "Our delivery staff keep marking orders \\"delivered\\" that were never shipped, so I want the order status rules in code.\\nPlease write `order_status.py` with a class `Order(order_id: str, clock=time.time)` that starts in status `\\"placed\\"` and has methods `pack()`, `ship(tracking_id: str)`, `deliver()` and `cancel(reason: str = \\"\\")`.\\nThe flow is placed → packed → shipped → delivered, and an order can be cancelled only before it ships.\\nAny other move must raise `InvalidTransition` (define it in the module), and shipping needs a real tracking id (`ValueError` otherwise).\\n`order.status` gives the current status and `order.history` the list of `(status, timestamp)` steps, starting with `(\\"placed\\", <time it was created>)`; timestamps come from `clock()`.\\nThe status must only ever change through these methods."}'),
    json.loads('{"hidden": "import random\\nimport time\\nfrom fractions import Fraction\\n\\nimport pytest\\n\\nfrom splitpay import settle, split_bill\\n\\n\\ndef expected_split(total, weights):\\n    s = sum(weights)\\n    base = [total * w // s for w in weights]\\n    rems = sorted(range(len(weights)), key=lambda i: (-(total * weights[i] % s), i))\\n    for i in rems[: total - sum(base)]:\\n        base[i] += 1\\n    return base\\n\\n\\n@pytest.mark.parametrize(\\n    \\"total,weights,expected\\",\\n    [\\n        (100, [1, 1, 1], [34, 33, 33]),\\n        (1001, [1, 1, 2], [250, 250, 501]),\\n        (10, [1] * 7, [2, 2, 2, 1, 1, 1, 1]),\\n        (7, [3, 0, 1], [5, 0, 2]),\\n        (0, [1, 2, 3], [0, 0, 0]),\\n        (99999, [5], [99999]),\\n        (5, [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 0, 0, 0]),\\n    ],\\n)\\ndef test_split_known_cases(total, weights, expected):\\n    assert split_bill(total, weights) == expected\\n\\n\\ndef test_split_huge_amount_is_exact():\\n    total = 10**18 + 1\\n    parts = split_bill(total, [1, 2])\\n    assert parts == [333333333333333334, 666666666666666667]\\n    assert sum(parts) == total\\n\\n\\ndef test_split_random_properties():\\n    rng = random.Random(2024)\\n    for _ in range(300):\\n        n = rng.randint(1, 12)\\n        weights = [rng.randint(0, 9) for _ in range(n)]\\n        if sum(weights) == 0:\\n            weights[0] = 1\\n        total = rng.randint(0, 5_000_000)\\n        parts = split_bill(total, weights)\\n        assert all(isinstance(p, int) for p in parts)\\n        assert sum(parts) == total\\n        for p, w in zip(parts, weights):\\n            assert abs(p - Fraction(total * w, sum(weights))) < 1\\n        assert parts == expected_split(total, weights)\\n\\n\\ndef test_split_does_not_modify_weights():\\n    weights = [1, 2, 3]\\n    split_bill(100, weights)\\n    assert weights == [1, 2, 3]\\n\\n\\n@pytest.mark.parametrize(\\"weights\\", [[], [0, 0], [1, -1, 2]])\\ndef test_split_bad_weights(weights):\\n    with pytest.raises(ValueError):\\n        split_bill(100, weights)\\n\\n\\n@pytest.mark.parametrize(\\"total\\", [100.5, 10.0, \\"100\\"])\\ndef test_split_rejects_non_integer_total(total):\\n    with pytest.raises((TypeError, ValueError)):\\n        split_bill(total, [1, 1])\\n\\n\\ndef test_split_is_fast_for_many_people():\\n    start = time.perf_counter()\\n    parts = split_bill(123_456_789, [1 + i % 5 for i in range(200_000)])\\n    assert sum(parts) == 123_456_789\\n    assert time.perf_counter() - start < 2\\n\\n\\ndef apply(balances, payments):\\n    left = dict(balances)\\n    for payer, payee, amount in payments:\\n        left[payer] += amount\\n        left[payee] -= amount\\n    return left\\n\\n\\ndef test_settle_simple():\\n    balances = {\\"Amit\\": 500, \\"Bunty\\": -300, \\"Chintu\\": -200}\\n    payments = settle(balances)\\n    assert sorted(payments) == [(\\"Bunty\\", \\"Amit\\", 300), (\\"Chintu\\", \\"Amit\\", 200)]\\n\\n\\ndef test_settle_all_even():\\n    assert list(settle({\\"a\\": 0, \\"b\\": 0})) == []\\n    assert list(settle({})) == []\\n\\n\\ndef test_settle_unbalanced_raises():\\n    with pytest.raises(ValueError):\\n        settle({\\"a\\": 100, \\"b\\": -99})\\n\\n\\ndef test_settle_rejects_float_money():\\n    with pytest.raises((TypeError, ValueError)):\\n        settle({\\"a\\": 10.5, \\"b\\": -10.5})\\n\\n\\ndef test_settle_random_properties():\\n    rng = random.Random(7)\\n    for _ in range(200):\\n        names = [f\\"p{i}\\" for i in range(rng.randint(2, 10))]\\n        balances = {n: rng.randint(-50_000, 50_000) for n in names[:-1]}\\n        balances[names[-1]] = -sum(balances.values())\\n        before = dict(balances)\\n        payments = [tuple(p) for p in settle(balances)]\\n        assert balances == before  # input untouched\\n        assert all(v == 0 for v in apply(balances, payments).values())\\n        for payer, payee, amount in payments:\\n            assert isinstance(amount, int) and amount > 0\\n            assert payer != payee\\n            assert balances[payer] < 0 < balances[payee]\\n        nonzero = sum(1 for v in balances.values() if v)\\n        assert len(payments) <= max(nonzero - 1, 0)\\n\\n\\ndef test_settle_same_balances_same_sheet_regardless_of_order():\\n    items = [(\\"Amit\\", 700), (\\"Bunty\\", -300), (\\"Chintu\\", -300), (\\"Dolly\\", 200), (\\"Eshan\\", -300)]\\n    first = [tuple(p) for p in settle(dict(items))]\\n    for seed in range(10):\\n        shuffled = items[:]\\n        random.Random(seed).shuffle(shuffled)\\n        assert [tuple(p) for p in settle(dict(shuffled))] == first\\n", "id": "21-bill-split-settlement", "meta": {"difficulty": 3, "files": ["splitpay.py"], "kind": "build", "timeout": 60}, "reference": {"splitpay.py": "\\"\\"\\"Split bills and settle balances in integer paise (no floating point anywhere).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport heapq\\nfrom collections.abc import Mapping, Sequence\\n\\n\\ndef _require_int(value: object, what: str) -> int:\\n    if isinstance(value, bool) or not isinstance(value, int):\\n        raise TypeError(f\\"{what} must be an integer number of paise, got {value!r}\\")\\n    return value\\n\\n\\ndef split_bill(total_paise: int, weights: Sequence[int]) -> list[int]:\\n    \\"\\"\\"Split `total_paise` by `weights` with the largest-remainder method.\\n\\n    Every part is the exact share rounded down or up, the parts sum to the total, and the\\n    leftover paise go to the largest remainders (earlier people win ties).\\n    \\"\\"\\"\\n    _require_int(total_paise, \\"total_paise\\")\\n    weights = list(weights)\\n    if not weights:\\n        raise ValueError(\\"need at least one weight\\")\\n    for w in weights:\\n        if isinstance(w, bool) or not isinstance(w, int):\\n            raise TypeError(f\\"weights must be whole numbers, got {w!r}\\")\\n        if w < 0:\\n            raise ValueError(\\"weights must not be negative\\")\\n    weight_sum = sum(weights)\\n    if weight_sum == 0:\\n        raise ValueError(\\"at least one weight must be positive\\")\\n\\n    sign = -1 if total_paise < 0 else 1\\n    total = abs(total_paise)\\n    parts = []\\n    remainders = []\\n    for i, w in enumerate(weights):\\n        q, r = divmod(total * w, weight_sum)\\n        parts.append(q)\\n        remainders.append((-r, i))\\n    leftover = total - sum(parts)\\n    for _, i in sorted(remainders)[:leftover]:\\n        parts[i] += 1\\n    return [sign * p for p in parts]\\n\\n\\ndef settle(balances: Mapping[str, int]) -> list[tuple[str, str, int]]:\\n    \\"\\"\\"Return payments (payer, payee, amount) that bring every balance to zero.\\n\\n    Greedy: the biggest debtor pays the biggest creditor. Uses at most n-1 payments and is\\n    deterministic (ties are broken by name, never by dict order).\\n    \\"\\"\\"\\n    for name, amount in balances.items():\\n        if not isinstance(name, str):\\n            raise TypeError(f\\"names must be strings, got {name!r}\\")\\n        _require_int(amount, f\\"balance of {name}\\")\\n    if sum(balances.values()) != 0:\\n        raise ValueError(f\\"balances must add up to zero, they add up to {sum(balances.values())}\\")\\n\\n    creditors = [(-amount, name) for name, amount in balances.items() if amount > 0]\\n    debtors = [(amount, name) for name, amount in balances.items() if amount < 0]\\n    heapq.heapify(creditors)\\n    heapq.heapify(debtors)\\n    payments: list[tuple[str, str, int]] = []\\n    while creditors and debtors:\\n        neg_credit, payee = heapq.heappop(creditors)\\n        debt, payer = heapq.heappop(debtors)\\n        amount = min(-neg_credit, -debt)\\n        payments.append((payer, payee, amount))\\n        if -neg_credit > amount:\\n            heapq.heappush(creditors, (neg_credit + amount, payee))\\n        if -debt > amount:\\n            heapq.heappush(debtors, (debt + amount, payer))\\n    return payments\\n"}, "starter": {}, "task_md": "Three brothers run our shop together and keep paying supplier bills from their own pockets, so we need a small settlement helper in `splitpay.py`. All amounts are integer paise.\\n`split_bill(total_paise: int, weights: list[int]) -> list[int]` splits a bill by whole-number weights (e.g. `[1, 1, 2]` when one person\'s share is double). The parts must add up to exactly the total, nobody may be off from their exact share by a paisa or more, and the leftover paise go to the people whose exact share was cut the most, with ties going to whoever comes first in the list.\\n`settle(balances: dict[str, int]) -> list[tuple[str, str, int]]` takes each person\'s net balance (positive means the others owe them, negative means they owe) and returns the payments `(payer, payee, amount_paise)` that make everyone even, using as few payments as you reasonably can.\\nBalances that don\'t add up to zero are a data-entry mistake and should raise `ValueError`.\\nWe print the settlement sheet every month, so the same balances must always give the same list."}'),
    json.loads('{"hidden": "import threading\\n\\nimport pytest\\n\\nfrom stock import OutOfStock, StockBook\\n\\n\\nclass ListLedger:\\n    def __init__(self):\\n        self.lock = threading.Lock()\\n        self.entries = []\\n\\n    def record(self, action, sku, qty):\\n        with self.lock:\\n            self.entries.append((action, sku, qty))\\n\\n\\nclass SlowLedger(ListLedger):\\n    \\"\\"\\"A ledger whose disk write is slow: each write waits (briefly) for the other till.\\n\\n    With correct locking the second till is not inside the critical section, the barrier\\n    times out and the call simply continues, so the test is deterministic either way.\\n    \\"\\"\\"\\n\\n    def __init__(self, parties=2):\\n        super().__init__()\\n        self.barrier = threading.Barrier(parties)\\n\\n    def record(self, action, sku, qty):\\n        try:\\n            self.barrier.wait(timeout=0.3)\\n        except threading.BrokenBarrierError:\\n            pass\\n        super().record(action, sku, qty)\\n\\n\\ndef run_together(n, target):\\n    results, lock = [], threading.Lock()\\n\\n    def wrapped():\\n        try:\\n            target()\\n            outcome = \\"ok\\"\\n        except OutOfStock:\\n            outcome = \\"out\\"\\n        with lock:\\n            results.append(outcome)\\n\\n    threads = [threading.Thread(target=wrapped) for _ in range(n)]\\n    for t in threads:\\n        t.start()\\n    for t in threads:\\n        t.join(timeout=10)\\n    assert not any(t.is_alive() for t in threads), \\"threads deadlocked\\"\\n    return sorted(results)\\n\\n\\ndef test_reserve_and_available():\\n    book = StockBook({\\"COOKER-5L\\": 3})\\n    book.reserve(\\"COOKER-5L\\", 2)\\n    assert book.available(\\"COOKER-5L\\") == 1\\n    assert book.available(\\"UNKNOWN\\") == 0\\n\\n\\ndef test_out_of_stock_leaves_stock_alone():\\n    ledger = ListLedger()\\n    book = StockBook({\\"COOKER-5L\\": 1}, ledger=ledger)\\n    with pytest.raises(OutOfStock):\\n        book.reserve(\\"COOKER-5L\\", 2)\\n    assert book.available(\\"COOKER-5L\\") == 1\\n    assert ledger.entries == []\\n\\n\\n@pytest.mark.parametrize(\\"method\\", [\\"reserve\\", \\"release\\", \\"restock\\"])\\n@pytest.mark.parametrize(\\"qty\\", [0, -1])\\ndef test_bad_qty_still_rejected(method, qty):\\n    book = StockBook({\\"X\\": 5})\\n    with pytest.raises(ValueError):\\n        getattr(book, method)(\\"X\\", qty)\\n    assert book.available(\\"X\\") == 5\\n\\n\\ndef test_release_and_restock_add_stock():\\n    ledger = ListLedger()\\n    book = StockBook({\\"DIYA-BOX\\": 2}, ledger=ledger)\\n    book.restock(\\"DIYA-BOX\\", 5)\\n    book.release(\\"DIYA-BOX\\", 1)\\n    book.restock(\\"NEW-SKU\\", 4)\\n    assert book.available(\\"DIYA-BOX\\") == 8\\n    assert book.available(\\"NEW-SKU\\") == 4\\n    assert ledger.entries == [(\\"restock\\", \\"DIYA-BOX\\", 5), (\\"release\\", \\"DIYA-BOX\\", 1), (\\"restock\\", \\"NEW-SKU\\", 4)]\\n\\n\\ndef test_two_tills_cannot_both_sell_the_last_cooker():\\n    ledger = SlowLedger()\\n    book = StockBook({\\"COOKER-5L\\": 1}, ledger=ledger)\\n    results = run_together(2, lambda: book.reserve(\\"COOKER-5L\\", 1))\\n    assert results == [\\"ok\\", \\"out\\"]\\n    assert book.available(\\"COOKER-5L\\") == 0\\n    assert ledger.entries == [(\\"reserve\\", \\"COOKER-5L\\", 1)]\\n\\n\\ndef test_concurrent_restocks_are_not_lost():\\n    book = StockBook({\\"DIYA-BOX\\": 10}, ledger=SlowLedger())\\n    run_together(2, lambda: book.restock(\\"DIYA-BOX\\", 5))\\n    assert book.available(\\"DIYA-BOX\\") == 20\\n\\n\\ndef test_concurrent_releases_are_not_lost():\\n    book = StockBook({\\"DIYA-BOX\\": 0}, ledger=SlowLedger())\\n    run_together(2, lambda: book.release(\\"DIYA-BOX\\", 3))\\n    assert book.available(\\"DIYA-BOX\\") == 6\\n\\n\\ndef test_reserve_racing_restock():\\n    book = StockBook({\\"OIL-1L\\": 5}, ledger=SlowLedger())\\n    barrier = threading.Barrier(2)\\n\\n    def sell():\\n        barrier.wait(timeout=5)\\n        book.reserve(\\"OIL-1L\\", 5)\\n\\n    def deliver():\\n        barrier.wait(timeout=5)\\n        book.restock(\\"OIL-1L\\", 10)\\n\\n    threads = [threading.Thread(target=sell), threading.Thread(target=deliver)]\\n    for t in threads:\\n        t.start()\\n    for t in threads:\\n        t.join(timeout=10)\\n    assert book.available(\\"OIL-1L\\") == 10\\n\\n\\ndef test_many_tills_sell_exactly_the_stock():\\n    ledger = ListLedger()\\n    book = StockBook({\\"SUGAR-1KG\\": 100}, ledger=ledger)\\n    n_threads, attempts = 8, 25\\n    barrier = threading.Barrier(n_threads)\\n    sold, lock = [0], threading.Lock()\\n\\n    def till():\\n        barrier.wait(timeout=5)\\n        for _ in range(attempts):\\n            try:\\n                book.reserve(\\"SUGAR-1KG\\", 1)\\n            except OutOfStock:\\n                continue\\n            with lock:\\n                sold[0] += 1\\n\\n    threads = [threading.Thread(target=till) for _ in range(n_threads)]\\n    for t in threads:\\n        t.start()\\n    for t in threads:\\n        t.join(timeout=10)\\n    assert sold[0] == 100\\n    assert book.available(\\"SUGAR-1KG\\") == 0\\n    assert len(ledger.entries) == 100\\n\\n\\ndef test_ledger_failure_changes_nothing_and_does_not_lock_up():\\n    class BrokenLedger:\\n        def __init__(self):\\n            self.fail = True\\n\\n        def record(self, action, sku, qty):\\n            if self.fail:\\n                raise OSError(\\"disk full\\")\\n\\n    ledger = BrokenLedger()\\n    book = StockBook({\\"RICE-5KG\\": 3}, ledger=ledger)\\n    with pytest.raises(OSError):\\n        book.reserve(\\"RICE-5KG\\", 1)\\n    with pytest.raises(OSError):\\n        book.restock(\\"RICE-5KG\\", 1)\\n    assert book.available(\\"RICE-5KG\\") == 3\\n    ledger.fail = False\\n    done = []\\n    t = threading.Thread(target=lambda: (book.reserve(\\"RICE-5KG\\", 1), done.append(True)))\\n    t.start()\\n    t.join(timeout=5)\\n    assert done, \\"StockBook stayed locked after a ledger error\\"\\n    assert book.available(\\"RICE-5KG\\") == 2\\n", "id": "22-stock-oversell-race", "meta": {"difficulty": 2, "files": ["stock.py"], "kind": "bugfix", "timeout": 60}, "reference": {"stock.py": "\\"\\"\\"Shared stock counts for all the tills in the shop.\\n\\nEvery till talks to one StockBook object inside the shop server (one thread per request).\\nEach change is also written to the ledger (a slow append to a file on disk).\\n\\nCheck-and-update of a count happens under a lock, so two tills can never both take the\\nlast unit and concurrent restocks are never lost. The ledger is written while the lock is\\nheld so that a failed ledger write leaves the stock unchanged, exactly as before.\\n\\"\\"\\"\\n\\nimport threading\\n\\n\\nclass OutOfStock(Exception):\\n    pass\\n\\n\\nclass NullLedger:\\n    def record(self, action, sku, qty):\\n        pass\\n\\n\\nclass StockBook:\\n    def __init__(self, stock=None, ledger=None):\\n        self._stock = dict(stock or {})\\n        self._ledger = ledger if ledger is not None else NullLedger()\\n        self._lock = threading.Lock()\\n\\n    def available(self, sku):\\n        with self._lock:\\n            return self._stock.get(sku, 0)\\n\\n    @staticmethod\\n    def _check_qty(qty):\\n        if qty <= 0:\\n            raise ValueError(\\"qty must be positive\\")\\n\\n    def reserve(self, sku, qty):\\n        \\"\\"\\"Take `qty` units for a customer\'s bill. Raises OutOfStock if there aren\'t enough.\\"\\"\\"\\n        self._check_qty(qty)\\n        with self._lock:\\n            available = self._stock.get(sku, 0)\\n            if qty > available:\\n                raise OutOfStock(f\\"{sku}: wanted {qty}, have {available}\\")\\n            self._ledger.record(\\"reserve\\", sku, qty)\\n            self._stock[sku] = available - qty\\n\\n    def _add(self, action, sku, qty):\\n        self._check_qty(qty)\\n        with self._lock:\\n            current = self._stock.get(sku, 0)\\n            self._ledger.record(action, sku, qty)\\n            self._stock[sku] = current + qty\\n\\n    def release(self, sku, qty):\\n        \\"\\"\\"Put units back when a bill is cancelled.\\"\\"\\"\\n        self._add(\\"release\\", sku, qty)\\n\\n    def restock(self, sku, qty):\\n        \\"\\"\\"Add units from a supplier delivery.\\"\\"\\"\\n        self._add(\\"restock\\", sku, qty)\\n"}, "starter": {"stock.py": "\\"\\"\\"Shared stock counts for all the tills in the shop.\\n\\nEvery till talks to one StockBook object inside the shop server (one thread per request).\\nEach change is also written to the ledger (a slow append to a file on disk).\\n\\"\\"\\"\\n\\n\\nclass OutOfStock(Exception):\\n    pass\\n\\n\\nclass NullLedger:\\n    def record(self, action, sku, qty):\\n        pass\\n\\n\\nclass StockBook:\\n    def __init__(self, stock=None, ledger=None):\\n        self._stock = dict(stock or {})\\n        self._ledger = ledger if ledger is not None else NullLedger()\\n\\n    def available(self, sku):\\n        return self._stock.get(sku, 0)\\n\\n    def reserve(self, sku, qty):\\n        \\"\\"\\"Take `qty` units for a customer\'s bill. Raises OutOfStock if there aren\'t enough.\\"\\"\\"\\n        if qty <= 0:\\n            raise ValueError(\\"qty must be positive\\")\\n        available = self._stock.get(sku, 0)\\n        if qty > available:\\n            raise OutOfStock(f\\"{sku}: wanted {qty}, have {available}\\")\\n        self._ledger.record(\\"reserve\\", sku, qty)\\n        self._stock[sku] = available - qty\\n\\n    def release(self, sku, qty):\\n        \\"\\"\\"Put units back when a bill is cancelled.\\"\\"\\"\\n        if qty <= 0:\\n            raise ValueError(\\"qty must be positive\\")\\n        current = self._stock.get(sku, 0)\\n        self._ledger.record(\\"release\\", sku, qty)\\n        self._stock[sku] = current + qty\\n\\n    def restock(self, sku, qty):\\n        \\"\\"\\"Add units from a supplier delivery.\\"\\"\\"\\n        if qty <= 0:\\n            raise ValueError(\\"qty must be positive\\")\\n        current = self._stock.get(sku, 0)\\n        self._ledger.record(\\"restock\\", sku, qty)\\n        self._stock[sku] = current + qty\\n"}, "task_md": "Bug report from the Diwali sale: we had 1 pressure cooker left and two tills sold it at the same moment. Both bills went through, the stock shows 0, and one customer went home without a cooker.\\nThe same evening two staff members each restocked 5 boxes of diyas at the same time, and the stock only went up by 5.\\nAll tills share one `StockBook` in `stock.py` inside the shop server, one thread per request, and every change is written to the ledger, which is a slow write to disk.\\nPlease fix `stock.py` so this can\'t happen again; the tills and the ledger keep using it exactly as they do now (`reserve`, `release`, `restock`, `available`, `OutOfStock`)."}'),
    json.loads('{"hidden": "import os\\n\\nimport pytest\\n\\nfrom saleslog import SalesLog\\n\\nFD_DIR = \\"/proc/self/fd\\"\\nneeds_proc = pytest.mark.skipif(not os.path.isdir(FD_DIR), reason=\\"needs /proc/self/fd\\")\\n\\n\\ndef open_fds():\\n    return len(os.listdir(FD_DIR))\\n\\n\\ndef test_total_sees_sales_right_away(tmp_path):\\n    log = SalesLog(str(tmp_path))\\n    log.record(\\"2024-11-01\\", \\"B-1\\", 15000)\\n    log.record(\\"2024-11-01\\", \\"B-2\\", 2550)\\n    assert log.total(\\"2024-11-01\\") == 17550\\n    log.record(\\"2024-11-01\\", \\"B-3\\", 450)\\n    assert log.total(\\"2024-11-01\\") == 18000\\n    log.close()\\n\\n\\ndef test_another_reader_sees_sales_without_close(tmp_path):\\n    writer = SalesLog(str(tmp_path))\\n    writer.record(\\"2024-11-02\\", \\"B-1\\", 999)\\n    reader = SalesLog(str(tmp_path))\\n    assert reader.total(\\"2024-11-02\\") == 999\\n    with open(tmp_path / \\"2024-11-02.csv\\", encoding=\\"utf-8\\") as f:\\n        assert f.read() == \\"B-1,999\\\\n\\"\\n    writer.close()\\n    reader.close()\\n\\n\\ndef test_days_are_separate_and_unknown_day_is_zero(tmp_path):\\n    log = SalesLog(str(tmp_path))\\n    log.record(\\"2024-11-01\\", \\"A\\", 100)\\n    log.record(\\"2024-11-02\\", \\"B\\", 200)\\n    assert log.total(\\"2024-11-01\\") == 100\\n    assert log.total(\\"2024-11-02\\") == 200\\n    assert log.total(\\"2024-12-25\\") == 0\\n    log.close()\\n\\n\\ndef test_appends_to_existing_file_from_yesterday(tmp_path):\\n    (tmp_path / \\"2024-11-03.csv\\").write_text(\\"OLD-1,500\\\\n\\", encoding=\\"utf-8\\")\\n    log = SalesLog(str(tmp_path))\\n    log.record(\\"2024-11-03\\", \\"NEW-1\\", 700)\\n    assert log.total(\\"2024-11-03\\") == 1200\\n    log.close()\\n\\n\\n@needs_proc\\ndef test_many_days_do_not_pile_up_open_files(tmp_path):\\n    log = SalesLog(str(tmp_path))\\n    before = open_fds()\\n    for i in range(300):\\n        log.record(f\\"day-{i:03d}\\", f\\"B-{i}\\", i)\\n    during = open_fds()\\n    assert during - before < 10, f\\"{during - before} files left open while the server runs\\"\\n    assert log.total(\\"day-299\\") == 299\\n    log.close()\\n\\n\\n@needs_proc\\ndef test_total_does_not_leave_files_open(tmp_path):\\n    log = SalesLog(str(tmp_path))\\n    log.record(\\"2024-11-04\\", \\"B-1\\", 100)\\n    log.total(\\"2024-11-04\\")\\n    before = open_fds()\\n    for _ in range(100):\\n        assert log.total(\\"2024-11-04\\") == 100\\n    assert open_fds() - before < 5\\n    log.close()\\n\\n\\n@needs_proc\\ndef test_corrupt_file_raises_value_error_without_leaking(tmp_path):\\n    (tmp_path / \\"2024-11-05.csv\\").write_text(\\"B-1,100\\\\nB-2,abc\\\\n\\", encoding=\\"utf-8\\")\\n    log = SalesLog(str(tmp_path))\\n    before = open_fds()\\n    kept = []  # the server\'s error reporter keeps recent exceptions (and their tracebacks)\\n    for _ in range(50):\\n        with pytest.raises(ValueError) as excinfo:\\n            log.total(\\"2024-11-05\\")\\n        kept.append(excinfo)\\n    assert open_fds() - before < 5\\n    log.close()\\n\\n\\ndef test_close_is_safe_to_call_twice_and_keeps_data(tmp_path):\\n    log = SalesLog(str(tmp_path))\\n    log.record(\\"2024-11-06\\", \\"B-1\\", 42)\\n    log.close()\\n    log.close()\\n    assert SalesLog(str(tmp_path)).total(\\"2024-11-06\\") == 42\\n\\n\\ndef test_many_sales_one_day(tmp_path):\\n    log = SalesLog(str(tmp_path))\\n    for i in range(1000):\\n        log.record(\\"2024-11-07\\", f\\"B-{i}\\", 10)\\n    assert log.total(\\"2024-11-07\\") == 10000\\n    log.close()\\n    lines = (tmp_path / \\"2024-11-07.csv\\").read_text(encoding=\\"utf-8\\").splitlines()\\n    assert len(lines) == 1000\\n    assert lines[0] == \\"B-0,10\\"\\n", "id": "23-sales-log-file-leak", "meta": {"difficulty": 2, "files": ["saleslog.py"], "kind": "bugfix", "timeout": 60}, "reference": {"saleslog.py": "\\"\\"\\"Day-wise sales log: one CSV file per day (`<day>.csv`, lines `order_id,amount_paise`).\\n\\nFiles are opened only for the duration of one operation (`with` blocks), so nothing leaks\\nfile descriptors and every recorded sale is on disk as soon as `record` returns.\\n\\"\\"\\"\\n\\nimport os\\nimport threading\\n\\n\\nclass SalesLog:\\n    def __init__(self, folder):\\n        self.folder = folder\\n        self._lock = threading.Lock()\\n\\n    def _path(self, day):\\n        return os.path.join(self.folder, f\\"{day}.csv\\")\\n\\n    def record(self, day, order_id, amount_paise):\\n        \\"\\"\\"Append one sale. `day` is like \\"2024-11-01\\".\\"\\"\\"\\n        line = f\\"{order_id},{amount_paise}\\\\n\\"\\n        with self._lock, open(self._path(day), \\"a\\", encoding=\\"utf-8\\") as f:\\n            f.write(line)\\n\\n    def total(self, day):\\n        \\"\\"\\"Total paise sold on `day` (0 if nothing was sold). Raises ValueError on a corrupt line.\\"\\"\\"\\n        path = self._path(day)\\n        if not os.path.exists(path):\\n            return 0\\n        total = 0\\n        with open(path, encoding=\\"utf-8\\") as f:\\n            for n, raw in enumerate(f, start=1):\\n                line = raw.strip()\\n                if not line:\\n                    continue\\n                parts = line.split(\\",\\")\\n                if len(parts) != 2:\\n                    raise ValueError(f\\"{path} line {n}: expected order_id,amount_paise\\")\\n                try:\\n                    total += int(parts[1])\\n                except ValueError:\\n                    raise ValueError(f\\"{path} line {n}: bad amount {parts[1]!r}\\") from None\\n        return total\\n\\n    def close(self):\\n        \\"\\"\\"Nothing stays open between calls; kept so existing callers keep working.\\"\\"\\"\\n\\n    def __enter__(self):\\n        return self\\n\\n    def __exit__(self, *exc):\\n        self.close()\\n"}, "starter": {"saleslog.py": "\\"\\"\\"Day-wise sales log: one CSV file per day (`<day>.csv`, lines `order_id,amount_paise`).\\"\\"\\"\\n\\nimport os\\n\\n\\nclass SalesLog:\\n    def __init__(self, folder):\\n        self.folder = folder\\n        self._files = {}\\n\\n    def _path(self, day):\\n        return os.path.join(self.folder, f\\"{day}.csv\\")\\n\\n    def _file_for(self, day):\\n        # keep the file open so we don\'t reopen it for every sale\\n        if day not in self._files:\\n            self._files[day] = open(self._path(day), \\"a\\", encoding=\\"utf-8\\")\\n        return self._files[day]\\n\\n    def record(self, day, order_id, amount_paise):\\n        \\"\\"\\"Append one sale. `day` is like \\"2024-11-01\\".\\"\\"\\"\\n        f = self._file_for(day)\\n        f.write(f\\"{order_id},{amount_paise}\\\\n\\")\\n\\n    def total(self, day):\\n        \\"\\"\\"Total paise sold on `day` (0 if nothing was sold). Raises ValueError on a corrupt line.\\"\\"\\"\\n        path = self._path(day)\\n        if not os.path.exists(path):\\n            return 0\\n        f = open(path, encoding=\\"utf-8\\")\\n        total = 0\\n        for n, line in enumerate(f, start=1):\\n            line = line.strip()\\n            if not line:\\n                continue\\n            order_id, amount = line.split(\\",\\")\\n            try:\\n                total += int(amount)\\n            except ValueError:\\n                raise ValueError(f\\"{path} line {n}: bad amount {amount!r}\\")\\n        return total\\n\\n    def close(self):\\n        for f in self._files.values():\\n            f.close()\\n"}, "task_md": "Our shop server dies every few weeks with `OSError: [Errno 24] Too many open files`, and we restart it.\\nThere\'s a second problem that may be related: the evening total for today from `SalesLog.total()` is often lower than the bills we printed, and after a restart the same day\'s total is right again.\\nThe code is in `saleslog.py`. Please find and fix the cause; the server keeps calling `record(day, order_id, amount_paise)`, `total(day)` and `close()` exactly as it does now, and the CSV files on disk must stay in the same format."}'),
    json.loads('{"hidden": "import json\\nimport os\\nimport subprocess\\nimport sys\\nimport textwrap\\nimport unicodedata\\n\\nimport pytest\\n\\nimport catalog\\nfrom catalog import load_catalog, receipt_label, save_catalog, search\\n\\nJA = \\"\\\\N{DEVANAGARI LETTER JA}\\"\\nNUKTA = \\"\\\\N{DEVANAGARI SIGN NUKTA}\\"\\nZA_PRECOMPOSED = \\"\\\\N{DEVANAGARI LETTER ZA}\\"\\nII = \\"\\\\N{DEVANAGARI VOWEL SIGN II}\\"\\nRA = \\"\\\\N{DEVANAGARI LETTER RA}\\"\\nAA_SIGN = \\"\\\\N{DEVANAGARI VOWEL SIGN AA}\\"\\n\\nJEERA_PRECOMPOSED = ZA_PRECOMPOSED + II + RA + AA_SIGN  # ज़ीरा typed one way\\nJEERA_DECOMPOSED = JA + NUKTA + II + RA + AA_SIGN  # ज़ीरा typed another way\\n\\nITEMS = [\\n    {\\"sku\\": \\"TATA-SALT-1KG\\", \\"name\\": \\"Tata Salt 1kg\\", \\"price_paise\\": 2800},\\n    {\\"sku\\": \\"HALDI-200\\", \\"name\\": \\"हल्दी पाउडर 200g\\", \\"price_paise\\": 6500},\\n    {\\"sku\\": \\"JEERA-100\\", \\"name\\": JEERA_PRECOMPOSED + \\" 100g\\", \\"price_paise\\": 5500},\\n    {\\"sku\\": \\"ATTA-5KG\\", \\"name\\": \\"आटा 5kg\\", \\"price_paise\\": 24000},\\n]\\n\\n\\ndef write_csv(path, rows, encoding, newline=\\"\\\\n\\"):\\n    lines = [\\"sku,name,price_paise\\"] + [f\\"{r[\'sku\']},{r[\'name\']},{r[\'price_paise\']}\\" for r in rows]\\n    path.write_bytes(newline.join(lines + [\\"\\"]).encode(encoding))\\n\\n\\ndef test_load_plain_utf8(tmp_path):\\n    p = tmp_path / \\"catalog.csv\\"\\n    write_csv(p, ITEMS, \\"utf-8\\")\\n    assert load_catalog(str(p)) == ITEMS\\n\\n\\ndef test_load_excel_csv_utf8_with_bom_and_crlf(tmp_path):\\n    p = tmp_path / \\"catalog.csv\\"\\n    write_csv(p, ITEMS, \\"utf-8-sig\\", newline=\\"\\\\r\\\\n\\")\\n    assert load_catalog(str(p)) == ITEMS\\n\\n\\ndef test_save_writes_utf8_and_round_trips(tmp_path):\\n    p = tmp_path / \\"out.csv\\"\\n    save_catalog(str(p), ITEMS)\\n    text = p.read_bytes().decode(\\"utf-8-sig\\")  # must be UTF-8 (a BOM is fine)\\n    assert ITEMS[1][\\"name\\"] in text\\n    assert load_catalog(str(p)) == ITEMS\\n\\n\\ndef run_in_c_locale(tmp_path, body):\\n    \\"\\"\\"Run `body` in a fresh Python whose locale encoding is ASCII (like the old billing PC).\\"\\"\\"\\n    module_dir = os.path.dirname(os.path.abspath(catalog.__file__))\\n    script = tmp_path / \\"child.py\\"\\n    script.write_text(textwrap.dedent(body), encoding=\\"utf-8\\")\\n    env = {k: v for k, v in os.environ.items() if not k.startswith((\\"LC_\\", \\"PYTHON\\")) and k != \\"LANG\\"}\\n    env.update({\\"LC_ALL\\": \\"C\\", \\"LANG\\": \\"C\\", \\"PYTHONUTF8\\": \\"0\\", \\"PYTHONCOERCECLOCALE\\": \\"0\\",\\n                \\"PYTHONPATH\\": module_dir, \\"PYTHONDONTWRITEBYTECODE\\": \\"1\\"})\\n    proc = subprocess.run([sys.executable, str(script), str(tmp_path)], cwd=module_dir, env=env,\\n                          capture_output=True, timeout=30)\\n    assert proc.returncode == 0, proc.stderr.decode(\\"utf-8\\", \\"replace\\")\\n    return json.loads(proc.stdout.decode(\\"ascii\\"))\\n\\n\\ndef test_save_and_load_on_a_non_utf8_computer(tmp_path):\\n    (tmp_path / \\"items.json\\").write_text(json.dumps(ITEMS), encoding=\\"ascii\\")\\n    back = run_in_c_locale(tmp_path, \\"\\"\\"\\n        import json, os, sys\\n        import catalog\\n        folder = sys.argv[1]\\n        with open(os.path.join(folder, \\"items.json\\"), encoding=\\"ascii\\") as f:\\n            items = json.load(f)\\n        catalog.save_catalog(os.path.join(folder, \\"c.csv\\"), items)\\n        print(json.dumps(catalog.load_catalog(os.path.join(folder, \\"c.csv\\"))))\\n    \\"\\"\\")\\n    assert back == ITEMS\\n    assert (tmp_path / \\"c.csv\\").read_bytes().decode(\\"utf-8-sig\\").count(\\"\\\\n\\") == len(ITEMS) + 1\\n\\n\\ndef test_load_excel_file_on_a_non_utf8_computer(tmp_path):\\n    write_csv(tmp_path / \\"excel.csv\\", ITEMS, \\"utf-8-sig\\", newline=\\"\\\\r\\\\n\\")\\n    back = run_in_c_locale(tmp_path, \\"\\"\\"\\n        import json, os, sys\\n        import catalog\\n        print(json.dumps(catalog.load_catalog(os.path.join(sys.argv[1], \\"excel.csv\\"))))\\n    \\"\\"\\")\\n    assert back == ITEMS\\n\\n\\n@pytest.mark.parametrize(\\"query\\", [JEERA_DECOMPOSED, JEERA_PRECOMPOSED, \\" \\" + JEERA_DECOMPOSED + \\" \\"])\\ndef test_search_finds_same_word_typed_differently(query):\\n    assert [i[\\"sku\\"] for i in search(ITEMS, query)] == [\\"JEERA-100\\"]\\n\\n\\ndef test_search_decomposed_catalog_name():\\n    items = [{\\"sku\\": \\"J\\", \\"name\\": JEERA_DECOMPOSED, \\"price_paise\\": 1}]\\n    assert search(items, JEERA_PRECOMPOSED) == items\\n\\n\\ndef test_search_english_still_case_insensitive():\\n    assert [i[\\"sku\\"] for i in search(ITEMS, \\"TATA salt\\")] == [\\"TATA-SALT-1KG\\"]\\n    assert search(ITEMS, \\"basmati\\") == []\\n\\n\\ndef check_label(name, max_bytes):\\n    label = receipt_label(name, max_bytes)\\n    assert isinstance(label, str)\\n    assert len(label.encode(\\"utf-8\\")) <= max_bytes\\n    assert name.startswith(label)\\n    if label != name:\\n        assert not unicodedata.category(name[len(label)]).startswith(\\"M\\"), (\\n            f\\"cut {name!r} at {len(label)}: the next character is a combining mark (matra)\\"\\n        )\\n    return label\\n\\n\\ndef test_label_short_names_unchanged():\\n    assert receipt_label(\\"Tata Salt 1kg\\") == \\"Tata Salt 1kg\\"\\n    assert receipt_label(ITEMS[3][\\"name\\"], 24) == ITEMS[3][\\"name\\"]\\n\\n\\ndef test_label_english_cut_by_bytes():\\n    assert receipt_label(\\"Fortune Sunflower Oil 1 Litre Pouch\\", 24) == \\"Fortune Sunflower Oil 1 \\"\\n    assert receipt_label(\\"Tata Salt\\", 4) == \\"Tata\\"\\n\\n\\ndef test_label_never_crashes_and_fits_for_every_length():\\n    for item in ITEMS:\\n        for max_bytes in range(0, 40):\\n            check_label(item[\\"name\\"], max_bytes)\\n\\n\\ndef test_label_cuts_between_letters():\\n    cheeni = \\"चीनी 1kg\\"  # चीनी 1kg\\n    assert check_label(cheeni, 7) == \\"ची\\"  # \\"ची\\": no half-encoded letter, no crash\\n\\n\\ndef test_label_keeps_matra_with_its_letter():\\n    atta = \\"आटा\\"  # आटा: the last 3 bytes are the matra of ट\\n    assert check_label(atta, 7) == \\"आ\\"\\n    assert check_label(atta, 9) == atta\\n", "id": "24-hindi-catalog-unicode", "meta": {"difficulty": 3, "files": ["catalog.py"], "kind": "bugfix", "timeout": 60}, "reference": {"catalog.py": "\\"\\"\\"Product catalog helpers for the billing counter.\\n\\nThe catalog is a CSV file with the columns sku,name,price_paise. Names can be in English or Hindi.\\nFiles are always UTF-8 (a BOM written by Excel is accepted), whatever the computer\'s locale.\\n\\"\\"\\"\\n\\nimport csv\\nimport unicodedata\\n\\nFIELDS = [\\"sku\\", \\"name\\", \\"price_paise\\"]\\n\\n\\ndef load_catalog(path):\\n    \\"\\"\\"Read the catalog CSV and return a list of dicts (price_paise as int).\\"\\"\\"\\n    items = []\\n    # utf-8-sig also strips the byte-order mark that Excel puts at the start of \\"CSV UTF-8\\" files\\n    with open(path, newline=\\"\\", encoding=\\"utf-8-sig\\") as f:\\n        for row in csv.DictReader(f):\\n            items.append({\\"sku\\": row[\\"sku\\"], \\"name\\": row[\\"name\\"], \\"price_paise\\": int(row[\\"price_paise\\"])})\\n    return items\\n\\n\\ndef save_catalog(path, items):\\n    \\"\\"\\"Write the catalog CSV (same columns as load_catalog reads).\\"\\"\\"\\n    with open(path, \\"w\\", newline=\\"\\", encoding=\\"utf-8\\") as f:\\n        writer = csv.DictWriter(f, fieldnames=FIELDS)\\n        writer.writeheader()\\n        for item in items:\\n            writer.writerow({k: item[k] for k in FIELDS})\\n\\n\\ndef _fold(text):\\n    # The same letter can be typed as different code points (e.g. ज़ as U+095B or as ज + nukta U+093C);\\n    # normalizing both sides to one form makes them compare equal.\\n    return unicodedata.normalize(\\"NFD\\", text).casefold()\\n\\n\\ndef search(items, query):\\n    \\"\\"\\"Products whose name contains `query`, ignoring case and Unicode spelling differences.\\"\\"\\"\\n    q = _fold(query.strip())\\n    return [item for item in items if q in _fold(item[\\"name\\"])]\\n\\n\\ndef receipt_label(name, max_bytes=24):\\n    \\"\\"\\"The product name as printed on the thermal printer, which takes at most `max_bytes` UTF-8 bytes.\\n\\n    Cuts on character boundaries and never separates a vowel sign (matra) or other combining\\n    mark from the letter it belongs to.\\n    \\"\\"\\"\\n    if max_bytes < 0:\\n        raise ValueError(\\"max_bytes must not be negative\\")\\n    if len(name.encode(\\"utf-8\\")) <= max_bytes:\\n        return name\\n    end = 0\\n    used = 0\\n    for i, ch in enumerate(name):\\n        used += len(ch.encode(\\"utf-8\\"))\\n        if used > max_bytes:\\n            break\\n        end = i + 1\\n    while 0 < end < len(name) and unicodedata.category(name[end]).startswith(\\"M\\"):\\n        end -= 1\\n    return name[:end]\\n"}, "starter": {"catalog.py": "\\"\\"\\"Product catalog helpers for the billing counter.\\n\\nThe catalog is a CSV file with the columns sku,name,price_paise. Names can be in English or Hindi.\\n\\"\\"\\"\\n\\nimport csv\\n\\nFIELDS = [\\"sku\\", \\"name\\", \\"price_paise\\"]\\n\\n\\ndef load_catalog(path):\\n    \\"\\"\\"Read the catalog CSV and return a list of dicts (price_paise as int).\\"\\"\\"\\n    items = []\\n    with open(path, newline=\\"\\") as f:\\n        for row in csv.DictReader(f):\\n            items.append({\\"sku\\": row[\\"sku\\"], \\"name\\": row[\\"name\\"], \\"price_paise\\": int(row[\\"price_paise\\"])})\\n    return items\\n\\n\\ndef save_catalog(path, items):\\n    \\"\\"\\"Write the catalog CSV (same columns as load_catalog reads).\\"\\"\\"\\n    with open(path, \\"w\\", newline=\\"\\") as f:\\n        writer = csv.DictWriter(f, fieldnames=FIELDS)\\n        writer.writeheader()\\n        for item in items:\\n            writer.writerow({k: item[k] for k in FIELDS})\\n\\n\\ndef search(items, query):\\n    \\"\\"\\"Products whose name contains `query`, ignoring case.\\"\\"\\"\\n    q = query.strip().lower()\\n    return [item for item in items if q in item[\\"name\\"].lower()]\\n\\n\\ndef receipt_label(name, max_bytes=24):\\n    \\"\\"\\"The product name as printed on the thermal printer, which takes at most `max_bytes` UTF-8 bytes.\\"\\"\\"\\n    data = name.encode(\\"utf-8\\")\\n    if len(data) <= max_bytes:\\n        return name\\n    return data[:max_bytes].decode(\\"utf-8\\")\\n"}, "task_md": "Since we started entering product names in Hindi, `catalog.py` has been giving us trouble.\\nThe catalog my nephew saved from Excel as \\"CSV UTF-8\\" fails to load with `KeyError: \'sku\'`, and on the old billing PC (its system language isn\'t set to UTF-8) saving the catalog crashes with `UnicodeEncodeError`.\\nA customer searched for ज़ीरा on the phone and got nothing, although ज़ीरा (cumin) is right there in the catalog.\\nAnd `receipt_label` sometimes crashes with `UnicodeDecodeError` for Hindi names; when it doesn\'t crash, the printer sometimes shows a letter with its matra chopped off.\\nPlease fix these; the other code calls `load_catalog`, `save_catalog`, `search` and `receipt_label` exactly as they are now, and English names must keep working the same way."}'),
    json.loads('{"hidden": "import random\\nimport time\\nfrom datetime import date, timedelta\\n\\nfrom dispatch import Order, dispatch_queue\\n\\nD1 = date(2024, 11, 1)\\nD2 = date(2024, 11, 2)\\nD3 = date(2024, 11, 3)\\n\\n\\ndef ids(orders):\\n    return [o.order_id for o in orders]\\n\\n\\ndef test_express_first_then_by_date():\\n    orders = [\\n        Order(\\"n-d3\\", False, D3),\\n        Order(\\"e-d2\\", True, D2),\\n        Order(\\"n-d1\\", False, D1),\\n        Order(\\"e-d1\\", True, D1),\\n    ]\\n    assert ids(dispatch_queue(orders)) == [\\"e-d1\\", \\"e-d2\\", \\"n-d1\\", \\"n-d3\\"]\\n\\n\\ndef test_same_day_is_first_come_first_served():\\n    orders = [Order(f\\"o{i:02d}\\", i % 3 == 0, D1) for i in range(20)]\\n    result = ids(dispatch_queue(orders))\\n    express = [f\\"o{i:02d}\\" for i in range(20) if i % 3 == 0]\\n    normal = [f\\"o{i:02d}\\" for i in range(20) if i % 3 != 0]\\n    assert result == express + normal\\n\\n\\ndef test_no_promised_date_goes_last_in_its_group_in_placed_order():\\n    orders = [\\n        Order(\\"pickup-1\\", False, None),\\n        Order(\\"n-d2\\", False, D2),\\n        Order(\\"e-pickup\\", True, None),\\n        Order(\\"pickup-2\\", False, None),\\n        Order(\\"e-d3\\", True, D3),\\n        Order(\\"n-d1\\", False, D1),\\n    ]\\n    assert ids(dispatch_queue(orders)) == [\\"e-d3\\", \\"e-pickup\\", \\"n-d1\\", \\"n-d2\\", \\"pickup-1\\", \\"pickup-2\\"]\\n\\n\\ndef test_does_not_reorder_the_callers_list():\\n    orders = [Order(\\"a\\", False, D3), Order(\\"b\\", True, D1), Order(\\"c\\", False, D1)]\\n    original = list(orders)\\n    result = dispatch_queue(orders)\\n    assert orders == original\\n    assert [o.order_id for o in orders] == [\\"a\\", \\"b\\", \\"c\\"]\\n    assert ids(result) == [\\"b\\", \\"c\\", \\"a\\"]\\n    result.append(Order(\\"x\\", False, None))\\n    assert len(orders) == 3\\n\\n\\ndef test_empty_and_single():\\n    assert list(dispatch_queue([])) == []\\n    one = [Order(\\"only\\", False, None)]\\n    assert ids(dispatch_queue(one)) == [\\"only\\"]\\n\\n\\ndef test_matches_stable_reference_on_random_data():\\n    rng = random.Random(42)\\n    days = [D1 + timedelta(days=k) for k in range(4)] + [None]\\n    for _ in range(50):\\n        orders = [Order(f\\"o{i}\\", rng.random() < 0.3, rng.choice(days)) for i in range(60)]\\n        expected = sorted(\\n            orders, key=lambda o: (not o.express, o.promised_by is None, o.promised_by or date.min)\\n        )\\n        assert ids(dispatch_queue(orders)) == ids(expected)\\n\\n\\ndef test_big_queue_is_fast():\\n    rng = random.Random(1)\\n    days = [D1 + timedelta(days=k) for k in range(30)] + [None]\\n    orders = [Order(f\\"o{i}\\", rng.random() < 0.2, rng.choice(days)) for i in range(100_000)]\\n    start = time.perf_counter()\\n    result = dispatch_queue(orders)\\n    assert time.perf_counter() - start < 2\\n    assert len(result) == 100_000\\n\\n\\ndef test_running_twice_gives_the_same_queue():\\n    orders = [Order(f\\"o{i}\\", i % 4 == 0, [D2, D1, None][i % 3]) for i in range(30)]\\n    first = dispatch_queue(orders)\\n    assert ids(dispatch_queue(orders)) == ids(first)\\n    assert ids(dispatch_queue(list(first))) == ids(first)\\n\\n\\ndef test_express_pickup_still_before_normal_dated_orders():\\n    orders = [Order(\\"n-d1\\", False, D1), Order(\\"e-none\\", True, None)]\\n    assert ids(dispatch_queue(orders)) == [\\"e-none\\", \\"n-d1\\"]\\n", "id": "25-dispatch-sort-order", "meta": {"difficulty": 1, "files": ["dispatch.py"], "kind": "bugfix", "timeout": 60}, "reference": {"dispatch.py": "\\"\\"\\"Which order the packers should work on next.\\"\\"\\"\\n\\nfrom dataclasses import dataclass\\nfrom datetime import date\\n\\n\\n@dataclass\\nclass Order:\\n    order_id: str\\n    express: bool\\n    promised_by: date | None  # None = no promised date (e.g. pickup at the counter)\\n\\n\\ndef _dispatch_key(order):\\n    # express first; then dated orders by date; undated last. sorted() is stable, so\\n    # orders with the same key keep the order they were placed in (first come, first served).\\n    return (not order.express, order.promised_by is None, order.promised_by or date.min)\\n\\n\\ndef dispatch_queue(orders):\\n    \\"\\"\\"Return the orders in the sequence the packers should work on them.\\n\\n    `orders` is the list of open orders in the order they were placed. It is not modified.\\n    \\"\\"\\"\\n    return sorted(orders, key=_dispatch_key)\\n"}, "starter": {"dispatch.py": "\\"\\"\\"Which order the packers should work on next.\\"\\"\\"\\n\\nfrom dataclasses import dataclass\\nfrom datetime import date\\nfrom functools import cmp_to_key\\n\\n\\n@dataclass\\nclass Order:\\n    order_id: str\\n    express: bool\\n    promised_by: date | None  # None = no promised date (e.g. pickup at the counter)\\n\\n\\ndef _compare(a, b):\\n    # express orders first\\n    if a.express != b.express:\\n        return -1 if a.express else 1\\n    # then the earliest promised date first\\n    return -1 if a.promised_by <= b.promised_by else 1\\n\\n\\ndef dispatch_queue(orders):\\n    \\"\\"\\"Return the orders in the sequence the packers should work on them.\\n\\n    `orders` is the list of open orders in the order they were placed.\\n    \\"\\"\\"\\n    orders.sort(key=cmp_to_key(_compare))\\n    return orders\\n"}, "task_md": "The packers say the dispatch screen is wrong: among orders promised for the same day, the newest orders show up first, but it should be first come, first served.\\nOrders without a promised date (counter pickups) crash the screen with a `TypeError`; they should come after the dated orders of their group (express or normal).\\nAlso, ever since the dispatch screen went live, the \\"orders placed today\\" list on the counter is shown in a strange order, and both screens get their orders from the same list.\\nThe code is `dispatch_queue(orders)` in `dispatch.py`; please fix it without changing how it\'s called."}'),
    json.loads('{"hidden": "import base64\\nimport json\\nimport os\\nimport pickle\\nimport time\\n\\nimport pytest\\n\\nfrom cart_session import MAX_QTY, add_item, load_cart, parse_qty, save_cart\\n\\nCOOKIE_UNSAFE = set(\' \\\\t\\\\r\\\\n;,\\"\\\\\\\\\')\\n\\n\\nclass Evil:\\n    def __init__(self, path):\\n        self.path = path\\n\\n    def __reduce__(self):\\n        return (os.mkdir, (self.path,))\\n\\n\\ndef encodings_of(obj):\\n    \\"\\"\\"The same payload in the ways a cookie could plausibly be encoded.\\"\\"\\"\\n    raw = json.dumps(obj).encode(\\"utf-8\\")\\n    return [\\n        base64.urlsafe_b64encode(raw).decode(\\"ascii\\"),\\n        base64.b64encode(raw).decode(\\"ascii\\"),\\n        raw.decode(\\"utf-8\\"),\\n    ]\\n\\n\\ndef test_round_trip():\\n    cart = {\\"ATTA-5KG\\": 2, \\"TOOR_DAL.1KG\\": 1, \\"MAGGI-12PK\\": MAX_QTY}\\n    cookie = save_cart(cart)\\n    assert isinstance(cookie, str)\\n    assert load_cart(cookie) == cart\\n\\n\\ndef test_cookie_value_is_cookie_safe():\\n    cookie = save_cart({\\"ATTA-5KG\\": 2, \\"SUGAR-1KG\\": 3})\\n    assert not (set(cookie) & COOKIE_UNSAFE)\\n    assert cookie.isascii()\\n\\n\\ndef test_empty_cookie_is_empty_cart():\\n    assert load_cart(\\"\\") == {}\\n    assert load_cart(save_cart({})) == {}\\n\\n\\ndef test_old_pickle_cookie_is_rejected_without_running_code(tmp_path):\\n    marker = tmp_path / \\"pwned\\"\\n    cookie = base64.urlsafe_b64encode(pickle.dumps(Evil(str(marker)))).decode(\\"ascii\\")\\n    with pytest.raises(ValueError):\\n        load_cart(cookie)\\n    assert not marker.exists()\\n\\n\\ndef test_plain_pickled_cart_is_rejected():\\n    cookie = base64.urlsafe_b64encode(pickle.dumps({\\"ATTA-5KG\\": 2})).decode(\\"ascii\\")\\n    with pytest.raises(ValueError):\\n        load_cart(cookie)\\n\\n\\n@pytest.mark.parametrize(\\"cookie\\", [\\"!!!\\", \\"abc\\", \\"e30\\", \\"%7B%7D\\", \\"a\\" * 10, \\"=\\" * 8, \\"अटा\\"])\\ndef test_garbage_cookie_raises_value_error(cookie):\\n    with pytest.raises(ValueError):\\n        load_cart(cookie)\\n\\n\\n@pytest.mark.parametrize(\\n    \\"bad\\",\\n    [\\n        [1, 2, 3],\\n        \\"ATTA-5KG\\",\\n        42,\\n        None,\\n        {\\"ATTA-5KG\\": -5},\\n        {\\"ATTA-5KG\\": 0},\\n        {\\"ATTA-5KG\\": MAX_QTY + 1},\\n        {\\"ATTA-5KG\\": 1_000_000_000},\\n        {\\"ATTA-5KG\\": \\"2\\"},\\n        {\\"ATTA-5KG\\": 1.5},\\n        {\\"ATTA-5KG\\": True},\\n        {\\"ATTA-5KG\\": None},\\n        {\\"ATTA-5KG\\": {\\"nested\\": 1}},\\n    ],\\n)\\ndef test_tampered_cart_never_loads(bad):\\n    for cookie in encodings_of(bad):\\n        with pytest.raises(ValueError):\\n            load_cart(cookie)\\n    try:\\n        cookie = save_cart(bad)\\n    except (ValueError, TypeError):\\n        return\\n    with pytest.raises(ValueError):\\n        load_cart(cookie)\\n\\n\\n@pytest.mark.parametrize(\\"text,expected\\", [(\\"3\\", 3), (\\"2*12\\", 24), (\\" 2 * 6 \\", 12), (\\"99\\", 99), (\\"1\\", 1), (\\"12*2\\", 24)])\\ndef test_parse_qty_normal_input(text, expected):\\n    assert parse_qty(text) == expected\\n\\n\\n@pytest.mark.parametrize(\\"text\\", [\\"0\\", \\"100\\", \\"-1\\", \\"abc\\", \\"\\", \\"  \\", \\"2*\\", \\"*3\\", \\"2**3\\", \\"50*2\\", \\"1.5\\", \\"2*0\\",\\n                                  \\"True\\", \\"[5]\\", \\"0x10\\"])\\ndef test_parse_qty_rejects_bad_input(text):\\n    with pytest.raises(ValueError):\\n        parse_qty(text)\\n\\n\\ndef test_parse_qty_does_not_run_code(tmp_path):\\n    marker = tmp_path / \\"pwned\\"\\n    for attack in (\\n        f\\"__import__(\'os\').mkdir({str(marker)!r})\\",\\n        f\\"[__import__(\'os\').mkdir({str(marker)!r}), 1][1]\\",\\n        f\\"(lambda: __import__(\'os\').mkdir({str(marker)!r}) or 1)()\\",\\n    ):\\n        with pytest.raises(ValueError):\\n            parse_qty(attack)\\n    assert not marker.exists()\\n\\n\\ndef test_parse_qty_huge_expressions_are_rejected_quickly():\\n    start = time.perf_counter()\\n    for attack in (\\"9\\" * 5000, \\"99999999*99999999\\", \\"9**999999\\"):\\n        with pytest.raises(ValueError):\\n            parse_qty(attack)\\n    assert time.perf_counter() - start < 1\\n\\n\\ndef test_add_item_still_works_and_caps():\\n    cart = add_item({}, \\"ATTA-5KG\\", \\"2\\")\\n    cart = add_item(cart, \\"ATTA-5KG\\", \\"2*3\\")\\n    assert cart == {\\"ATTA-5KG\\": 8}\\n    with pytest.raises(ValueError):\\n        add_item({\\"ATTA-5KG\\": 98}, \\"ATTA-5KG\\", \\"2\\")\\n    assert load_cart(save_cart(cart)) == cart\\n", "id": "26-cart-cookie-review", "meta": {"difficulty": 2, "files": ["cart_session.py"], "kind": "review", "timeout": 60}, "reference": {"cart_session.py": "\\"\\"\\"Shopping cart for the online order page.\\n\\nThe cart lives in the customer\'s browser as a cookie, so the server stays stateless.\\nA cart is a dict {sku: qty}.\\n\\nEverything that comes from the browser is untrusted: the cookie is plain JSON in URL-safe\\nbase64 (never pickle) and is validated on the way in, and typed quantities are parsed with a\\ntiny grammar (never eval).\\n\\"\\"\\"\\n\\nimport base64\\nimport binascii\\nimport json\\nimport re\\n\\nMAX_QTY = 99\\nMAX_ITEMS = 200\\nMAX_COOKIE_LENGTH = 4096\\n_SKU = re.compile(r\\"[A-Za-z0-9][A-Za-z0-9_.\\\\-]{0,63}\\")\\n_QTY = re.compile(r\\"\\\\s*(\\\\d{1,3})\\\\s*(?:\\\\*\\\\s*(\\\\d{1,3})\\\\s*)?\\")\\n\\n\\ndef _validate_cart(cart):\\n    if not isinstance(cart, dict):\\n        raise TypeError(\\"cart must be a mapping of sku to quantity\\")\\n    if len(cart) > MAX_ITEMS:\\n        raise ValueError(f\\"at most {MAX_ITEMS} different items\\")\\n    clean = {}\\n    for sku, qty in cart.items():\\n        if not isinstance(sku, str) or not _SKU.fullmatch(sku):\\n            raise ValueError(f\\"bad sku: {sku!r}\\")\\n        if isinstance(qty, bool) or not isinstance(qty, int) or not 1 <= qty <= MAX_QTY:\\n            raise ValueError(f\\"bad quantity for {sku}\\")\\n        clean[sku] = qty\\n    return clean\\n\\n\\ndef save_cart(cart):\\n    \\"\\"\\"Turn a cart into the cookie value we send to the browser.\\"\\"\\"\\n    data = json.dumps(_validate_cart(cart), separators=(\\",\\", \\":\\"), sort_keys=True).encode(\\"utf-8\\")\\n    return base64.urlsafe_b64encode(data).decode(\\"ascii\\")\\n\\n\\ndef load_cart(cookie):\\n    \\"\\"\\"Turn the cookie value from the browser back into a cart. Raises ValueError for a bad cookie.\\"\\"\\"\\n    if not cookie:\\n        return {}\\n    if not isinstance(cookie, str) or len(cookie) > MAX_COOKIE_LENGTH:\\n        raise ValueError(\\"bad cart cookie\\")\\n    try:\\n        raw = base64.urlsafe_b64decode(cookie.encode(\\"ascii\\"))\\n        cart = json.loads(raw.decode(\\"utf-8\\"))\\n    except (ValueError, binascii.Error, UnicodeError):\\n        raise ValueError(\\"bad cart cookie\\") from None\\n    try:\\n        return _validate_cart(cart)\\n    except (TypeError, ValueError):\\n        raise ValueError(\\"bad cart cookie\\") from None\\n\\n\\ndef parse_qty(text):\\n    \\"\\"\\"Quantity typed by the customer: a number, or a product like \\"2*12\\" for two dozen.\\"\\"\\"\\n    if not isinstance(text, str):\\n        raise TypeError(\\"quantity must be text\\")\\n    match = _QTY.fullmatch(text)\\n    if not match:\\n        raise ValueError(f\\"quantity must be a number like 3 or 2*12, between 1 and {MAX_QTY}\\")\\n    qty = int(match.group(1)) * int(match.group(2) or 1)\\n    if not 1 <= qty <= MAX_QTY:\\n        raise ValueError(f\\"quantity must be between 1 and {MAX_QTY}\\")\\n    return qty\\n\\n\\ndef add_item(cart, sku, qty_text):\\n    \\"\\"\\"Return a new cart with the typed quantity of `sku` added.\\"\\"\\"\\n    qty = parse_qty(qty_text)\\n    new_cart = dict(cart)\\n    new_cart[sku] = new_cart.get(sku, 0) + qty\\n    if new_cart[sku] > MAX_QTY:\\n        raise ValueError(f\\"at most {MAX_QTY} of one item\\")\\n    return new_cart\\n"}, "starter": {"cart_session.py": "\\"\\"\\"Shopping cart for the online order page.\\n\\nThe cart lives in the customer\'s browser as a cookie, so the server stays stateless.\\nA cart is a dict {sku: qty}.\\n\\"\\"\\"\\n\\nimport base64\\nimport pickle\\n\\nMAX_QTY = 99\\n\\n\\ndef save_cart(cart):\\n    \\"\\"\\"Turn a cart into the cookie value we send to the browser.\\"\\"\\"\\n    return base64.urlsafe_b64encode(pickle.dumps(cart)).decode(\\"ascii\\")\\n\\n\\ndef load_cart(cookie):\\n    \\"\\"\\"Turn the cookie value from the browser back into a cart. Raises ValueError for a bad cookie.\\"\\"\\"\\n    if not cookie:\\n        return {}\\n    try:\\n        return pickle.loads(base64.urlsafe_b64decode(cookie))\\n    except Exception:\\n        raise ValueError(\\"bad cart cookie\\")\\n\\n\\ndef parse_qty(text):\\n    \\"\\"\\"Quantity typed by the customer: a number, or a product like \\"2*12\\" for two dozen.\\"\\"\\"\\n    qty = eval(text)\\n    if not isinstance(qty, int) or qty < 1 or qty > MAX_QTY:\\n        raise ValueError(f\\"quantity must be between 1 and {MAX_QTY}\\")\\n    return qty\\n\\n\\ndef add_item(cart, sku, qty_text):\\n    \\"\\"\\"Return a new cart with the typed quantity of `sku` added.\\"\\"\\"\\n    qty = parse_qty(qty_text)\\n    new_cart = dict(cart)\\n    new_cart[sku] = new_cart.get(sku, 0) + qty\\n    if new_cart[sku] > MAX_QTY:\\n        raise ValueError(f\\"at most {MAX_QTY} of one item\\")\\n    return new_cart\\n"}, "task_md": "We\'re about to launch the online order page for our shop, and `cart_session.py` is the part that keeps the customer\'s cart in a browser cookie and reads the quantity they type into the box.\\nA friend wrote it and it works in our testing, but nobody has reviewed it.\\nPlease review it the way you would before something goes live on the public internet, and fix whatever you wouldn\'t let ship.\\nKeep the functions and how they are called (`save_cart`, `load_cart`, `parse_qty`, `add_item`, `MAX_QTY`); normal customers shouldn\'t notice a difference, and typing `2*12` for two dozen must keep working.\\nIt\'s fine if carts saved by the old version stop loading after the launch."}'),
    json.loads('{"hidden": "import logging\\nimport re\\nimport threading\\n\\nimport pytest\\n\\nimport payments\\nfrom payments import PaymentError, charge_card, recent_charges\\n\\nCARD = \\"4111111111111111\\"\\nCVV = \\"846\\"\\nEXPIRY = \\"12/27\\"\\nPHONE = \\"9876543210\\"\\nAPI_KEY = \\"sk_live_51HxQaZ0secretKEY\\"\\nORDER = \\"ORD-2024-0001\\"\\n\\n\\nclass FakeGateway:\\n    url = \\"https://pay.example-gateway.in/v1/charges\\"\\n    api_key = API_KEY\\n\\n    def __init__(self, fail=False):\\n        self.fail = fail\\n        self.requests = []\\n        self.lock = threading.Lock()\\n\\n    def charge(self, request):\\n        with self.lock:\\n            self.requests.append(dict(request))\\n            n = len(self.requests)\\n        if self.fail:\\n            raise RuntimeError(\\"card declined: insufficient funds\\")\\n        return {\\"txn_id\\": f\\"TXN-{n:05d}\\"}\\n\\n\\ndef leaked(text):\\n    found = []\\n    if CARD in re.sub(r\\"[ -]\\", \\"\\", text):\\n        found.append(\\"card number\\")\\n    if re.search(rf\\"(?<!\\\\d){CVV}(?!\\\\d)\\", text):\\n        found.append(\\"cvv\\")\\n    if PHONE in text:\\n        found.append(\\"phone\\")\\n    if API_KEY in text:\\n        found.append(\\"api key\\")\\n    return found\\n\\n\\ndef all_log_text(caplog):\\n    parts = []\\n    for record in caplog.records:\\n        parts.append(record.getMessage())\\n        if record.exc_info:\\n            parts.append(logging.Formatter().formatException(record.exc_info))\\n        if record.exc_text:\\n            parts.append(record.exc_text)\\n        if record.stack_info:\\n            parts.append(record.stack_info)\\n    return \\"\\\\n\\".join(parts) + \\"\\\\n\\" + caplog.text\\n\\n\\ndef charge(gateway, order_id=ORDER, amount=129900):\\n    return charge_card(gateway, order_id, amount, CARD, EXPIRY, CVV, PHONE)\\n\\n\\ndef test_successful_charge_returns_txn_and_gateway_gets_full_details():\\n    gw = FakeGateway()\\n    txn = charge(gw)\\n    assert txn == \\"TXN-00001\\"\\n    sent = gw.requests[0]\\n    assert CARD in sent.values()\\n    assert CVV in sent.values()\\n    assert 129900 in sent.values()\\n\\n\\ndef test_success_logs_contain_no_secrets(caplog):\\n    caplog.set_level(logging.DEBUG)\\n    charge(FakeGateway())\\n    text = all_log_text(caplog)\\n    assert leaked(text) == []\\n    assert ORDER in text, \\"support must still find the charge by order id in the logs\\"\\n\\n\\ndef test_failure_raises_payment_error_without_secrets(caplog):\\n    caplog.set_level(logging.DEBUG)\\n    with pytest.raises(PaymentError) as excinfo:\\n        charge(FakeGateway(fail=True), order_id=\\"ORD-2024-0002\\")\\n    message = str(excinfo.value) + repr(excinfo.value) + repr(excinfo.value.args)\\n    assert leaked(message) == []\\n    text = all_log_text(caplog)\\n    assert leaked(text) == []\\n    assert \\"ORD-2024-0002\\" in text\\n\\n\\ndef test_card_with_spaces_is_not_logged_either(caplog):\\n    caplog.set_level(logging.DEBUG)\\n    charge_card(FakeGateway(), \\"ORD-2024-0003\\", 5000, \\"4111 1111 1111 1111\\", EXPIRY, CVV, \\"+91 98765 43210\\")\\n    text = all_log_text(caplog)\\n    assert \\"4111 1111 1111 1111\\" not in text\\n    assert CARD not in text.replace(\\" \\", \\"\\")\\n    assert re.search(rf\\"(?<!\\\\d){CVV}(?!\\\\d)\\", text) is None\\n    assert \\"98765 43210\\" not in text and PHONE not in text.replace(\\" \\", \\"\\")\\n\\n\\ndef test_dashboard_has_no_card_or_cvv_but_has_order_id():\\n    charge(FakeGateway(), order_id=\\"ORD-2024-0004\\")\\n    with pytest.raises(PaymentError):\\n        charge(FakeGateway(fail=True), order_id=\\"ORD-2024-0005\\")\\n    shown = repr(recent_charges())\\n    assert CARD not in shown\\n    assert re.search(rf\\"(?<!\\\\d){CVV}(?!\\\\d)\\", shown) is None\\n    assert \\"ORD-2024-0004\\" in shown\\n\\n\\ndef test_dashboard_history_does_not_grow_forever():\\n    gw = FakeGateway()\\n    for i in range(5000):\\n        charge(gw, order_id=f\\"BULK-{i}\\", amount=100)\\n    recent = recent_charges()\\n    assert len(recent) < 5000\\n    assert \\"BULK-4999\\" in repr(recent)  # the newest are kept\\n\\n\\ndef test_dashboard_list_is_a_copy():\\n    charge(FakeGateway(), order_id=\\"ORD-2024-0006\\")\\n    before = len(recent_charges())\\n    recent_charges().clear()\\n    assert len(recent_charges()) == before\\n\\n\\n@pytest.mark.parametrize(\\"amount\\", [0, -100])\\ndef test_non_positive_amount_rejected_before_calling_gateway(amount):\\n    gw = FakeGateway()\\n    with pytest.raises(ValueError):\\n        charge(gw, amount=amount)\\n    assert gw.requests == []\\n\\n\\n@pytest.mark.parametrize(\\"amount\\", [1299.5, \\"129900\\"])\\ndef test_non_integer_amount_rejected(amount):\\n    gw = FakeGateway()\\n    with pytest.raises((TypeError, ValueError)):\\n        charge(gw, amount=amount)\\n    assert gw.requests == []\\n\\n\\ndef test_concurrent_charges_all_succeed():\\n    gw = FakeGateway()\\n    barrier = threading.Barrier(8)\\n    txns, lock = [], threading.Lock()\\n\\n    def worker(n):\\n        barrier.wait(timeout=5)\\n        for i in range(50):\\n            txn = charge(gw, order_id=f\\"T{n}-{i}\\", amount=100)\\n            with lock:\\n                txns.append(txn)\\n\\n    threads = [threading.Thread(target=worker, args=(n,)) for n in range(8)]\\n    for t in threads:\\n        t.start()\\n    for t in threads:\\n        t.join(timeout=10)\\n    assert len(txns) == 400 and len(set(txns)) == 400\\n    recent_charges()\\n\\n\\ndef test_module_has_payment_error():\\n    assert issubclass(payments.PaymentError, Exception)\\n", "id": "27-payment-logs-review", "meta": {"difficulty": 2, "files": ["payments.py"], "kind": "review", "timeout": 60}, "reference": {"payments.py": "\\"\\"\\"Card payments for the online order page, through our payment gateway.\\n\\nCard numbers, CVVs, phone numbers and the gateway key never reach logs, exception messages\\nor the support dashboard: only masked values do. The dashboard history is bounded.\\n\\"\\"\\"\\n\\nimport logging\\nimport threading\\nfrom collections import deque\\n\\nlog = logging.getLogger(\\"shop.payments\\")\\n\\nRECENT_LIMIT = 200\\n_recent = deque(maxlen=RECENT_LIMIT)  # shown on the support dashboard\\n_recent_lock = threading.Lock()\\n\\n\\nclass PaymentError(Exception):\\n    pass\\n\\n\\ndef mask_card(card_number):\\n    digits = \\"\\".join(ch for ch in str(card_number) if ch.isdigit())\\n    return f\\"**** **** **** {digits[-4:]}\\" if len(digits) >= 4 else \\"****\\"\\n\\n\\ndef mask_phone(phone):\\n    digits = \\"\\".join(ch for ch in str(phone) if ch.isdigit())\\n    return f\\"******{digits[-4:]}\\" if len(digits) >= 4 else \\"****\\"\\n\\n\\ndef charge_card(gateway, order_id, amount_paise, card_number, expiry, cvv, phone):\\n    \\"\\"\\"Charge a card through `gateway` and return the gateway\'s transaction id.\\n\\n    `gateway` has `.url`, `.api_key` and `.charge(request: dict) -> {\\"txn_id\\": ...}`, and raises on failure.\\n    Raises PaymentError if the charge fails.\\n    \\"\\"\\"\\n    if isinstance(amount_paise, bool) or not isinstance(amount_paise, int):\\n        raise TypeError(\\"amount_paise must be an integer number of paise\\")\\n    if amount_paise <= 0:\\n        raise ValueError(\\"amount must be positive\\")\\n    request = {\\n        \\"order_id\\": order_id,\\n        \\"amount_paise\\": amount_paise,\\n        \\"card_number\\": card_number,\\n        \\"expiry\\": expiry,\\n        \\"cvv\\": cvv,\\n        \\"phone\\": phone,\\n    }\\n    safe = {\\n        \\"order_id\\": order_id,\\n        \\"amount_paise\\": amount_paise,\\n        \\"card\\": mask_card(card_number),\\n        \\"phone\\": mask_phone(phone),\\n    }\\n    log.info(\\"charging order %s: amount_paise=%s card=%s\\", order_id, amount_paise, safe[\\"card\\"])\\n    log.debug(\\"POST %s\\", gateway.url)\\n    try:\\n        response = gateway.charge(request)\\n    except Exception as exc:  # noqa: BLE001 - any gateway failure becomes a PaymentError\\n        # the gateway\'s own message may echo request fields, so only log the error type\\n        log.error(\\"charge failed for order %s (%s)\\", order_id, type(exc).__name__)\\n        with _recent_lock:\\n            _recent.append({**safe, \\"status\\": \\"failed\\"})\\n        raise PaymentError(f\\"payment failed for order {order_id}\\") from None\\n    txn_id = response[\\"txn_id\\"]\\n    with _recent_lock:\\n        _recent.append({**safe, \\"status\\": \\"charged\\", \\"txn_id\\": txn_id})\\n    log.info(\\"charged order %s: txn %s\\", order_id, txn_id)\\n    return txn_id\\n\\n\\ndef recent_charges():\\n    \\"\\"\\"The latest charges, for the support dashboard (masked).\\"\\"\\"\\n    with _recent_lock:\\n        return [dict(item) for item in _recent]\\n"}, "starter": {"payments.py": "\\"\\"\\"Card payments for the online order page, through our payment gateway.\\"\\"\\"\\n\\nimport logging\\n\\nlog = logging.getLogger(\\"shop.payments\\")\\n\\n_recent = []  # shown on the support dashboard\\n\\n\\nclass PaymentError(Exception):\\n    pass\\n\\n\\ndef charge_card(gateway, order_id, amount_paise, card_number, expiry, cvv, phone):\\n    \\"\\"\\"Charge a card through `gateway` and return the gateway\'s transaction id.\\n\\n    `gateway` has `.url`, `.api_key` and `.charge(request: dict) -> {\\"txn_id\\": ...}`, and raises on failure.\\n    Raises PaymentError if the charge fails.\\n    \\"\\"\\"\\n    if amount_paise <= 0:\\n        raise ValueError(\\"amount must be positive\\")\\n    request = {\\n        \\"order_id\\": order_id,\\n        \\"amount_paise\\": amount_paise,\\n        \\"card_number\\": card_number,\\n        \\"expiry\\": expiry,\\n        \\"cvv\\": cvv,\\n        \\"phone\\": phone,\\n    }\\n    log.info(\\"charging order %s: %s\\", order_id, request)\\n    log.debug(\\"POST %s Authorization: Bearer %s\\", gateway.url, gateway.api_key)\\n    _recent.append(request)\\n    try:\\n        response = gateway.charge(request)\\n    except Exception as exc:\\n        log.error(\\"charge failed for %s: %s\\", request, exc)\\n        raise PaymentError(f\\"payment failed for {request}: {exc}\\")\\n    log.info(\\"charged order %s (customer phone %s): txn %s\\", order_id, phone, response[\\"txn_id\\"])\\n    return response[\\"txn_id\\"]\\n\\n\\ndef recent_charges():\\n    \\"\\"\\"The latest charges, for the support dashboard.\\"\\"\\"\\n    return list(_recent)\\n"}, "task_md": "Next week we switch on card payments for online orders. The code is in `payments.py` and it works against the gateway\'s sandbox.\\nOur logs are shipped to a third-party log service that the whole team and two freelancers can read, and the support dashboard shows `recent_charges()`.\\nPlease do a production review of `payments.py` and fix what you find. The checkout page calls `charge_card(gateway, order_id, amount_paise, card_number, expiry, cvv, phone)` and expects a transaction id back or a `PaymentError`, and support still needs to be able to find a charge by its order id, so keep that working.\\nThe server runs for months without a restart."}'),
    json.loads('{"hidden": "import os\\nimport random\\nimport time\\nimport tracemalloc\\n\\nimport pytest\\n\\nfrom sales_report import summarize, top_skus\\n\\nHEADER = \\"date,sku,qty,price_paise\\\\n\\"\\nFD_DIR = \\"/proc/self/fd\\"\\n\\n\\ndef make_rows(count, seed=5, skus=40, days=60):\\n    rng = random.Random(seed)\\n    sku_names = [f\\"SKU-{i:03d}\\" for i in range(skus)]\\n    prices = {s: rng.randint(500, 50_000) for s in sku_names}\\n    for _ in range(count):\\n        day = f\\"2024-{1 + rng.randrange(days) // 28:02d}-{1 + rng.randrange(28):02d}\\"\\n        sku = rng.choice(sku_names)\\n        yield day, sku, rng.randint(1, 5), prices[sku]\\n\\n\\ndef write_log(path, rows, newline=\\"\\\\n\\"):\\n    with open(path, \\"w\\", encoding=\\"utf-8\\", newline=\\"\\") as f:\\n        f.write(HEADER.replace(\\"\\\\n\\", newline))\\n        for day, sku, qty, price in rows:\\n            f.write(f\\"{day},{sku},{qty},{price}{newline}\\")\\n    return path\\n\\n\\ndef expected_summary(rows):\\n    revenue = units = 0\\n    by_day, by_sku, by_rev = {}, {}, {}\\n    for day, sku, qty, price in rows:\\n        revenue += qty * price\\n        units += qty\\n        by_day[day] = by_day.get(day, 0) + qty * price\\n        by_sku[sku] = by_sku.get(sku, 0) + qty\\n        by_rev[sku] = by_rev.get(sku, 0) + qty * price\\n    summary = {\\"revenue_paise\\": revenue, \\"units\\": units, \\"by_day\\": dict(sorted(by_day.items())),\\n               \\"by_sku\\": dict(sorted(by_sku.items()))}\\n    ranked = sorted(by_rev.items(), key=lambda kv: (-kv[1], kv[0]))\\n    return summary, ranked\\n\\n\\ndef test_small_log_numbers_unchanged(tmp_path):\\n    rows = [\\n        (\\"2024-03-01\\", \\"ATTA-5KG\\", 2, 24000),\\n        (\\"2024-03-01\\", \\"SALT-1KG\\", 1, 2800),\\n        (\\"2024-03-02\\", \\"ATTA-5KG\\", 1, 24000),\\n        (\\"2024-03-02\\", \\"OIL-1L\\", 3, 15500),\\n    ]\\n    p = write_log(tmp_path / \\"s.csv\\", rows)\\n    result = summarize(str(p))\\n    assert result == {\\n        \\"revenue_paise\\": 2 * 24000 + 2800 + 24000 + 3 * 15500,\\n        \\"units\\": 7,\\n        \\"by_day\\": {\\"2024-03-01\\": 50800, \\"2024-03-02\\": 70500},\\n        \\"by_sku\\": {\\"ATTA-5KG\\": 3, \\"OIL-1L\\": 3, \\"SALT-1KG\\": 1},\\n    }\\n    assert list(result[\\"by_day\\"]) == sorted(result[\\"by_day\\"])\\n    assert list(result[\\"by_sku\\"]) == sorted(result[\\"by_sku\\"])\\n    assert [tuple(x) for x in top_skus(str(p), 2)] == [(\\"ATTA-5KG\\", 72000), (\\"OIL-1L\\", 46500)]\\n\\n\\ndef test_random_log_matches_expected(tmp_path):\\n    rows = list(make_rows(3000))\\n    p = write_log(tmp_path / \\"s.csv\\", rows)\\n    summary, ranked = expected_summary(rows)\\n    assert summarize(str(p)) == summary\\n    assert [tuple(x) for x in top_skus(str(p), 5)] == ranked[:5]\\n    assert [tuple(x) for x in top_skus(str(p), 1000)] == ranked\\n\\n\\ndef test_top_skus_ties_broken_by_sku(tmp_path):\\n    rows = [(\\"2024-01-01\\", s, 1, 100) for s in (\\"C\\", \\"A\\", \\"B\\")] + [(\\"2024-01-01\\", \\"Z\\", 1, 500)]\\n    p = write_log(tmp_path / \\"t.csv\\", rows)\\n    assert [tuple(x) for x in top_skus(str(p), 3)] == [(\\"Z\\", 500), (\\"A\\", 100), (\\"B\\", 100)]\\n\\n\\ndef test_windows_line_endings_and_blank_lines(tmp_path):\\n    p = tmp_path / \\"w.csv\\"\\n    p.write_bytes(b\\"date,sku,qty,price_paise\\\\r\\\\n2024-01-01,A,2,100\\\\r\\\\n\\\\r\\\\n2024-01-02,B,1,50\\\\r\\\\n\\")\\n    assert summarize(str(p)) == {\\"revenue_paise\\": 250, \\"units\\": 3, \\"by_day\\": {\\"2024-01-01\\": 200, \\"2024-01-02\\": 50},\\n                                 \\"by_sku\\": {\\"A\\": 2, \\"B\\": 1}}\\n\\n\\ndef test_header_only_and_empty_file(tmp_path):\\n    empty_summary = {\\"revenue_paise\\": 0, \\"units\\": 0, \\"by_day\\": {}, \\"by_sku\\": {}}\\n    p = tmp_path / \\"h.csv\\"\\n    p.write_text(HEADER, encoding=\\"utf-8\\")\\n    assert summarize(str(p)) == empty_summary\\n    assert list(top_skus(str(p), 3)) == []\\n    e = tmp_path / \\"e.csv\\"\\n    e.write_text(\\"\\", encoding=\\"utf-8\\")\\n    assert summarize(str(e)) == empty_summary\\n\\n\\n@pytest.mark.parametrize(\\"bad_line\\", [\\"2024-01-01,A,2\\", \\"2024-01-01,A,two,100\\", \\"2024-01-01,A,2,100,extra\\"])\\ndef test_malformed_line_raises_value_error(tmp_path, bad_line):\\n    p = tmp_path / \\"bad.csv\\"\\n    p.write_text(HEADER + \\"2024-01-01,A,1,100\\\\n\\" + bad_line + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    with pytest.raises(ValueError):\\n        summarize(str(p))\\n    with pytest.raises(ValueError):\\n        top_skus(str(p), 3)\\n\\n\\n@pytest.mark.skipif(not os.path.isdir(FD_DIR), reason=\\"needs /proc/self/fd\\")\\ndef test_files_are_closed_even_on_errors(tmp_path):\\n    good = write_log(tmp_path / \\"g.csv\\", make_rows(100))\\n    bad = tmp_path / \\"bad.csv\\"\\n    bad.write_text(HEADER + \\"2024-01-01,A,1,100\\\\n2024-01-01,A,x,100\\\\n\\", encoding=\\"utf-8\\")\\n    before = len(os.listdir(FD_DIR))\\n    kept = []\\n    for _ in range(30):\\n        summarize(str(good))\\n        top_skus(str(good), 3)\\n        with pytest.raises(ValueError) as excinfo:\\n            summarize(str(bad))\\n        kept.append(excinfo)\\n    assert len(os.listdir(FD_DIR)) - before < 5\\n\\n\\ndef test_memory_does_not_grow_with_file_size(tmp_path):\\n    p = write_log(tmp_path / \\"big.csv\\", make_rows(50_000, skus=30, days=30))\\n    size_mb = p.stat().st_size / 1e6\\n    assert size_mb > 1\\n    tracemalloc.start()\\n    try:\\n        tracemalloc.reset_peak()\\n        summarize(str(p))\\n        _, peak_summarize = tracemalloc.get_traced_memory()\\n        tracemalloc.reset_peak()\\n        top_skus(str(p), 5)\\n        _, peak_top = tracemalloc.get_traced_memory()\\n    finally:\\n        tracemalloc.stop()\\n    limit = 1_000_000  # 1 MB, less than the file itself\\n    assert peak_summarize < limit, f\\"summarize used {peak_summarize / 1e6:.1f} MB for a {size_mb:.1f} MB file\\"\\n    assert peak_top < limit, f\\"top_skus used {peak_top / 1e6:.1f} MB for a {size_mb:.1f} MB file\\"\\n\\n\\ndef test_large_log_is_reasonably_fast(tmp_path):\\n    p = write_log(tmp_path / \\"big.csv\\", make_rows(100_000))\\n    start = time.perf_counter()\\n    summarize(str(p))\\n    top_skus(str(p), 10)\\n    assert time.perf_counter() - start < 2\\n", "id": "28-sales-report-memory-review", "meta": {"difficulty": 3, "files": ["sales_report.py"], "kind": "review", "timeout": 60}, "reference": {"sales_report.py": "\\"\\"\\"Nightly sales report from a branch\'s sales log.\\n\\nThe log is a CSV file with the header `date,sku,qty,price_paise` and one line per item sold.\\nThe till appends to it all year, so the file is read as a stream, one line at a time: memory\\nuse depends on the number of distinct days and SKUs, never on the size of the file.\\n\\"\\"\\"\\n\\nimport heapq\\n\\n\\ndef iter_rows(path):\\n    \\"\\"\\"Yield (date, sku, qty, price_paise) for each sale, streaming the file.\\"\\"\\"\\n    with open(path, encoding=\\"utf-8\\", newline=None) as f:\\n        f.readline()  # header\\n        for n, line in enumerate(f, start=2):\\n            if not line.strip():\\n                continue\\n            parts = line.rstrip(\\"\\\\r\\\\n\\").split(\\",\\")\\n            if len(parts) != 4:\\n                raise ValueError(f\\"{path} line {n}: expected 4 fields, got {len(parts)}\\")\\n            date, sku, qty, price = parts\\n            try:\\n                yield date, sku, int(qty), int(price)\\n            except ValueError:\\n                raise ValueError(f\\"{path} line {n}: bad number\\") from None\\n\\n\\ndef read_rows(path):\\n    \\"\\"\\"All rows as dicts. Loads the whole file: only for small files; reports use iter_rows.\\"\\"\\"\\n    return [{\\"date\\": d, \\"sku\\": s, \\"qty\\": q, \\"price_paise\\": p} for d, s, q, p in iter_rows(path)]\\n\\n\\ndef _totals(path):\\n    revenue = units = 0\\n    by_day: dict[str, int] = {}\\n    by_sku_units: dict[str, int] = {}\\n    by_sku_revenue: dict[str, int] = {}\\n    for date, sku, qty, price in iter_rows(path):\\n        amount = qty * price\\n        revenue += amount\\n        units += qty\\n        by_day[date] = by_day.get(date, 0) + amount\\n        by_sku_units[sku] = by_sku_units.get(sku, 0) + qty\\n        by_sku_revenue[sku] = by_sku_revenue.get(sku, 0) + amount\\n    return revenue, units, by_day, by_sku_units, by_sku_revenue\\n\\n\\ndef summarize(path):\\n    \\"\\"\\"Totals for the report.\\n\\n    Returns {\\"revenue_paise\\": int, \\"units\\": int, \\"by_day\\": {date: revenue_paise}, \\"by_sku\\": {sku: units}},\\n    with by_day and by_sku sorted by key.\\n    \\"\\"\\"\\n    revenue, units, by_day, by_sku_units, _ = _totals(path)\\n    return {\\n        \\"revenue_paise\\": revenue,\\n        \\"units\\": units,\\n        \\"by_day\\": dict(sorted(by_day.items())),\\n        \\"by_sku\\": dict(sorted(by_sku_units.items())),\\n    }\\n\\n\\ndef top_skus(path, n=10):\\n    \\"\\"\\"The `n` SKUs with the most revenue as [(sku, revenue_paise)], highest first, ties by sku.\\"\\"\\"\\n    if n <= 0:\\n        return []\\n    totals = _totals(path)[4]\\n    return heapq.nsmallest(n, totals.items(), key=lambda kv: (-kv[1], kv[0]))\\n"}, "starter": {"sales_report.py": "\\"\\"\\"Nightly sales report from a branch\'s sales log.\\n\\nThe log is a CSV file with the header `date,sku,qty,price_paise` and one line per item sold.\\nThe till appends to it all year.\\n\\"\\"\\"\\n\\n\\ndef read_rows(path):\\n    f = open(path)\\n    lines = f.read().splitlines()\\n    rows = []\\n    for n, line in enumerate(lines[1:], start=2):\\n        if not line.strip():\\n            continue\\n        parts = line.split(\\",\\")\\n        if len(parts) != 4:\\n            raise ValueError(f\\"{path} line {n}: expected 4 fields, got {len(parts)}\\")\\n        date, sku, qty, price = parts\\n        try:\\n            rows.append({\\"date\\": date, \\"sku\\": sku, \\"qty\\": int(qty), \\"price_paise\\": int(price)})\\n        except ValueError:\\n            raise ValueError(f\\"{path} line {n}: bad number\\")\\n    return rows\\n\\n\\ndef summarize(path):\\n    \\"\\"\\"Totals for the report.\\n\\n    Returns {\\"revenue_paise\\": int, \\"units\\": int, \\"by_day\\": {date: revenue_paise}, \\"by_sku\\": {sku: units}},\\n    with by_day and by_sku sorted by key.\\n    \\"\\"\\"\\n    rows = read_rows(path)\\n    revenue = sum(r[\\"qty\\"] * r[\\"price_paise\\"] for r in rows)\\n    units = sum(r[\\"qty\\"] for r in rows)\\n    by_day = {}\\n    for r in rows:\\n        by_day[r[\\"date\\"]] = by_day.get(r[\\"date\\"], 0) + r[\\"qty\\"] * r[\\"price_paise\\"]\\n    by_sku = {}\\n    for r in rows:\\n        by_sku[r[\\"sku\\"]] = by_sku.get(r[\\"sku\\"], 0) + r[\\"qty\\"]\\n    return {\\n        \\"revenue_paise\\": revenue,\\n        \\"units\\": units,\\n        \\"by_day\\": dict(sorted(by_day.items())),\\n        \\"by_sku\\": dict(sorted(by_sku.items())),\\n    }\\n\\n\\ndef top_skus(path, n=10):\\n    \\"\\"\\"The `n` SKUs with the most revenue as [(sku, revenue_paise)], highest first, ties by sku.\\"\\"\\"\\n    rows = read_rows(path)\\n    rows.sort(key=lambda r: r[\\"sku\\"])\\n    totals = {}\\n    for r in rows:\\n        totals[r[\\"sku\\"]] = totals.get(r[\\"sku\\"], 0) + r[\\"qty\\"] * r[\\"price_paise\\"]\\n    ranked = sorted(totals.items(), key=lambda kv: (-kv[1], kv[0]))\\n    return ranked[:n]\\n"}, "task_md": "So far `sales_report.py` has only run for our own shop, and next month it goes to all 40 branches.\\nEach branch\'s till appends every sale to one CSV log for the whole year; the biggest branch\'s log reaches about 6 GB by March, and the report runs every night on a small server with 1 GB of RAM.\\nPlease review the module for that rollout and fix what needs fixing.\\nThe nightly job calls `summarize(path)` and `top_skus(path, n)`, and the numbers in the report must stay exactly the same as now."}'),
    json.loads('{"hidden": "import itertools\\n\\nimport pytest\\n\\nfrom delivery import calc, delivery_fee\\n\\n\\ndef legacy_calc(d, amt, exp, mem, hr):\\n    \\"\\"\\"The original implementation, copied verbatim as the oracle.\\"\\"\\"\\n    if d < 0:\\n        raise ValueError(\\"bad distance\\")\\n    if d > 25:\\n        raise ValueError(\\"too far\\")\\n    if amt >= 50000 and exp == False:  # noqa: E712\\n        return 0\\n    if mem == True:  # noqa: E712\\n        if d <= 3:\\n            f = 0\\n        else:\\n            if d <= 7:\\n                f = 1500\\n            else:\\n                f = 1500 + int((d - 7) * 500)\\n        if exp == True:  # noqa: E712\\n            f = f + 3000\\n        if hr >= 22 or hr < 6:\\n            f = f + 1000\\n    else:\\n        if d <= 3:\\n            f = 2000\\n        else:\\n            if d <= 7:\\n                f = 3500\\n            else:\\n                f = 3500 + int((d - 7) * 500)\\n        if exp == True:  # noqa: E712\\n            f = f + 3000\\n        if hr >= 22 or hr < 6:\\n            f = f + 2500\\n    if f > 15000:\\n        f = 15000\\n    return f\\n\\n\\nDISTANCES = [0, 0.5, 1, 2.9, 3, 3.1, 4.5, 6.9, 7, 7.1, 7.3, 7.7, 8, 9.9, 12.4, 15, 19.7, 23.3, 25]\\nAMOUNTS = [0, 100, 49_999, 50_000, 120_000]\\nHOURS = list(range(24))\\nGRID = list(itertools.product(DISTANCES, AMOUNTS, [False, True], [False, True], HOURS))\\n\\n\\ndef test_calc_positional_matches_old_behaviour_everywhere():\\n    mismatches = [args for args in GRID if calc(*args) != legacy_calc(*args)]\\n    assert mismatches == [], f\\"{len(mismatches)} fees changed, e.g. calc{mismatches[0]}\\"\\n\\n\\ndef test_delivery_fee_without_rain_matches_old_behaviour():\\n    for d, amt, exp, mem, hr in GRID:\\n        got = delivery_fee(distance_km=d, order_paise=amt, express=exp, member=mem, hour=hr, rain=False)\\n        assert got == legacy_calc(d, amt, exp, mem, hr), (d, amt, exp, mem, hr)\\n\\n\\ndef test_delivery_fee_defaults():\\n    assert delivery_fee(5, 10_000) == legacy_calc(5, 10_000, False, False, 12)\\n    assert delivery_fee(2, 10_000, member=True) == 0\\n    assert delivery_fee(10, 10_000, express=True, hour=23) == legacy_calc(10, 10_000, True, False, 23)\\n\\n\\ndef test_rain_adds_15_rupees_to_non_zero_fees_with_cap():\\n    checked_nonzero = checked_zero = checked_cap = 0\\n    for d, amt, exp, mem, hr in GRID:\\n        old = legacy_calc(d, amt, exp, mem, hr)\\n        wet = delivery_fee(d, amt, express=exp, member=mem, hour=hr, rain=True)\\n        if old == 0:\\n            assert wet == 0, (d, amt, exp, mem, hr)\\n            checked_zero += 1\\n        else:\\n            uncapped = delivery_fee(d, amt, express=exp, member=mem, hour=hr)\\n            assert wet == min(uncapped + 1500, 15000), (d, amt, exp, mem, hr)\\n            checked_nonzero += 1\\n            checked_cap += wet == 15000\\n    assert checked_nonzero and checked_zero and checked_cap\\n\\n\\ndef test_rain_examples():\\n    assert delivery_fee(5, 10_000, rain=True) == 3500 + 1500\\n    assert delivery_fee(5, 60_000, rain=True) == 0  # free delivery stays free\\n    assert delivery_fee(1, 10_000, member=True, rain=True) == 0\\n    assert delivery_fee(25, 10_000, express=True, hour=23, rain=True) == 15000\\n\\n\\ndef test_calc_keeps_ignoring_rain():\\n    assert calc(5, 10_000, False, False, 12) == 3500\\n\\n\\n@pytest.mark.parametrize(\\"d\\", [-0.1, -5, 25.1, 40])\\ndef test_out_of_range_distance_still_rejected(d):\\n    with pytest.raises(ValueError):\\n        calc(d, 10_000, False, False, 12)\\n    with pytest.raises(ValueError):\\n        delivery_fee(d, 10_000)\\n    with pytest.raises(ValueError):\\n        delivery_fee(d, 10_000, rain=True)\\n\\n\\ndef test_results_are_whole_paise():\\n    for d, amt, exp, mem, hr in GRID[::7]:\\n        fee = delivery_fee(d, amt, express=exp, member=mem, hour=hr, rain=True)\\n        assert isinstance(fee, int) and not isinstance(fee, bool)\\n", "id": "29-delivery-fee-refactor", "meta": {"difficulty": 2, "files": ["delivery.py"], "kind": "refactor", "timeout": 60}, "reference": {"delivery.py": "\\"\\"\\"Delivery fee rules. All money is in paise.\\"\\"\\"\\n\\nMAX_DISTANCE_KM = 25\\nFREE_DELIVERY_FROM_PAISE = 50_000  # non-express orders of Rs 500 or more deliver free\\nNEAR_KM = 3\\nMID_KM = 7\\nPER_KM_BEYOND_MID_PAISE = 500\\nEXPRESS_PAISE = 3_000\\nRAIN_PAISE = 1_500\\nFEE_CAP_PAISE = 15_000\\nNIGHT_STARTS, NIGHT_ENDS = 22, 6  # 22:00 to 05:59\\n\\n# (fee up to NEAR_KM, fee up to MID_KM, night surcharge)\\n_MEMBER = (0, 1_500, 1_000)\\n_REGULAR = (2_000, 3_500, 2_500)\\n\\n\\ndef _distance_fee(distance_km, near_fee, mid_fee):\\n    if distance_km <= NEAR_KM:\\n        return near_fee\\n    if distance_km <= MID_KM:\\n        return mid_fee\\n    # int() truncates; kept exactly as the original calculation\\n    return mid_fee + int((distance_km - MID_KM) * PER_KM_BEYOND_MID_PAISE)\\n\\n\\ndef _is_night(hour):\\n    return hour >= NIGHT_STARTS or hour < NIGHT_ENDS\\n\\n\\ndef delivery_fee(distance_km, order_paise, express=False, member=False, hour=12, rain=False):\\n    \\"\\"\\"Delivery fee in paise.\\n\\n    Raises ValueError for a negative distance or one beyond MAX_DISTANCE_KM.\\n    \\"\\"\\"\\n    if distance_km < 0:\\n        raise ValueError(\\"distance_km must not be negative\\")\\n    if distance_km > MAX_DISTANCE_KM:\\n        raise ValueError(f\\"we deliver up to {MAX_DISTANCE_KM} km\\")\\n    if order_paise >= FREE_DELIVERY_FROM_PAISE and not express:\\n        return 0\\n\\n    near_fee, mid_fee, night_fee = _MEMBER if member else _REGULAR\\n    fee = _distance_fee(distance_km, near_fee, mid_fee)\\n    if express:\\n        fee += EXPRESS_PAISE\\n    if _is_night(hour):\\n        fee += night_fee\\n    if rain and fee > 0:\\n        fee += RAIN_PAISE\\n    return min(fee, FEE_CAP_PAISE)\\n\\n\\ndef calc(d, amt, exp, mem, hr):\\n    \\"\\"\\"Old name and argument order, kept for the billing counter software.\\"\\"\\"\\n    return delivery_fee(d, amt, express=exp, member=mem, hour=hr)\\n"}, "starter": {"delivery.py": "def calc(d, amt, exp, mem, hr):\\n    # d = km, amt = order value (paise), exp = express?, mem = member?, hr = hour of day (0-23)\\n    if d < 0:\\n        raise ValueError(\\"bad distance\\")\\n    if d > 25:\\n        raise ValueError(\\"too far\\")\\n    if amt >= 50000 and exp == False:\\n        return 0\\n    if mem == True:\\n        if d <= 3:\\n            f = 0\\n        else:\\n            if d <= 7:\\n                f = 1500\\n            else:\\n                f = 1500 + int((d - 7) * 500)\\n        if exp == True:\\n            f = f + 3000\\n        if hr >= 22 or hr < 6:\\n            f = f + 1000\\n    else:\\n        if d <= 3:\\n            f = 2000\\n        else:\\n            if d <= 7:\\n                f = 3500\\n            else:\\n                f = 3500 + int((d - 7) * 500)\\n        if exp == True:\\n            f = f + 3000\\n        if hr >= 22 or hr < 6:\\n            f = f + 2500\\n    if f > 15000:\\n        f = 15000\\n    return f\\n"}, "task_md": "Nobody in the team can read `calc()` in `delivery.py` any more, and we need to change it, so please clean it up first.\\nThe change: when it\'s raining, every delivery fee that isn\'t zero goes up by ₹15, and the ₹150 maximum still applies to the total.\\nThe new delivery app should call it with readable names: `delivery_fee(distance_km, order_paise, express=False, member=False, hour=12, rain=False)`, returning the fee in paise like `calc` does.\\nThe old billing counter software will keep calling `calc(d, amt, exp, mem, hr)` with positional arguments for a few more months, and for it every fee must stay exactly what it is today, to the paisa."}'),
    json.loads('{"hidden": "import copy\\nimport random\\n\\nimport pytest\\n\\nfrom invoice import invoice_totals, make_invoice\\n\\n\\ndef legacy_make_invoice(order):\\n    \\"\\"\\"The original implementation, copied verbatim as the oracle.\\"\\"\\"\\n    # order = {\\"invoice_no\\": \\"...\\", \\"date\\": \\"2024-11-05\\", \\"customer\\": \\"...\\", \\"items\\": [{\\"name\\", \\"qty\\", \\"price_paise\\"}],\\n    #          \\"discount_pct\\": 10}   (customer and discount_pct are optional)\\n    if len(order[\\"items\\"]) == 0:\\n        raise ValueError(\\"no items\\")\\n    s = \\"\\"\\n    s = s + \\"SHARMA KIRANA STORE\\\\n\\"\\n    s = s + \\"Invoice: \\" + order[\\"invoice_no\\"] + \\"   Date: \\" + order[\\"date\\"] + \\"\\\\n\\"\\n    if \\"customer\\" in order and order[\\"customer\\"]:\\n        s = s + \\"Customer: \\" + order[\\"customer\\"] + \\"\\\\n\\"\\n    else:\\n        s = s + \\"Customer: Cash\\\\n\\"\\n    s = s + \\"-\\" * 45 + \\"\\\\n\\"\\n    t = 0\\n    for i in order[\\"items\\"]:\\n        a = i[\\"qty\\"] * i[\\"price_paise\\"]\\n        t = t + a\\n        n = i[\\"name\\"]\\n        if len(n) > 20:\\n            n = n[:19] + \\"~\\"\\n        s = s + n.ljust(20) + str(i[\\"qty\\"]).rjust(4) + \\" x \\" + (\\"%.2f\\" % (i[\\"price_paise\\"] / 100)).rjust(8)\\n        s = s + (\\"%.2f\\" % (a / 100)).rjust(10) + \\"\\\\n\\"\\n    s = s + \\"-\\" * 45 + \\"\\\\n\\"\\n    s = s + \\"Subtotal\\".ljust(35) + (\\"%.2f\\" % (t / 100)).rjust(10) + \\"\\\\n\\"\\n    d = 0\\n    if \\"discount_pct\\" in order:\\n        if order[\\"discount_pct\\"]:\\n            d = int(t * order[\\"discount_pct\\"] / 100)\\n            s = s + (\\"Discount (\\" + str(order[\\"discount_pct\\"]) + \\"%)\\").ljust(35) + (\\"-%.2f\\" % (d / 100)).rjust(10) + \\"\\\\n\\"\\n    net = t - d\\n    r = round(net / 100) * 100 - net\\n    if r != 0:\\n        s = s + \\"Round off\\".ljust(35) + (\\"%+.2f\\" % (r / 100)).rjust(10) + \\"\\\\n\\"\\n    s = s + \\"TOTAL\\".ljust(35) + (\\"%.2f\\" % ((net + r) / 100)).rjust(10) + \\"\\\\n\\"\\n    s = s + \\"-\\" * 45 + \\"\\\\n\\"\\n    s = s + \\"Thank you! Visit again.\\\\n\\"\\n    return s\\n\\n\\ndef legacy_totals(order):\\n    \\"\\"\\"The numbers the original bill prints, recomputed the original way.\\"\\"\\"\\n    t = sum(i[\\"qty\\"] * i[\\"price_paise\\"] for i in order[\\"items\\"])\\n    d = int(t * order[\\"discount_pct\\"] / 100) if order.get(\\"discount_pct\\") else 0\\n    net = t - d\\n    r = round(net / 100) * 100 - net\\n    return {\\"subtotal_paise\\": t, \\"discount_paise\\": d, \\"round_off_paise\\": r, \\"total_paise\\": net + r}\\n\\n\\nNAMES = [\\"Atta 5kg\\", \\"Toor Dal 1kg\\", \\"Fortune Sunflower Oil 1 Litre Pouch\\", \\"Salt\\", \\"Maggi Masala Noodles 12 pack\\",\\n         \\"Haldi 200g\\", \\"Parle-G\\", \\"Amul Butter 500g\\", \\"Basmati Rice Premium 5kg\\", \\"Eggs (tray of 30)\\"]\\n\\n\\ndef random_order(rng, n):\\n    order = {\\n        \\"invoice_no\\": f\\"INV-{n:05d}\\",\\n        \\"date\\": f\\"2024-{rng.randint(1, 12):02d}-{rng.randint(1, 28):02d}\\",\\n        \\"items\\": [\\n            {\\"name\\": rng.choice(NAMES), \\"qty\\": rng.randint(1, 12), \\"price_paise\\": rng.randint(1, 250_000)}\\n            for _ in range(rng.randint(1, 8))\\n        ],\\n    }\\n    if rng.random() < 0.7:\\n        order[\\"customer\\"] = rng.choice([\\"Mrs. Gupta\\", \\"Ramesh\\", \\"\\", \\"Hotel Annapurna (credit)\\"])\\n    if rng.random() < 0.6:\\n        order[\\"discount_pct\\"] = rng.choice([0, 5, 10, 12.5, 2.5, 7, 15, 33.3])\\n    return order\\n\\n\\nEXAMPLES = [\\n    {\\"invoice_no\\": \\"INV-1\\", \\"date\\": \\"2024-11-05\\", \\"items\\": [{\\"name\\": \\"Atta 5kg\\", \\"qty\\": 1, \\"price_paise\\": 24000}]},\\n    {\\"invoice_no\\": \\"INV-2\\", \\"date\\": \\"2024-11-05\\", \\"customer\\": \\"Mrs. Gupta\\", \\"discount_pct\\": 12.5,\\n     \\"items\\": [{\\"name\\": \\"Toor Dal 1kg\\", \\"qty\\": 3, \\"price_paise\\": 14999},\\n               {\\"name\\": \\"Fortune Sunflower Oil 1 Litre Pouch\\", \\"qty\\": 2, \\"price_paise\\": 15550}]},\\n    # net 250 paise and 350 paise: the original rounds halves to the even rupee\\n    {\\"invoice_no\\": \\"INV-3\\", \\"date\\": \\"2024-11-06\\", \\"items\\": [{\\"name\\": \\"Salt\\", \\"qty\\": 1, \\"price_paise\\": 250}]},\\n    {\\"invoice_no\\": \\"INV-4\\", \\"date\\": \\"2024-11-06\\", \\"items\\": [{\\"name\\": \\"Salt\\", \\"qty\\": 1, \\"price_paise\\": 350}]},\\n    # 33.3% of 12000 is 3995.9999999999995 in floating point: the original bill truncates it to 3995\\n    {\\"invoice_no\\": \\"INV-5\\", \\"date\\": \\"2024-11-07\\", \\"customer\\": \\"\\", \\"discount_pct\\": 33.3,\\n     \\"items\\": [{\\"name\\": \\"Parle-G\\", \\"qty\\": 120, \\"price_paise\\": 100}]},\\n    {\\"invoice_no\\": \\"INV-6\\", \\"date\\": \\"2024-11-07\\", \\"discount_pct\\": 0, \\"customer\\": None,\\n     \\"items\\": [{\\"name\\": \\"Exactly twenty chars\\", \\"qty\\": 1, \\"price_paise\\": 100},\\n               {\\"name\\": \\"Twenty-one characters\\", \\"qty\\": 1000, \\"price_paise\\": 99}]},\\n    {\\"invoice_no\\": \\"INV-7\\", \\"date\\": \\"2024-11-08\\", \\"discount_pct\\": 10.0,\\n     \\"items\\": [{\\"name\\": \\"Gold coin (Dhanteras)\\", \\"qty\\": 1, \\"price_paise\\": 734_567_891}]},\\n]\\n\\n\\n@pytest.mark.parametrize(\\"order\\", EXAMPLES, ids=[o[\\"invoice_no\\"] for o in EXAMPLES])\\ndef test_printed_bill_unchanged_examples(order):\\n    assert make_invoice(copy.deepcopy(order)) == legacy_make_invoice(copy.deepcopy(order))\\n\\n\\ndef test_printed_bill_unchanged_random_orders():\\n    rng = random.Random(1234)\\n    for n in range(400):\\n        order = random_order(rng, n)\\n        assert make_invoice(copy.deepcopy(order)) == legacy_make_invoice(copy.deepcopy(order)), order\\n\\n\\n@pytest.mark.parametrize(\\"order\\", EXAMPLES, ids=[o[\\"invoice_no\\"] for o in EXAMPLES])\\ndef test_totals_match_the_bill_examples(order):\\n    totals = invoice_totals(copy.deepcopy(order))\\n    assert {k: totals[k] for k in legacy_totals(order)} == legacy_totals(order)\\n\\n\\ndef test_totals_match_the_printed_bill_random_orders():\\n    rng = random.Random(99)\\n    for n in range(300):\\n        order = random_order(rng, n)\\n        totals = invoice_totals(copy.deepcopy(order))\\n        expected = legacy_totals(order)\\n        assert {k: totals[k] for k in expected} == expected, order\\n        printed_total = [line for line in legacy_make_invoice(order).splitlines() if line.startswith(\\"TOTAL\\")][0]\\n        assert printed_total.split()[-1] == \\"%.2f\\" % (totals[\\"total_paise\\"] / 100)\\n\\n\\ndef test_totals_are_whole_paise_and_add_up():\\n    rng = random.Random(5)\\n    for n in range(100):\\n        totals = invoice_totals(random_order(rng, n))\\n        for key in (\\"subtotal_paise\\", \\"discount_paise\\", \\"round_off_paise\\", \\"total_paise\\"):\\n            assert isinstance(totals[key], int) and not isinstance(totals[key], bool), key\\n        assert totals[\\"total_paise\\"] == totals[\\"subtotal_paise\\"] - totals[\\"discount_paise\\"] + totals[\\"round_off_paise\\"]\\n        assert totals[\\"total_paise\\"] % 100 == 0\\n\\n\\ndef test_no_items_still_rejected():\\n    order = {\\"invoice_no\\": \\"INV-0\\", \\"date\\": \\"2024-11-05\\", \\"items\\": []}\\n    with pytest.raises(ValueError):\\n        make_invoice(order)\\n    with pytest.raises(ValueError):\\n        invoice_totals(order)\\n\\n\\ndef test_order_is_not_modified():\\n    rng = random.Random(8)\\n    for n in range(50):\\n        order = random_order(rng, n)\\n        before = copy.deepcopy(order)\\n        make_invoice(order)\\n        invoice_totals(order)\\n        assert order == before\\n\\n\\ndef test_same_order_same_bill():\\n    order = EXAMPLES[1]\\n    assert make_invoice(order) == make_invoice(order)\\n    assert invoice_totals(order) == invoice_totals(order)\\n", "id": "30-invoice-refactor", "meta": {"difficulty": 3, "files": ["invoice.py"], "kind": "refactor", "timeout": 60}, "reference": {"invoice.py": "\\"\\"\\"Counter invoices: the numbers (`invoice_totals`) and the printed bill (`make_invoice`).\\n\\norder = {\\"invoice_no\\": str, \\"date\\": \\"2024-11-05\\", \\"customer\\": str (optional),\\n         \\"items\\": [{\\"name\\": str, \\"qty\\": int, \\"price_paise\\": int}, ...], \\"discount_pct\\": number (optional)}\\n\\nBoth functions use the same calculation, so the app and the printed bill always agree.\\nThe printed layout is byte-for-byte the historical one.\\n\\"\\"\\"\\n\\nSHOP_NAME = \\"SHARMA KIRANA STORE\\"\\nFOOTER = \\"Thank you! Visit again.\\"\\nWIDTH = 45\\nNAME_WIDTH = 20\\nLABEL_WIDTH = 35\\nMONEY_WIDTH = 10\\n\\n\\ndef _rupees(paise):\\n    # historical formatting (float division), kept so the printed bill does not change\\n    return \\"%.2f\\" % (paise / 100)\\n\\n\\ndef _line_amount(item):\\n    return item[\\"qty\\"] * item[\\"price_paise\\"]\\n\\n\\ndef invoice_totals(order):\\n    \\"\\"\\"Subtotal, discount, round-off and total in paise, exactly as printed on the bill.\\"\\"\\"\\n    items = order[\\"items\\"]\\n    if not items:\\n        raise ValueError(\\"no items\\")\\n    subtotal = sum(_line_amount(item) for item in items)\\n    pct = order.get(\\"discount_pct\\") or 0\\n    # truncation and float maths kept from the original bill\\n    discount = int(subtotal * pct / 100) if pct else 0\\n    net = subtotal - discount\\n    round_off = round(net / 100) * 100 - net  # nearest rupee (Python\'s round: halves to even)\\n    return {\\n        \\"subtotal_paise\\": subtotal,\\n        \\"discount_paise\\": discount,\\n        \\"round_off_paise\\": round_off,\\n        \\"total_paise\\": net + round_off,\\n    }\\n\\n\\ndef _item_line(item):\\n    name = item[\\"name\\"]\\n    if len(name) > NAME_WIDTH:\\n        name = name[: NAME_WIDTH - 1] + \\"~\\"\\n    return (\\n        name.ljust(NAME_WIDTH)\\n        + str(item[\\"qty\\"]).rjust(4)\\n        + \\" x \\"\\n        + _rupees(item[\\"price_paise\\"]).rjust(8)\\n        + _rupees(_line_amount(item)).rjust(MONEY_WIDTH)\\n    )\\n\\n\\ndef _total_line(label, amount_text):\\n    return label.ljust(LABEL_WIDTH) + amount_text.rjust(MONEY_WIDTH)\\n\\n\\ndef make_invoice(order):\\n    \\"\\"\\"The printed bill as text.\\"\\"\\"\\n    totals = invoice_totals(order)\\n    rule = \\"-\\" * WIDTH\\n    lines = [\\n        SHOP_NAME,\\n        \\"Invoice: \\" + order[\\"invoice_no\\"] + \\"   Date: \\" + order[\\"date\\"],\\n        \\"Customer: \\" + (order.get(\\"customer\\") or \\"Cash\\"),\\n        rule,\\n    ]\\n    lines += [_item_line(item) for item in order[\\"items\\"]]\\n    lines.append(rule)\\n    lines.append(_total_line(\\"Subtotal\\", _rupees(totals[\\"subtotal_paise\\"])))\\n    if order.get(\\"discount_pct\\"):\\n        label = \\"Discount (\\" + str(order[\\"discount_pct\\"]) + \\"%)\\"\\n        lines.append(_total_line(label, \\"-\\" + _rupees(totals[\\"discount_paise\\"])))\\n    if totals[\\"round_off_paise\\"] != 0:\\n        lines.append(_total_line(\\"Round off\\", \\"%+.2f\\" % (totals[\\"round_off_paise\\"] / 100)))\\n    lines.append(_total_line(\\"TOTAL\\", _rupees(totals[\\"total_paise\\"])))\\n    lines.append(rule)\\n    lines.append(FOOTER)\\n    return \\"\\\\n\\".join(lines) + \\"\\\\n\\"\\n"}, "starter": {"invoice.py": "def make_invoice(order):\\n    # order = {\\"invoice_no\\": \\"...\\", \\"date\\": \\"2024-11-05\\", \\"customer\\": \\"...\\", \\"items\\": [{\\"name\\", \\"qty\\", \\"price_paise\\"}],\\n    #          \\"discount_pct\\": 10}   (customer and discount_pct are optional)\\n    if len(order[\\"items\\"]) == 0:\\n        raise ValueError(\\"no items\\")\\n    s = \\"\\"\\n    s = s + \\"SHARMA KIRANA STORE\\\\n\\"\\n    s = s + \\"Invoice: \\" + order[\\"invoice_no\\"] + \\"   Date: \\" + order[\\"date\\"] + \\"\\\\n\\"\\n    if \\"customer\\" in order and order[\\"customer\\"]:\\n        s = s + \\"Customer: \\" + order[\\"customer\\"] + \\"\\\\n\\"\\n    else:\\n        s = s + \\"Customer: Cash\\\\n\\"\\n    s = s + \\"-\\" * 45 + \\"\\\\n\\"\\n    t = 0\\n    for i in order[\\"items\\"]:\\n        a = i[\\"qty\\"] * i[\\"price_paise\\"]\\n        t = t + a\\n        n = i[\\"name\\"]\\n        if len(n) > 20:\\n            n = n[:19] + \\"~\\"\\n        s = s + n.ljust(20) + str(i[\\"qty\\"]).rjust(4) + \\" x \\" + (\\"%.2f\\" % (i[\\"price_paise\\"] / 100)).rjust(8)\\n        s = s + (\\"%.2f\\" % (a / 100)).rjust(10) + \\"\\\\n\\"\\n    s = s + \\"-\\" * 45 + \\"\\\\n\\"\\n    s = s + \\"Subtotal\\".ljust(35) + (\\"%.2f\\" % (t / 100)).rjust(10) + \\"\\\\n\\"\\n    d = 0\\n    if \\"discount_pct\\" in order:\\n        if order[\\"discount_pct\\"]:\\n            d = int(t * order[\\"discount_pct\\"] / 100)\\n            s = s + (\\"Discount (\\" + str(order[\\"discount_pct\\"]) + \\"%)\\").ljust(35) + (\\"-%.2f\\" % (d / 100)).rjust(10) + \\"\\\\n\\"\\n    net = t - d\\n    r = round(net / 100) * 100 - net\\n    if r != 0:\\n        s = s + \\"Round off\\".ljust(35) + (\\"%+.2f\\" % (r / 100)).rjust(10) + \\"\\\\n\\"\\n    s = s + \\"TOTAL\\".ljust(35) + (\\"%.2f\\" % ((net + r) / 100)).rjust(10) + \\"\\\\n\\"\\n    s = s + \\"-\\" * 45 + \\"\\\\n\\"\\n    s = s + \\"Thank you! Visit again.\\\\n\\"\\n    return s\\n"}, "task_md": "`make_invoice(order)` in `invoice.py` prints the bills at our counter. It works, but every change to it breaks something, so please restructure it into something we can maintain.\\nWe also need the new mobile app to show the same numbers without the text, so add `invoice_totals(order) -> dict` with `subtotal_paise`, `discount_paise`, `round_off_paise` and `total_paise` (whole paise). These must always agree with the printed bill.\\nThe printed bill must stay exactly the same, character for character: our printer templates and the tax auditor\'s checks depend on it."}'),
]

In [ ]:
#@title 6. Pick the tasks, and check every reference solution passes here
os.makedirs(GRADE_DIR, exist_ok=True)
TASKS = EXAM_TASKS[:LIMIT] if LIMIT else list(EXAM_TASKS)
print(f"{len(EXAM_TASKS)} tasks; using {len(TASKS)}:", dict(Counter(t["meta"]["kind"] for t in TASKS)))
# Check the checker here, before any model: every task's reference solution must pass its hidden tests on this
# machine. A task whose reference fails is left out (it would only measure this machine).
rows, _ = check_references(TASKS, GRADE_DIR)
print("\n".join(reference_table(rows)))
RESULTS["reference_check"] = [{"task": r["task"], "total": r["grade"]["total"], "problems": r["problems"],
                               "hidden": f"{r['grade']['hidden_passed']}/{r['grade']['hidden_total']}"} for r in rows]
RESULTS["tasks_left_out"] = sorted(r["task"] for r in rows if r["problems"])
if RESULTS["tasks_left_out"]:
    print("Left out (their reference fails here):", ", ".join(RESULTS["tasks_left_out"]))
TASKS = [t for t in TASKS if t["id"] not in RESULTS["tasks_left_out"]]
RESULTS.update(tasks_total=len(EXAM_TASKS), tasks_used=len(TASKS), task_ids=[t["id"] for t in TASKS],
               kinds=dict(Counter(t["meta"]["kind"] for t in TASKS)))
save_results()
if not TASKS:
    raise RuntimeError("No task left to give the models (see above)")
print("\n--- What a model is sent (the first task):\n" + prompt_for(TASKS[0]))

In [ ]:
#@title 7. Each model: download, load, do every task, grade each reply
for model in MODELS:
    entry = {"model": model}
    RESULTS["models"].append(entry)
    try:
        take_exam(OLLAMA_URL, model, TASKS, entry, MINUTES_PER_MODEL, OPTIONS, THINK, THINKING_NUM_PREDICT, SAMPLES,
                  save=save_results, models_dir=MODELS_DIR, workdir=GRADE_DIR)
    except Exception as e:  # record it and go on to the next model
        entry.setdefault("errors", []).append(f"{e!r}"[:500])
        failed(model, e)
    if entry.get("pulled"):
        tidy_up(OLLAMA_URL, model, entry, DELETE_AFTER)
    else:  # a download that stopped part-way leaves its pieces: remove them, so the next model has the disk
        for part in glob.glob(os.path.join(MODELS_DIR, "blobs", "*-partial*")):
            os.remove(part)
    save_results()
    print("\n".join(table(RESULTS)))

In [ ]:
#@title 8. Results
print("\n".join(table(RESULTS)))
for m in RESULTS["models"]:
    if m.get("tasks"):
        print(f"\n{m['model']}: the 5 weakest tasks")
        print("\n".join(weakest(m)))
    for error in m.get("errors") or []:
        print(f"{m['model']}: {error}")
for error in RESULTS["errors"]:
    print("Error:", error)
save_results()
print("Saved:", os.path.join(WORKING_DIR, RESULTS_FILE))

## What next
`exam.json` keeps every reply and every grade (with the test, lint and bandit output), so each weak task can be
read. The weakest parts say what to teach next: if models rarely write tests or explain themselves on their own,
that is what a fine-tune (or a better default prompt in the coding agent) should add.